In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'llm.py': '"""Greedy chat generation shared by the screening runs: vLLM (LLM_BACKEND=vllm, default) or transformers (LLM_BACKEND=hf).\n\n    m = LLM("Qwen/Qwen3-4B-Instruct-2507", max_len=16384, tp=1)\n    texts = m.gen([[{"role": "system", "content": ...}, {"role": "user", "content": ...}], ...], max_tokens=512)\n\nOne process loads one repo; trying the next candidate (fp16 -> AWQ, vllm -> hf) is multirun.py\'s job, in a new process.\nPrompts are tokenized here (chat template, enable_thinking off unless think=True) and passed as token ids, so no double BOS.\nA prompt longer than max_len - max_tokens comes back as None and is counted in m.overflow (never crashes the batch).\nText after </think> is kept when a model still thinks. m.ntok_in / m.ntok_out count tokens of every call.\n"""\nimport os\nimport time\n\n# ChatML, for tokenizers that ship no chat template (vilm/vinallama-2.7b-chat): the format its model card describes\nCHATML = ("{% for m in messages %}{{ \'<|im_start|>\' + m[\'role\'] + \'\\n\' + m[\'content\'] + \'<|im_end|>\\n\' }}{% endfor %}"\n          "{% if add_generation_prompt %}{{ \'<|im_start|>assistant\\n\' }}{% endif %}")\n\n\nclass LLM:\n    def __init__(self, repo, max_len=16384, tp=1, think=False, **kw):\n        self.repo, self.max_len, self.think = repo, max_len, think\n        self.overflow = self.ntok_in = self.ntok_out = 0\n        self.backend = os.environ.get("LLM_BACKEND", "vllm")\n        t0 = time.time()\n        if self.backend == "vllm":\n            from vllm import LLM as V\n            self.eng = V(model=repo, dtype="float16", max_model_len=max_len, tensor_parallel_size=tp, seed=0,\n                         gpu_memory_utilization=float(os.environ.get("GPU_UTIL", "0.90")), trust_remote_code=True, **kw)\n            self.tok = self.eng.get_tokenizer()\n        else:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            import transformers.integrations.sdpa_attention as sd\n            if hasattr(sd, "use_gqa_in_sdpa"):  # T4: enable_gqa only runs in the math kernel (heads x L x L scores, OOM): repeat K/V instead\n                sd.use_gqa_in_sdpa = lambda *a, **k: False\n            dkw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"\n            self.tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)\n            cuda = torch.cuda.is_available()\n            load = lambda remote: AutoModelForCausalLM.from_pretrained(repo, **{dkw: torch.float16 if cuda else torch.float32},\n                                                                       device_map="auto" if cuda else None, trust_remote_code=remote).eval()\n            try:\n                self.model = load(True)\n            except ImportError as e:  # repo code written for an older transformers (Phi-4-mini: LossKwargs); the built-in class works\n                print(f"[llm] remote code failed ({e}); loading the built-in transformers class", flush=True)\n                self.model = load(False)\n        if not getattr(self.tok, "chat_template", None):\n            self.tok.chat_template = CHATML\n            print(f"[llm] {repo}: no chat template, using ChatML", flush=True)\n        print(f"[llm] {repo} loaded with {self.backend} in {time.time() - t0:.0f}s", flush=True)\n\n    def ids(self, msgs):\n        kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=self.think)\n        try:\n            text = self.tok.apply_chat_template(msgs, **kw)\n        except Exception:  # templates without a system role: fold it into the first user turn\n            if msgs[0]["role"] != "system":\n                raise\n            text = self.tok.apply_chat_template([dict(msgs[1], content=msgs[0]["content"] + "\\n\\n" + msgs[1]["content"])] + msgs[2:], **kw)\n        if os.environ.get("STRIP_EMPTY_THINK") == "1":  # models fine-tuned by PIPE-T1-04 train.py on Qwen3.5 (no empty think block)\n            text = text.replace("<think>\\n\\n</think>\\n\\n", "")\n        return self.tok(text, add_special_tokens=False).input_ids\n\n    def gen(self, batch, max_tokens):\n        ids = [self.ids(m) for m in batch]\n        ok = [i for i, x in enumerate(ids) if len(x) <= self.max_len - max_tokens]\n        self.overflow += len(ids) - len(ok)\n        self.ntok_in += sum(len(ids[i]) for i in ok)\n        out = [None] * len(batch)\n        if self.backend == "vllm":\n            from vllm import SamplingParams\n            sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)\n            res = self.eng.generate([{"prompt_token_ids": ids[i]} for i in ok], sp, use_tqdm=False)\n            for i, r in zip(ok, res):\n                out[i] = r.outputs[0].text\n                self.ntok_out += len(r.outputs[0].token_ids)\n        else:\n            import torch\n            bs = int(os.environ.get("HF_BATCH", "4"))\n            pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id\n            dev = next(self.model.parameters()).device\n            order = sorted(ok, key=lambda i: len(ids[i]))\n            for k in range(0, len(order), bs):\n                part = order[k:k + bs]\n                n = max(len(ids[i]) for i in part)\n                x = torch.tensor([[pad] * (n - len(ids[i])) + ids[i] for i in part], device=dev)\n                mask = torch.tensor([[0] * (n - len(ids[i])) + [1] * len(ids[i]) for i in part], device=dev)\n                with torch.no_grad():\n                    y = self.model.generate(input_ids=x, attention_mask=mask, max_new_tokens=max_tokens, do_sample=False, pad_token_id=pad)\n                for i, row in zip(part, y[:, n:].tolist()):\n                    row = [t for t in row if t != pad]\n                    self.ntok_out += len(row)\n                    out[i] = self.tok.decode(row, skip_special_tokens=True)\n        return [o.split("</think>")[-1].strip() if o is not None else None for o in out]\n',
    'multirun.py': '"""Run one worker process per model over the GPUs of a Kaggle T4 x2 box.\n\n    ONLY=tag,tag python -u multirun.py jobs.json worker.py [-- extra worker args]      (ONLY empty or unset: every job)\n\njobs.json: [{"tag": "q3-4b", "tp": 1, "env": {"HARNESS": "4"}, "cands": [{"repo": "...", "backend": "vllm", ...}, ...], ...}, ...]\n(env: extra environment of that job\'s worker processes)\nEach candidate runs as `python -u worker.py --job \'<job + candidate as json>\' extra...` in its own process (a failed vLLM\nengine leaves a dirty process); exit code 0 ends the job, anything else tries the next candidate (fp16 -> AWQ, vllm -> hf).\ntp 2 jobs run one at a time on both GPUs first; tp 1 jobs then run two at a time, one per GPU. No GPU: one at a time.\nAfter a candidate its Hugging Face cache folder is deleted (Kaggle disk) unless the next candidate uses the same repo, or KEEP_HF=1.\nEvery worker line is echoed with a [tag] prefix; a summary of which candidate won is written to multirun_summary.json.\n"""\nimport json\nimport os\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nLOCK = threading.Lock()\n\n\ndef log(msg):\n    with LOCK:\n        print(msg, flush=True)\n\n\ndef ngpu():\n    try:\n        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())\n    except FileNotFoundError:\n        return 0\n\n\ndef run_job(job, worker, extra, gpus):\n    for k, cand in enumerate(job["cands"]):\n        spec = dict(job, **cand, cand_index=k)\n        spec.pop("cands")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", LLM_BACKEND=cand.get("backend", "vllm"), **job.get("env", {}))\n        t0 = time.time()\n        log(f"[{job[\'tag\']}] start candidate {k}: {cand[\'repo\']} ({env[\'LLM_BACKEND\']}) on GPU \'{gpus}\'")\n        p = subprocess.Popen([sys.executable, "-u", worker, "--job", json.dumps(spec, ensure_ascii=False)] + extra,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        tail = []\n        for line in p.stdout:\n            line = line.rstrip()\n            tail = (tail + [line])[-15:]\n            if "it/s]" not in line and "s/it]" not in line:  # drop tqdm bars\n                log(f"[{job[\'tag\']}] {line}")\n        rc = p.wait()\n        nxt = job["cands"][k + 1]["repo"] if k + 1 < len(job["cands"]) else None\n        if os.environ.get("KEEP_HF") != "1" and (rc == 0 or nxt != cand["repo"]):  # keep the weights for a retry of the same repo\n            hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"\n            shutil.rmtree(hub / ("models--" + cand["repo"].replace("/", "--")), ignore_errors=True)\n        log(f"[{job[\'tag\']}] candidate {k} rc {rc} after {time.time() - t0:.0f}s")\n        if rc == 0:\n            return dict(tag=job["tag"], repo=cand["repo"], cand_index=k, ok=True, sec=round(time.time() - t0))\n    return dict(tag=job["tag"], ok=False, tail=tail)\n\n\ndef main():\n    argv = sys.argv[1:]\n    extra = argv[argv.index("--") + 1:] if "--" in argv else []\n    argv = argv[:argv.index("--")] if "--" in argv else argv\n    jobs = json.load(open(argv[0], encoding="utf-8"))\n    worker = argv[1]\n    if os.environ.get("ONLY"):\n        keep = os.environ["ONLY"].split(",")\n        jobs = [j for j in jobs if j["tag"] in keep]\n    n = ngpu()\n    log(f"[multirun] {len(jobs)} jobs, {n} GPU(s): {[j[\'tag\'] for j in jobs]}")\n    done = []\n    if n < 2:\n        done = [run_job(j, worker, extra, "0" if n else "") for j in jobs]\n    else:\n        done = [run_job(j, worker, extra, "0,1") for j in jobs if j.get("tp", 1) > 1]\n        q = queue.Queue()\n        for j in jobs:\n            if j.get("tp", 1) == 1:\n                q.put(j)\n\n        def drain(g):\n            while True:\n                try:\n                    j = q.get_nowait()\n                except queue.Empty:\n                    return\n                done.append(run_job(j, worker, extra, g))\n\n        ts = [threading.Thread(target=drain, args=(str(g),)) for g in range(2)]\n        [t.start() for t in ts]\n        [t.join() for t in ts]\n    json.dump(done, open("multirun_summary.json", "w"), indent=1)\n    log("[multirun] summary: " + json.dumps([(d["tag"], d.get("repo"), d["ok"]) for d in done]))\n\n\nif __name__ == "__main__":\n    main()\n',
    'issue_worker.py': '"""GS-Issue worker: one local LLM finds the issues of each claim and proposes edits (same prompt for every model).\n\n    INPUTS=layer_a.jsonl,d_edit.jsonl python -u issue_worker.py --job \'{"tag": "q3-4b", "repo": "Qwen/Qwen3-4B-Instruct-2507", "tp": 1}\'\n\nRuns inside a Kaggle notebook built by tools/kaggle/build_nb.py with llm.py and multirun.py of experiments/AG-1.\nGreedy, thinking off. Input rows: {id, claim, evidence}. Output: out/<tag>__<input stem>.jsonl, one row per input row:\n{id, raw, parsed (or null)}; out/<tag>_summary.json. Env: LIMIT=n (first n rows of each input, smoke), MAX_TOKENS (900).\nThe prompt is the D-EDIT labelling guide (benchmark/gs_eval_v2/LABELING_GUIDE.md section 2), shortened; the taxonomy\nis benchmark/gs_issue/taxonomy.py (embedded next to this file).\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport time\nfrom pathlib import Path\n\nfrom taxonomy import CODES, KINDS\n\nSYSTEM = "Bạn là kiểm toán viên soát xét tuyên bố môi trường trong báo cáo của doanh nghiệp Việt Nam. Bạn chỉ trả lời bằng JSON."\nPROMPT = """Đọc TUYÊN BỐ (và BẰNG CHỨNG nếu có). Tìm các vấn đề khiến tuyên bố có thể gây hiểu nhầm, rồi đề xuất sửa.\n\nMã vấn đề (chỉ dùng các mã này):\n{codes}\n\nBốn loại sửa được phép:\n{kinds}\n\nQuy tắc:\n- Chỉ nêu vấn đề có thật trong câu; câu đã đủ chỉ số, số, kỳ, năm gốc, phạm vi, nguồn và không nói quá thì "issues": [].\n- Câu không phải tuyên bố môi trường (tiêu đề, mục lục, câu tài chính, định nghĩa) thì "is_green_claim": false và không sửa.\n- BẰNG CHỨNG chỉ để hiểu bối cảnh (ví dụ: tổng phát thải thật ra tăng, số của công ty mẹ khác số hợp nhất, mục tiêu chưa đạt).\n- Không thêm bất kỳ con số nào không có trong tuyên bố. Thông tin chưa biết thì viết chỗ trống "[bổ sung: ...]".\n- Không đổi nghĩa, không biến mục tiêu thành thành tích, không xoá thông tin đúng.\n- "span" và "target_span" chép NGUYÊN VĂN một đoạn của tuyên bố.\n\nTUYÊN BỐ: {claim}\nBẰNG CHỨNG: {evidence}\n\nTrả lời đúng một đối tượng JSON:\n{{"is_green_claim": true, "issues": [{{"code": "...", "span": "..."}}], "edits": [{{"kind": "...", "rule_code": "...", "target_span": "...", "replacement": "..."}}], "rewritten_claim": "tuyên bố sau khi áp mọi sửa"}}"""\n\n\ndef parse(text):\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(text)):\n            ch = text[i]\n            if in_str:\n                esc = (ch == "\\\\" and not esc)\n                if ch == \'"\' and not esc:\n                    in_str = False\n                continue\n            if ch == \'"\':\n                in_str = True\n            depth += {"{": 1, "}": -1}.get(ch, 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--job")\n    job = json.loads(ap.parse_args().job)\n    tag = job["tag"]\n    inputs = [p for p in os.environ.get("INPUTS", "").split(",") if p]\n    rows = []\n    for path in inputs:\n        part = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]\n        if os.environ.get("LIMIT"):\n            part = part[: int(os.environ["LIMIT"])]\n        rows += [(Path(path).stem, r) for r in part]\n    print(f"[{tag}] {len(rows)} rows from {inputs}, repo {job[\'repo\']}", flush=True)\n    from llm import LLM\n    t0 = time.time()\n    m = LLM(job["repo"], max_len=job.get("max_len", 4096), tp=job.get("tp", 1), **job.get("kw", {}))\n    load_s = time.time() - t0\n    codes = "\\n".join(f"- {k}: {v}" for k, v in CODES.items())\n    kinds = "\\n".join(f"- {k}: {v}" for k, v in KINDS.items())\n    prompts = [[{"role": "system", "content": SYSTEM},\n                {"role": "user", "content": PROMPT.format(codes=codes, kinds=kinds, claim=r["claim"],\n                                                          evidence=(r.get("evidence") or "(không có)")[:900])}] for _, r in rows]\n    t1 = time.time()\n    raw = []\n    for i in range(0, len(prompts), 64):\n        raw += m.gen(prompts[i:i + 64], max_tokens=int(os.environ.get("MAX_TOKENS", "900")))\n        print(f"[{tag}] generated {len(raw)}/{len(prompts)} ({time.time() - t1:.0f}s)", flush=True)\n    gen_s = time.time() - t1\n    Path("out").mkdir(exist_ok=True)\n    bad, files = 0, {}\n    for (stem, r), text in zip(rows, raw):\n        obj = parse(text)\n        bad += obj is None\n        files.setdefault(stem, []).append({"id": r["id"], "raw": text, "parsed": obj})\n    for stem, out in files.items():\n        with open(f"out/{tag}__{stem}.jsonl", "w", encoding="utf-8") as fh:\n            for o in out:\n                fh.write(json.dumps(o, ensure_ascii=False) + "\\n")\n    summary = {"tag": tag, "repo": job["repo"], "rows": len(rows), "unparsed": bad, "load_s": round(load_s), "gen_s": round(gen_s),\n               "ntok_in": getattr(m, "ntok_in", None), "ntok_out": getattr(m, "ntok_out", None), "overflow": getattr(m, "overflow", None)}\n    json.dump(summary, open(f"out/{tag}_summary.json", "w"), ensure_ascii=False)\n    print(f"[{tag}] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'taxonomy.py': '"""GS-Issue taxonomy: issue codes, families, edit kinds, and the fix oracle (shared by generator, scorer and worker).\n\nThe codes are those of benchmark/gs_eval_v2/LABELING_GUIDE.md section 2 (D-EDIT), unchanged.\nA family groups codes the guide itself lets overlap (MISSING_SCOPE says "công ty mẹ hay tập đoàn",\nENTITY_SCOPE_AMBIGUOUS says the same): scoring reports strict codes and families.\n"""\nimport re\nimport unicodedata\n\nCODES = {\n    "MISSING_PERIOD": "Thiếu kỳ báo cáo / năm của số liệu",\n    "MISSING_BASELINE": "Tuyên bố thay đổi (giảm, tăng, tiết kiệm) không có năm gốc",\n    "MISSING_SCOPE": "Thiếu ranh giới (nhà máy nào, công ty mẹ hay tập đoàn, phạm vi phát thải 1/2/3)",\n    "NO_FIGURE": "Tuyên bố hiệu quả môi trường mà không có số liệu",\n    "VAGUE_TERM": "Từ mơ hồ hoặc quảng cáo: xanh, thân thiện môi trường, bền vững, không ngừng",\n    "ABSOLUTE_CLAIM": "Tuyệt đối hoá: 100 %, hoàn toàn, không phát thải, tốt nhất",\n    "OFFSET_NEUTRALITY": "Trung hòa / net zero dựa vào bù trừ, không nêu tỷ lệ bù trừ",\n    "FUTURE_AS_ACHIEVED": "Cam kết / mục tiêu diễn đạt như đã đạt",\n    "UNVERIFIED_LABEL_OR_CERT": "Nhắc nhãn, chứng nhận, giải thưởng mà không nêu tổ chức cấp, phạm vi, hiệu lực",\n    "NO_EVIDENCE_REF": "Tuyên bố định lượng không dẫn nguồn / phương pháp",\n    "HIDDEN_TRADEOFF": "Nêu một mặt tốt, che mặt xấu lớn hơn liên quan trực tiếp",\n    "IRRELEVANT_CLAIM": "Khoe điều hiển nhiên hoặc bắt buộc theo luật như một thành tích",\n    "INTENSITY_VS_ABSOLUTE": "Dùng cường độ giảm để ngụ ý tổng phát thải giảm",\n    "ENTITY_SCOPE_AMBIGUOUS": "Không rõ số của công ty mẹ hay tập đoàn",\n}\nFAMILY = {"MISSING_SCOPE": "SCOPE", "ENTITY_SCOPE_AMBIGUOUS": "SCOPE",\n          "VAGUE_TERM": "VAGUE", "NO_FIGURE": "VAGUE"}\n\n\ndef family(code: str) -> str:\n    return FAMILY.get(code, code)\n\n\nKINDS = {\n    "add_scope": "thêm kỳ, năm gốc, ranh giới, phạm vi còn thiếu, dạng chỗ trống [bổ sung: ...]",\n    "placeholder": "thay cụm mơ hồ / câu không có số bằng chỗ trống [bổ sung: chỉ số và giá trị]",\n    "evidence_ref": "thêm dẫn chiếu nguồn: [bổ sung: nguồn xác minh]",\n    "soften": "bỏ hoặc làm nhẹ lời nói quá, nói lại đúng mức",\n}\n# The kind the guide prescribes for each code (several are acceptable for some).\nKIND_OK = {\n    "MISSING_PERIOD": {"add_scope"}, "MISSING_BASELINE": {"add_scope"}, "MISSING_SCOPE": {"add_scope"},\n    "ENTITY_SCOPE_AMBIGUOUS": {"add_scope"}, "NO_FIGURE": {"placeholder"}, "VAGUE_TERM": {"placeholder", "soften"},\n    "ABSOLUTE_CLAIM": {"soften"}, "OFFSET_NEUTRALITY": {"soften"}, "FUTURE_AS_ACHIEVED": {"soften"},\n    "UNVERIFIED_LABEL_OR_CERT": {"add_scope", "evidence_ref"}, "NO_EVIDENCE_REF": {"evidence_ref"},\n    "HIDDEN_TRADEOFF": {"add_scope", "evidence_ref", "soften"}, "IRRELEVANT_CLAIM": {"soften"},\n    "INTENSITY_VS_ABSOLUTE": {"soften", "add_scope"},\n}\n\nPH = re.compile(r"[\\[(][^\\])]*(?:bổ sung|cần nêu|cần bổ sung|cần ghi)[^\\])]*[\\])]", re.I)\n# What a placeholder must mention to fill each missing slot.\nSLOT_WORDS = {\n    "period": r"kỳ|năm(?! gốc)|thời gian|giai đoạn",\n    "baseline": r"gốc|so sánh|mốc",\n    "scope": r"phạm vi|ranh giới|nhà máy|cơ sở|công ty mẹ|hợp nhất|tập đoàn|đơn vị",\n    "figure": r"số|giá trị|chỉ số|mức|định lượng",\n    "source": r"nguồn|trang|tài liệu|báo cáo|phương pháp|xác minh|kiểm toán|bảo đảm",\n    "issuer": r"tổ chức|cấp|hiệu lực|phạm vi|đơn vị",\n    "context": r"tổng|toàn bộ|diễn biến|bối cảnh|cùng kỳ|tuyệt đối|phát thải",\n    "offset": r"tỷ lệ|bù trừ|tín chỉ|sổ đăng ký",\n}\n\n\ndef fold(s: str) -> str:\n    s = unicodedata.normalize("NFC", s or "").lower()\n    return re.sub(r"\\s+", " ", s).strip()\n\n\nNUM = re.compile(r"\\d+(?:[.,]\\d+)*")\n\n\ndef numbers(s: str) -> set[str]:\n    return {re.sub(r"[.,]", "", x) for x in NUM.findall(s or "")}\n\n\ndef placeholders(text: str) -> list[str]:\n    return [fold(m.group(0)) for m in PH.finditer(text or "")]\n\n\ndef slot_filled(before: str, after: str, slot: str) -> bool:\n    """A new placeholder (not in the text before) names the slot."""\n    old = placeholders(before)\n    new = [p for p in placeholders(after) if p not in old]\n    return any(re.search(SLOT_WORDS[slot], p) for p in new)\n\n\ndef oracle(case: dict, before: str, after: str) -> dict:\n    """Did the edited text fix this case\'s planted issue, and keep everything else?\n\n    resolves: the slot is asked for (drop mutations) or the planted phrase is gone or softened (inject mutations).\n    keeps_numbers: every number of the text before is still there. new_numbers: numbers the edit invented.\n    """\n    o = case["oracle"]\n    a = fold(after)\n    if o["type"] == "all":  # several planted issues: every one must be fixed\n        parts = [oracle(dict(case, oracle=p), before, after) for p in o["parts"]]\n        return {"resolves": all(p["resolves"] for p in parts), "keeps_numbers": all(p["keeps_numbers"] for p in parts),\n                "new_numbers": sorted(set().union(*(p["new_numbers"] for p in parts)))}\n    if o["type"] == "slot":\n        res = slot_filled(before, after, o["slot"])\n    elif o["type"] == "phrase":\n        gone = fold(o["phrase"]) not in a\n        asked = any(re.search(SLOT_WORDS[o.get("slot", "context")], p) for p in placeholders(after) if p not in placeholders(before))\n        res = gone or asked\n        if o.get("forbid"):  # e.g. FUTURE_AS_ACHIEVED: "đã giảm" must not remain\n            res = res and not re.search(o["forbid"], a)\n        if o.get("require"):\n            res = res and bool(re.search(o["require"], a))\n        if o.get("alt"):  # another acceptable fix, e.g. "theo quy định" for IRRELEVANT_CLAIM\n            res = res or bool(re.search(o["alt"], a))\n    else:\n        res = False\n    before_n, after_n = numbers(before), numbers(after)\n    droppable = numbers(o.get("phrase", ""))  # softening "100 %" may drop the 100\n    return {"resolves": bool(res), "keeps_numbers": (before_n - droppable) <= after_n,\n            "new_numbers": sorted(after_n - before_n)}\n',
    'jobs_32b.json': '[\n {\n  "tag": "q3-32b-awq",\n  "tp": 2,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-32B-AWQ"\n   }\n  ]\n }\n]',
    'gsi_layer_b1.jsonl': '{"id": "B-A-3db1442fbda33dc1", "claim": "ESG risks, opportunities and issues were considered throughout the research process via proprietary quality reviews.", "evidence": "67 of 185 investments in one Overlay Fund lacked an ESG quality-review score at the relevant time, nearly 25% of net assets. The SEC order found that prospectus language could lead a reasonable investor to believe all holdings had been reviewed, when they had not."}\n{"id": "B-A-e7988516f9ecb7b4", "claim": "Tự hào ra mắt Bộ sưu tập \'GreenLife 2026\': 100% thân thiện với môi trường, chế tác hoàn toàn từ sợi tự nhiên và hướng tới tương lai trung hòa carbon!", "evidence": ""}\n{"id": "B-A-649807f6b758955d", "claim": "Testing with municipal recycling facilities validated that K-Cup pods could be effectively recycled.", "evidence": "Two large recycling companies reported significant concerns about commercial feasibility and did not intend to accept the pods. The SEC order found the 2019 and 2020 statements incomplete and inaccurate because the negative feedback was omitted."}\n{"id": "B-A-136d376055d39c92", "claim": "Báo cáo phát triển bền vững Điểm nhấn Tổng quan về báo cáo 140 Mô hình phát triển bền vững của Traphaco 141 Định hướng chiến lược phát triển bền vững 142", "evidence": "báo cáo phát triển bền vững 04. 140 Tổng quan về báo cáo 141 Mô hình phát triển bền vững của Traphaco 142 Định hướng chiến lược phát triển bền vững BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Tổng quan về báo cáo Mô hình phát triển bền vững của Traphaco Cách tiếp cận với các vấn đề bền vững của Traphaco xuất phát từ mục tiêu tăng"}\n{"id": "B-A-741cd360ed607081", "claim": "Là dược phẩm được sản xuất trên nền tảng chuỗi cung ứng xanh từ nguồn nguyên liệu đến công nghệ sản xuất sạch, hệ thống phân phối, dịch vụ thân thiện với môi trường.", "evidence": "2 Tăng cường hiệu quả hoạt động thông qua Đầu tư công nghệ đạt chuẩn cao nhất ngành dược Việt Nam nhằm gia tăng hiệu quả sử dụng tài nguyên, và ứng dụng nhiều hơn các công nghệ sạch và thân thiện với môi trường; Hệ thống Quản trị hiện đại dựa trên nền tảng công nghệ thông tin tích hợp. 1 Thúc đẩy tăng trưởng kinh tế bền vững, hiệu quả và dài hạn; đã sở hữu 8 vùng trồng dược liệu đạt chứng nhận GACP-WHO. Đây là nền tảng quan trọng để củng cố chất lượng nguyên liệu đầu vào, tăng tính chủ động nguồn cung và phù hợp định hướng phát triển “xanh” của hệ thống. 1 2 3 Công ty duy trì hệ thống quản lý chất lượng, duy trì chứng nhận ISO, GPs và thực hiện chuẩn hóa hệ thống quy trình, quy trình sản xuấ"}\n{"id": "B-A-1240e16a405a7b5e", "claim": "Bước vào nhiệm kỳ 2026–2030, cho một giai đoạn phát triển năng động mới Traphaco Top 10 Doanh nghiệp bền vững CSI 2025 và Ngôi sao CSI 2016-2025 đã sẵn sàng", "evidence": "TRAPHACO NĂM 2025 TRAPHACO - Báo cáo thường niên 2025 Thông điệp của Chủ tịch Hội đồng Quản trị Bước vào nhiệm kỳ 2026–2030, đã sẵn sàng cho một giai đoạn phát triển năng động mới Cấp nhà Ngôi sao CSI 2016-2025 nước Doanh nghiệp vì Người lao động Doanh nghiệp xuất sắc và bền vững châu Á năm 2022 Giải thưởng WIPO Thương hiệu Quốc gia (Vietnam"}\n{"id": "B-A-496e776d20dcb624", "claim": "Đây không chỉ là nâng cấp bền vững, đưa Traphaco tiến gần hơn đến mục tiêu dẫn đầu ngành dược phẩm logistics, mà là nâng cấp toàn bộ năng lực cạnh tranh của Traphaco.", "evidence": "Bên cạnh đó, việc chủ động tái cấu trúc hệ thống phân phối, tăng cường đầu tư R&D, đẩy mạnh chuyển đổi số và nâng cao năng lực quản trị chất lượng đã tạo nền tảng vững chắc để Công ty đáp ứng yêu cầu tuân thủ và nâng cao năng lực cạnh tranh. Qua đó, Traphaco không chỉ duy trì tăng trưởng ổn định mà còn từng bước khẳng định vai trò tiên phong trong việc thích ứng với chuẩn mực mới của ngành dược, hướng tới phát triển bền vững và đồng hành cùng chiến lược quốc gia đến năm 2030, tầm nhìn 2045. Tốc độ thực thi Tăng trưởng bằng năng lực thực thi Điều thị trường chờ đợi ở Traphaco trong giai đoạn tới không chỉ là tăng trưởng quy mô, mà là một bước bứt phá về tầm vóc. Không chỉ là kết quả kinh doan"}\n{"id": "B-A-b41acfb71e80d254", "claim": "Để đạt được chứng nhận BE, mỗi sản phẩm phải trải qua quy trình nghiên cứu, thử nghiệm và thẩm định khắt khe, kéo dài nhiều năm với chi phí lớn và tỷ lệ rủi ro cao.", "evidence": "đồng thời phát triển vùng trồng dược liệu đạt chuẩn GACP-WHO nhằm kiểm soát chất lượng từ đầu chuỗi giá trị, giảm rủi ro nguyên liệu và bảo đảm ổn định sản xuất. Trong lĩnh vực nghiên cứu – phát triển, Công ty tăng cường quản trị rủi ro về tiến độ và chất lượng thông qua chuẩn hóa quy trình, ứng dụng AI trong nghiên cứu sinh khả dụng, tương đương sinh học và đa dạng hóa danh mục sản phẩm cho kênh OTC sản phẩm mới là chiến lược Công tác nghiên cứu 150% 119% 120% 18 19 6 Sản phẩm triển khai thị trường Vượt kế hoạch năm Thẩm định tiêu chuẩn chính thức Đạt kế hoạch năm"}\n{"id": "B-A-3ab25941850d6084", "claim": "Đây không chỉ là nền tảng cho tăng trưởng bền vững, mà còn là cam kết dài hạn của Traphaco đối với sức khỏe cộng đồng và niềm tin của người tiêu dùng Việt Nam.", "evidence": "Hệ thống Quản trị hiện đại dựa trên nền tảng công nghệ thông tin tích hợp. HIỆU QUẢ VÀ BỀN VỮNG TĂNG TRƯỞNG MỤC TIÊU 1 (SD8) Thúc đẩy tăng trưởng kinh tế bền vững, hiệu quả và dài hạn; Trong bối cảnh đó, phát triển sản phẩm mới không còn là cuộc đua về số lượng, mà là bài toán về chiều sâu khoa học, năng lực công nghệ và hệ thống kiểm soát chất lượng đủ vững để tạo dựng niềm tin lâu dài. Phát triển sản phẩm mới – Trụ cột tăng trưởng dài hạn Năm 2025, Traphaco tiếp tục khẳng định Đông dược – Đổi mới trên nền tảng cốt lõi lựa chọn chiến lược nhất quán:"}\n{"id": "B-A-570bba8f335c7ebd", "claim": "Nhìn lại cả nhiệm kỳ vừa qua, dù đối mặt với thách thức nghiêm trọng từ đại dịch COVID-19 và sự bất ổn kinh tế vĩ mô, Traphaco vẫn duy trì đà tăng trưởng bền vững về cả doanh thu và lợi nhuận trong suốt 5 năm.", "evidence": "Nhìn rộng ra cả nhiệm kỳ 2021–2025, Traphaco đã duy trì được quỹ đạo tăng trưởng ổn định trong bối cảnh đầy biến động, với doanh thu thuần hợp nhất và lợi nhuận trước thuế đạt tốc độ tăng trưởng kép tương ứng là 6,8% và 5,4%; biên lợi nhuận gộp duy trì quanh mức 53,6%; tỷ lệ cổ tức tiền mặt được giữ vững ở mức 30% - 40%, đảm bảo lợi ích thiết thực cho cổ đông. QUẢN TRỊ DOANH NGHIỆP TRAPHACO - Báo cáo thường niên 2025 Báo cáo Hoạt động của HĐQT nhiệm kỳ 2021–2025 Kết quả nổi bật Nhiệm kỳ 2021–2025 là một giai đoạn đầy thách thức khi nền kinh tế và xã hội chịu tác động nặng nề từ dịch bệnh Covid-19, các xung đột địa chính trị và sự bất ổn của chuỗi cung ứng toàn cầu."}\n{"id": "B-A-5852096862d27e8c", "claim": "Asset, Daewoong Pharmaceutical cùng SCIC, Công ty sẽ tiếp tục phát triển bền vững và chinh phục những cột mốc mới.", "evidence": "7 Vì môi trường và thế hệ trẻ – Lan tỏa tư duy phát triển bền vững Hướng tới mục tiêu phát triển bền vững và Net Zero, năm 2025 Traphaco tiếp tục đồng hành cùng thế hệ trẻ thông qua Năng suất ở đây không chỉ là sản lượng; đó là năng suất trong quyết định, trong phối hợp, trong cách thức triển khai, trong sáng tạo ứng dụng chuyển đối số và trong hiệu quả cuối cùng. Với nền tảng đã được củng cố, với chiến lược đã được định hình, với những động lực tăng trưởng mới đang ngày càng rõ nét, chúng ta có đầy đủ cơ sở để tin tưởng rằng: Traphaco sẽ tiếp tục đột phá, phát triển nhanh hơn và tạo ra nhiều giá trị bền vững hơn cho cổ đông, nhà đầu tư, người lao động và cộng đồng trong giai đoạn mới."}\n{"id": "B-A-73ba3ed1c786b06f", "claim": "Trước những sóng gió của thị trường, Traphaco đã chứng minh được bản lĩnh, sức mạnh và giá trị bền vững của một thương hiệu dược phẩm hàng đầu Việt Nam.", "evidence": "CHUNG CỦA Nâng cao hiệu quả kinh doanh bằng quy mô kinh doanh và nâng cao năng suất chất lượng tại Traphaco nhằm nâng hiệu quả và gia tăng giá trị của Traphaco, trở thành Thương hiệu Xanh hàng đầu Việt Nam. 3 XÃ HỘI TĂNG TRƯỞNG 5 Tiên phong trong việc hưởng ứng cuộc vận động người Việt Nam ưu tiên dùng: sử dụng nguồn nguyên liệu Việt, tri thức Việt để phục vụ chăm sóc sức khỏe. HIỆU QUẢ VÀ BỀN VỮNG Đầu tư hoạt động nghiên cứu phát triển. Cùng với dự án nhà máy EU-GMP đang được chuẩn bị triển khai, chuỗi giá trị xanh và chính sách phát triển con người nhất quán, Traphaco đã sẵn sàng bước sang giai đoạn tăng trưởng mới, khẳng định vai trò doanh nghiệp dược phẩm hàng đầu Việt Nam. LỢI NHUẬN SAU"}\n{"id": "B-A-ac048202e44e986d", "claim": "Năm 2025, Ban Tổng giám đốc đã triển khai Nghị quyết Đại hội đồng cổ đông ngày 18/4/2025 và các chỉ đạo của Hội đồng quản trị tập trung vào tái cấu trúc toàn diện, hiện đại hóa vận hành, chuyển đổi số và nâng cao năng lực cốt lõi nhằm tạo nền tảng tăng trưởng bền vững trong dài hạn.", "evidence": "4% -9% 306 Việc triển khai đồng bộ tái cấu trúc tổ chức, hiện đại hóa logistics, chuyển đổi số toàn diện, nâng cao quản trị chất lượng và đẩy mạnh R&D đã tạo nền tảng vững chắc cho tăng trưởng bền vững. Công ty không chỉ giữ vững vị thế dẫn đầu Đông dược, mở rộng thành công mảng tân dược chất lượng cao, mà còn ghi nhận tăng trưởng tích cực trên các kênh bán hàng hiện đại và OTC vượt kế hoạch. TABLE TRIỂN VỌNG CỔ PHIẾU TRA NĂM 2026 Sẵn sàng tăng tốc trong năm 2026, Traphaco định hướng phát triển bền vững trên nền tảng đã được củng cố, với thông điệp hành động “Tốc độ thực thi – Năng suất bứt phá”, tập trung vào các chiến lược trọng tâm: 1 Phát triển danh mục sản phẩm theo định hướng “Đông dượ"}\n{"id": "B-A-838129b30348fde4", "claim": "Những nỗ lực không ngừng trong việc thích ứng với thị trường, nâng cao năng lực cạnh tranh và phát triển bền vững đã giúp Traphaco tiếp tục được ghi nhận qua nhiều giải thưởng danh giá trong năm 2025: Tóm tắt thông tin tài chính 2021 - 2025 Chỉ tiêu ĐVT 2021 2022 2023 2024 2025 CAGR 2021-2025 Kết quả kinh doanh", "evidence": "Việc phát triển đa kênh không chỉ mở rộng độ phủ thị trường mà còn giúp Công ty từng bước tích hợp bán hàng, thương hiệu và chăm sóc khách hàng trên cùng một hệ điều hành chiến lược. Trong bối cảnh đó, kết quả đạt được trong năm 2025 với tốc độ tăng trưởng kép (CAGR) doanh thu và lợi nhuận lần lượt đạt 5,3% và 5,56% đã khẳng định năng lực thích ứng và đổi mới của Traphaco. Việc kiên định phát triển Đông dược theo hướng cao cấp, đồng thời đẩy mạnh Tân dược chất lượng cao, không chỉ giúp nâng cao giá trị gia tăng mà còn đáp ứng xu hướng Đánh giá tình hình tài chính công ty năm 2025 Dù doanh thu tăng trưởng qua các năm, đặc biệt đạt 2.657 tỷ đồng năm 2025, biên lợi nhuận gộp vẫn được giữ vững ở"}\n{"id": "B-A-5d28790862a3502a", "claim": "Qua đó, Traphaco không chỉ duy trì tăng trưởng ổn định mà còn từng bước khẳng định vai trò tiên phong trong việc thích ứng với chuẩn mực mới của ngành dược, hướng tới phát triển bền vững và đồng hành cùng chiến lược quốc gia đến năm 2030, tầm nhìn 2045.", "evidence": "Việc phát triển đa kênh không chỉ mở rộng độ phủ thị trường mà còn giúp Công ty từng bước tích hợp bán hàng, thương hiệu và chăm sóc khách hàng trên cùng một hệ điều hành chiến lược. Trong bối cảnh đó, kết quả đạt được trong năm 2025 với tốc độ tăng trưởng kép (CAGR) doanh thu và lợi nhuận lần lượt đạt 5,3% và 5,56% đã khẳng định năng lực thích ứng và đổi mới của Traphaco. Việc kiên định phát triển Đông dược theo hướng cao cấp, đồng thời đẩy mạnh Tân dược chất lượng cao, không chỉ giúp nâng cao giá trị gia tăng mà còn đáp ứng xu hướng Đánh giá tình hình tài chính công ty năm 2025 Cùng với dự án nhà máy EU-GMP đang được chuẩn bị triển khai, chuỗi giá trị xanh và chính sách phát triển con ngườ"}\n{"id": "B-A-67965bb78cb0253b", "claim": "Dù doanh thu tăng trưởng qua các năm, đặc biệt đạt 2.657 tỷ đồng năm 2025, biên lợi nhuận gộp vẫn được giữ vững ở mức 53,59%, cho thấy năng lực kiểm soát chi phí hiệu quả và tính bền vững trong mô hình kinh doanh.", "evidence": "Nhìn rộng ra cả nhiệm kỳ 2021–2025, Traphaco đã duy trì được quỹ đạo tăng trưởng ổn định trong bối cảnh đầy biến động, với doanh thu thuần hợp nhất và lợi nhuận trước thuế đạt tốc độ tăng trưởng kép tương ứng là 6,8% và 5,4%; biên lợi nhuận gộp duy trì quanh mức 53,6%; tỷ lệ cổ tức tiền mặt được giữ vững ở mức 30% - 40%, đảm bảo lợi ích thiết thực cho cổ đông. EBITDA ổn định trong khoảng 420–459 tỷ đồng, phản ánh năng lực tạo lợi nhuận và dòng tiền bền vững từ hoạt động kinh doanh cốt lõi. Đáng chú ý, dù bối cảnh thị trường có nhiều biến động và áp lực chi phí gia tăng, Công ty vẫn duy trì biên lợi nhuận ở mức hợp lý, với EBIT margin dao động 13%–16% và EBITDA margin 17%–20%. Điều này cho th"}\n{"id": "B-A-74b33d58f0c48910", "claim": "Kết quả này đến từ việc kiểm soát chi phí hiệu quả, tối ưu cấu trúc tài chính và duy trì lợi thế biên lợi nhuận cao, tạo nền tảng vững chắc để Công ty tiếp tục củng cố vị thế và hướng tới tăng trưởng bền vững trong các giai đoạn tiếp theo.", "evidence": "Sự kết hợp hài hòa giữa hiệu quả hoạt động, chất lượng lợi nhuận và an toàn tài chính tạo nền tảng vững chắc để Traphaco duy trì khả năng sinh lời bền vững và sẵn sàng cho các giai đoạn tăng trưởng tiếp theo. TRA PMC IMP DBD DHG DMC OPC MKP Phân tích Dupont cho thấy, dù biên lợi nhuận ròng chịu áp lực thu hẹp, Công ty đã chủ động bù đắp bằng việc nâng cao hiệu suất sử dụng tài sản, thể hiện qua vòng quay tài sản cải thiện rõ rệt trong năm 2025. Điều này mở rộng quy mô trong giai đoạn trước. Trong thời gian tới, việc tiếp tục tập trung vào các dòng sản phẩm chiến lược, nâng cao chất lượng và tối ưu biên lợi nhuận sẽ là động lực quan trọng giúp Traphaco cải thiện hơn nữa các chỉ số sinh lời, c"}\n{"id": "B-A-1e7655bd58936dbe", "claim": "Ở chiều ngược lại, chính sách thanh toán với nhà cung cấp được điều tiết phù hợp, với số ngày phải trả duy trì ở mức cao (~188 ngày), góp phần cải thiện dòng tiền hoạt động và nâng cao hiệu quả sử dụng vốn lưu động, đồng thời vẫn đảm bảo duy trì uy tín và quan hệ hợp tác bền vững với các đối tác chiến lược.", "evidence": "Vòng quay các khoản phải thu có xu hướng giảm, tương ứng với số ngày phải thu bình quân tăng lên 48,17 ngày năm 2025, chủ yếu do Công ty chủ động điều chỉnh chính sách bán hàng, đặc biệt tại kênh ETC, nhằm duy trì năng lực cạnh tranh và mở rộng thị phần. Đây là sự đánh đổi có kiểm soát, góp phần củng cố quan hệ với khách hàng và thúc đẩy tăng trưởng doanh thu. Nhờ việc điều phối linh hoạt các cấu phần vốn lưu động, chu kỳ chuyển đổi tiền mặt được duy trì ở mức hợp lý Công tác quản lý ngân sách được nâng cao thông qua hệ thống báo cáo và cơ chế cảnh báo tự động khi chi tiêu vượt ngân sách, hỗ trợ Ban lãnh đạo kịp thời điều chỉnh và đảm bảo mục tiêu tài chính chung. Việc tự động hóa phân bổ ng"}\n{"id": "B-A-e7c461e49cea3584", "claim": "Trong giai đoạn 2021–2025, Công ty đã chủ động tái cơ cấu tài sản theo hướng linh hoạt và nâng cao hiệu quả sử dụng vốn.", "evidence": "Sự chuyển dịch này phản ánh định hướng tối ưu hóa mô hình vận hành, giảm áp lực chi phí cố định, đồng thời gia tăng tính thanh khoản và khả năng thích ứng với biến động thị trường. Việc duy trì tỷ trọng cao của tài sản lưu động cũng tạo điều kiện thuận lợi để Công ty chủ động nguồn vốn ngắn hạn, nâng cao hiệu quả quản trị dòng tiền và tận dụng cơ hội kinh doanh. Về cơ cấu nguồn vốn, Công ty tiếp tục khẳng định nền tảng tài chính an toàn và mức độ tự chủ cao, với vốn chủ sở hữu luôn duy trì trên 70% tổng nguồn vốn trong suốt giai đoạn. Sự kết hợp hài hòa giữa hiệu quả hoạt động, chất lượng lợi nhuận và an toàn tài chính tạo nền tảng vững chắc để Traphaco duy trì khả năng sinh lời bền vững và "}\n{"id": "B-A-669ed2888aae147c", "claim": "Đối với hàng tồn kho, Công ty tiếp tục tối ưu hóa cơ cấu tồn kho và nâng cao khả năng đáp ứng thị trường, thể hiện qua việc vòng quay tồn kho duy trì ổn định và số ngày tồn kho được kiểm soát ở mức hợp lý, đảm bảo cân đối giữa dự trữ an toàn và hiệu quả sử dụng vốn.", "evidence": "và ETC, qua đó nâng cao khả năng thích ứng với nhu cầu thị trường. Việc thành lập Công ty TNHH MTV Dược phẩm Traphaco ngoài việc đáp ứng yêu cầu của pháp luật, còn giúp phân định rõ chức năng giữa công tác giới thiệu sản phẩm, bán hàng và phân phối, qua đó giảm rủi ro chồng chéo, tăng cường kiểm soát tồn kho, công nợ và dòng tiền, đồng thời nâng cao hiệu quả giám sát chuỗi cung ứng. Bên cạnh đó, trung tâm xử lý đơn hàng được vận hành ổn định, góp phần chuẩn hóa quy trình tiếp nhận, xử lý và giao hàng, hạn chế sai sót và rủi ro gián đoạn phân phối. Tỷ lệ nợ phải trả được kiểm soát ở mức hợp lý (dao động 24–30%), cho thấy Công ty có sự cân đối linh hoạt giữa việc tận dụng đòn bẩy tài chính và "}\n{"id": "B-A-9810960b3d20935f", "claim": "4% -9% 306 Việc triển khai đồng bộ tái cấu trúc tổ chức, hiện đại hóa logistics, chuyển đổi số toàn diện, nâng cao quản trị chất lượng và đẩy mạnh R&D đã tạo nền tảng vững chắc cho tăng trưởng bền vững.", "evidence": "Trong bối cảnh các quy định về thuế và hóa đơn điện tử được siết chặt nhằm tăng cường minh bạch, Ban Điều hành Traphaco đã chủ động, linh hoạt và quyết liệt tận dụng môi trường cạnh tranh minh bạch để bứt phá. Năm 2025, Ban Tổng giám đốc đã triển khai Nghị quyết Đại hội đồng cổ đông ngày 18/4/2025 và các chỉ đạo của Hội đồng quản trị tập trung vào tái cấu trúc toàn diện, hiện đại hóa vận hành, chuyển đổi số và nâng cao năng lực cốt lõi nhằm tạo nền tảng tăng trưởng bền vững trong dài hạn. Đẩy mạnh Chuyển đổi số, ứng dụng công nghệ trong nâng cao năng suất điều hành và chất lượng quản trị Bên cạnh đó, việc chủ động tái cấu trúc hệ thống phân phối, tăng cường đầu tư R&D, đẩy mạnh chuyển đổi số"}\n{"id": "B-A-ba10d063fd7190ce", "claim": "Cùng với dự án nhà máy EU-GMP đang được chuẩn bị triển khai, chuỗi giá trị xanh và chính sách phát triển con người nhất quán, Traphaco đã sẵn sàng bước sang giai đoạn tăng trưởng mới, khẳng định vai trò doanh nghiệp dược phẩm hàng đầu Việt Nam.", "evidence": "Năng suất ở đây không chỉ là sản lượng; đó là năng suất trong quyết định, trong phối hợp, trong cách thức triển khai, trong sáng tạo ứng dụng chuyển đối số và trong hiệu quả cuối cùng. Với nền tảng đã được củng cố, với chiến lược đã được định hình, với những động lực tăng trưởng mới đang ngày càng rõ nét, chúng ta có đầy đủ cơ sở để tin tưởng rằng: Traphaco sẽ tiếp tục đột phá, phát triển nhanh hơn và tạo ra nhiều giá trị bền vững hơn cho cổ đông, nhà đầu tư, người lao động và cộng đồng trong giai đoạn mới. thành lập Công ty TNHH Một thành viên Dược phẩm Traphaco nhằm tối ưu hóa hệ thống phân phối; và kiện toàn Ban Điều hành để sẵn sàng cho giai đoạn tăng trưởng mới. Ngoài ra, dự án nâng cấp"}\n{"id": "B-A-79f9e078dd0f26aa", "claim": "Triển khai đánh giá và được cấp Chứng chỉ Halal (Chứng chỉ thực phẩm tuân thủ luật Hồi giáo cấp tháng 5/2025), Tuân thủ các quy định liên quan đến bảo tồn đa dạng sinh học, phát triển bền vững và chia sẻ công bằng lợi ích giữa các bên liên quan, hướng tới phù hợp với các tiêu chuẩn của Công ước Đa dạng sinh học.", "evidence": "Hoàn thành đánh giá và cấp chứng nhận Halal (Chứng chỉ thực phẩm tuân thủ luật Hồi giáo cấp tháng 5/2025), Được Chủ tịch UBND tỉnh Lào Cai tặng Cờ thi đua “Đơn vị xuất sắc trong phong trào thi đua thực hiện nghị quyết của ban thường vụ tỉnh uỷ về chiến lược phát triển Nông nghiệp hàng hoá tỉnh Lào Cai năm Về môi trường làm việc và tuân thủ Traphaco duy trì môi trường làm việc an toàn, kỷ cương và tuân thủ đầy đủ các quy định của pháp luật về lao động, bảo hiểm, an toàn vệ sinh lao động và các tiêu chuẩn liên quan. Song song với đó, Công ty từng bước ứng dụng công nghệ trong quản trị nhân sự, nâng cao hiệu quả quản lý hồ sơ, chấm công, tiền lương và đánh giá hiệu suất. Gắn kết con người Traph"}\n{"id": "B-A-1625154d9f41bb88", "claim": "Thường xuyên cập nhật, thử nghiệm để đưa vào ứng dụng các công nghệ sản xuất mới nhằm tiết kiệm năng lượng, nguyên nhiên liệu, bảo vệ môi trường và nâng cao hiệu quả sản xuất kinh doanh, góp phần tích cực xây dựng nền kinh tế xanh, kinh tế tuần hoàn Xúc tiến các điều kiện, phục vụ mục tiêu có đơn hàng xuất khẩu trong năm 2026 Tìm kiếm các cơ hội xuất khẩu sản phẩm ra thị trường quốc tế", "evidence": "Nâng cao hiệu quả kinh doanh bằng quy mô kinh doanh và nâng cao năng suất chất lượng tại Traphaco nhằm nâng hiệu quả và gia tăng giá trị của Traphaco, trở thành Thương hiệu Xanh hàng đầu Việt Nam. Tăng cường hiệu quả hoạt động thông qua Đầu tư công nghệ đạt chuẩn cao nhất ngành dược Việt Nam nhằm gia tăng hiệu quả sử dụng tài nguyên, và ứng dụng nhiều hơn các công nghệ sạch và thân thiện với môi trường; TABLE TRIỂN VỌNG CỔ PHIẾU TRA NĂM 2026 Sẵn sàng tăng tốc trong năm 2026, Traphaco định hướng phát triển bền vững trên nền tảng đã được củng cố, với thông điệp hành động “Tốc độ thực thi – Năng suất bứt phá”, tập trung vào các chiến lược trọng tâm: 1 Phát triển danh mục sản phẩm theo định hướn"}\n{"id": "B-A-7ad6d653dde703ac", "claim": "Với vai trò là đơn vị sản xuất trong hệ thống, Traphaco CNC hướng tới mục tiêu vận hành an toàn – tuân thủ – hiệu quả, đồng hành cùng chiến lược sản phẩm xanh và phát triển bền vững của Traphaco.", "evidence": "Công ty TNHH MTV Dược phẩm Traphaco cam kết vận hành hệ thống logistics theo hướng An toàn – Hiệu quả – Tuân thủ – Phát triển bền vững, đồng hành cùng chiến lược chung của Traphaco với tinh thần: Cam kết Mục tiêu chung Tốc độ thực thi Năng suất Bứt phá Công ty luôn chú trọng công tác tuân thủ pháp luật, đảm bảo các yêu cầu về GDP, quản lý chất lượng, quản trị rủi ro và an toàn trong vận hành. BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Định hướng chiến lược phát triển bền vững Căn cứ theo nội dung cụ thể của 17 mục tiêu phát triển bền vững của Liên hợp quốc (SDG), đồng thời tham chiếu với mục tiêu của Việt Nam (VSDG) và các vấn đề được xem xét tại Diễn đàn chính trị cấp c"}\n{"id": "B-A-8b1a179b349c75da", "claim": "Sáng tạo sản phẩm xanh chăm Thu nhập bình quân của người lao động năm 2026: Tăng ít nhất 5% so với năm 2025.", "evidence": "Trung thực Kết nối Chủ động Trách nhiệm DƯỢC PHẨM Sáng tạo Sản phẩm xanh chăm sóc sức khỏe con người Là doanh nghiệp Dược số 1 Việt Nam về tốc độ tăng trưởng doanh thu Thu nhập bình quân của người lao động năm 2025 đạt 22,8 triệu đồng/người/tháng tăng khoảng 2,2% so với năm 2024 (22,3 triệu đồng/người/tháng). Chấp hành nghiêm chỉnh các yêu cầu của pháp luật, đảm bảo công tác an toàn và chất lượng sản phẩm cung cấp ra thị trường, không có vi phạm nào trong năm 2025. Báo cáo kết quả hoạt động sản xuất kinh doanh 2025"}\n{"id": "B-A-143fb8a0df2bb444", "claim": "Kính thưa Quý vị, hành trình phía trước còn nhiều thử thách, nhưng với bản lĩnh của người con Tây Nguyên – kiên cường như cây K’nia, rạng rỡ như hoa cà phê – tôi tin tưởng rằng Bamepharm sẽ tiếp tục gặt hái được những thành công mới, mang lại giá trị gia tăng bền vững cho Cổ đông và sự an tâm tuyệt đối cho khách hàng.", "evidence": "Chính vì vậy, chiến lược của Bamepharm không chỉ dừng lại ở việc kinh doanh đơn thuần, mà còn là trách nhiệm đảm bảo an ninh y tế địa phương. Năm 1976, khi đất nước vừa dứt tiếng súng, giữa vùng đất Tây Nguyên còn hoang sơ và đầy rẫy những khó khăn về dịch bệnh, Bamepharm đã ra đời với một sứ mệnh giản đơn nhưng thiêng liêng: Đưa thuốc về bản. Khi đó, những tiền bối của chúng tôi đã đi bộ băng rừng, vượt thác để mang từng viên thuốc sốt rét, từng ống tiêm đến với bà con. Năng suất ở đây không chỉ là sản lượng; đó là năng suất trong quyết định, trong phối hợp, trong cách thức triển khai, trong sáng tạo ứng dụng chuyển đối số và trong hiệu quả cuối cùng. Với nền tảng đã được củng cố, với chiến"}\n{"id": "B-A-94797bbc3c4f6c81", "claim": "Sử dụng vốn an toàn và hiệu quả, đảm bảo lợi nhuận.", "evidence": "(năm 2025: 24,83 ngày), qua đó khẳng định năng lực quản trị tài sản ngắn hạn hiệu quả và đảm bảo dòng tiền phục vụ hoạt động kinh doanh ổn định. Công ty đang chủ động kiểm soát và tối ưu hiệu quả vận hành, hướng tới mục tiêu tăng trưởng bền vững. Đối với hàng tồn kho, Công ty tiếp tục tối ưu hóa cơ cấu tồn kho và nâng cao khả năng đáp ứng thị trường, thể hiện qua việc vòng quay tồn kho duy trì ổn định và số ngày tồn kho được kiểm soát ở mức hợp lý, đảm bảo cân đối giữa dự trữ an toàn và hiệu quả sử dụng vốn. Xây dựng nội quy, quy chế, quy trình, biện pháp bảo đảm an toàn vệ sinh lao động (ATVSLĐ), an toàn vệ sinh thực phẩm (ATVSTP), phòng chống cháy nổ (PCCN), trong công ty. Quản lý theo dõi"}\n{"id": "B-A-8d9440306822a7e0", "claim": "Công ty TNHH MTV Dược phẩm Traphaco cam kết vận hành hệ thống logistics theo hướng An toàn – Hiệu quả – Tuân thủ – Phát triển bền vững, đồng hành cùng chiến lược chung của Traphaco với tinh thần: Cam kết Mục tiêu chung Tốc độ thực thi Năng suất Bứt phá Công ty luôn chú trọng công tác tuân thủ pháp luật, đảm bảo các yêu cầu về GDP, quản lý chất lượng, quản trị rủi ro và an toàn trong vận hành.", "evidence": "7 TỐC ĐỘ THỰC THI NĂNG SUẤT BỨT PHÁ MỌI HOẠT ĐỘNG HƯỚNG THEO CHỦ ĐỀ NĂM: 8 Luôn đảm bảo: Năng suất - Chất lượng - An toàn - Hiệu quả - Tuân thủ pháp luật. 86 87 Nhận thức rõ rủi ro pháp lý là một trong những rủi ro trọng yếu, Traphaco tăng cường công tác quản trị tuân thủ thông qua việc thường xuyên cập nhật các quy định pháp luật mới, rà soát quy trình nội bộ và đánh giá mức độ phù hợp của mô hình hoạt động. Trong năm 2025, Công ty đã triển khai tái cấu trúc mạnh mẽ hệ thống phân phối theo hướng tập trung, tinh gọn đầu mối, thành lập Công ty TNHH MTV Dược phẩm Traphaco làm đầu mối logistics và phân phối, nhằm đảm bảo tuân thủ quy định pháp luật, nâng cao tính minh bạch và giảm thiểu rủi ro "}\n{"id": "B-A-0909ec6431a591d6", "claim": "Chi tiết: Các giải thưởng đạt được Huân chương lao động hạng nhất lần 2 (2022) Thương hiệu Quốc gia Việt Nam (2022, 2024) Top 10 Doanh nghiệp bền vững liên tiếp các năm,", "evidence": "Giải Vàng Chất lượng Quốc gia Top 5 Công ty Đông dược uy tín nhất Việt Nam Top 40 Thương hiệu công ty giá trị nhất Việt Nam - Forbes VN 2 Huân chương Lao động hạng Nhất Value) 7 lần liên tiếp Doanh nghiệp tiêu biểu vì sự phát triển dược liệu Việt Top 50 Thương hiệu dẫn đầu Việt Nam - Forbes VN Giải thưởng Chất lượng Quốc tế Châu Á Thái Bình Dương"}\n{"id": "B-A-8d42ce06ae9c4de9", "claim": "Công ty tiếp tục được vinh danh Top 10 Doanh nghiệp Bền vững Việt Nam 2025 và nhận giải CSI Star cho giai đoạn 2016–2025, khẳng định năng lực quản trị bền vững và tuân thủ các chuẩn mực quốc tế trong dài hạn.", "evidence": "Giải thưởng Ngôi sao CSI giai đoạn 2016-2025. Trên nền tảng kết quả hoạt động và định hướng phát triển bền vững, Traphaco tiếp tục được ghi nhận thông qua nhiều danh hiệu, giải thưởng uy tín trong nước và quốc tế, qua đó củng cố hình ảnh doanh nghiệp dược Việt Nam có thương hiệu mạnh, năng lực quản trị tốt, trách nhiệm xã hội cao và định hướng phát triển dài hạn rõ ràng: Với thành tích này, Traphaco cũng được vinh danh “Ngôi Sao CSI” giai đoạn 2016-2025. Bước vào nhiệm kỳ 2026–2030, cho một giai đoạn phát triển năng động mới Traphaco Top 10 Doanh nghiệp bền vững CSI 2025 và Ngôi sao CSI 2016-2025 đã sẵn sàng"}\n{"id": "B-A-c5be942d270913e8", "claim": "Thông qua việc triển khai đồng bộ các hoạt động cập nhật pháp luật, đánh giá tuân thủ, kiểm soát hợp đồng, tư vấn pháp lý và tham gia xây dựng chính sách, Traphaco tiếp tục củng cố năng lực quản trị rủi ro pháp lý, bảo vệ hiệu quả lợi ích hợp pháp của Công ty, đồng thời hỗ trợ thực hiện các mục tiêu phát triển bền vững trong năm 2026.", "evidence": "Nhận thức rõ rủi ro pháp lý là một trong những rủi ro trọng yếu, Traphaco tăng cường công tác quản trị tuân thủ thông qua việc thường xuyên cập nhật các quy định pháp luật mới, rà soát quy trình nội bộ và đánh giá mức độ phù hợp của mô hình hoạt động. Trong năm 2025, Công ty đã triển khai tái cấu trúc mạnh mẽ hệ thống phân phối theo hướng tập trung, tinh gọn đầu mối, thành lập Công ty TNHH MTV Dược phẩm Traphaco làm đầu mối logistics và phân phối, nhằm đảm bảo tuân thủ quy định pháp luật, nâng cao tính minh bạch và giảm thiểu rủi ro pháp lý phát sinh trong hoạt động phân phối. Việc triển khai đồng bộ quản trị rủi ro trên các phương diện chiến lược, pháp lý, tài chính và hoạt Trước bối cảnh t"}\n{"id": "B-A-86821299dba14566", "claim": "Việc triển khai đồng bộ quản trị rủi ro trên các phương diện chiến lược, pháp lý, tài chính và hoạt Trước bối cảnh trên, Traphaco xác định quản trị rủi ro hoạt động là nội dung xuyên suốt, gắn với mục tiêu tăng trưởng bền vững và chuẩn hóa vận hành.", "evidence": "Công ty tiếp tục duy trì mô hình quản trị rủi ro doanh nghiệp ba cấp độ, đảm bảo việc nhận diện, đánh giá và kiểm soát rủi ro được triển khai xuyên suốt, đồng bộ và gắn chặt với các hoạt động sản xuất – kinh doanh trong toàn hệ thống. QUẢN TRỊ RỦI RO CHIẾN LƯỢC Trong quản lý hợp đồng và giao dịch, Công ty tập trung soát xét pháp lý đối với các hợp đồng có giá trị lớn, hợp đồng mẫu và các giao dịch phức tạp, đồng thời đẩy mạnh chuẩn hóa hệ thống mẫu hợp đồng nhằm giảm thiểu rủi ro phát sinh và tối ưu nguồn lực pháp chế. Năm 2025 đánh dấu một giai đoạn nhiều biến động đối với ngành dược phẩm, khi doanh nghiệp đồng thời chịu tác động từ áp lực thị trường, thay đổi chính sách pháp luật và yêu cầ"}\n{"id": "B-A-badddca4d1a5f7cc", "claim": "phát triển nhóm Premium dựa trên 3 giá trị cốt lõi Tinh hoa – Hiện đại – Bền vững;", "evidence": "Công ty đã linh hoạt, sáng tạo, thích ứng với tình hình thị trường, giữ vững thị phần số 1 của nhóm Bổ não, Bổ gan, điều trị trĩ (Theo IQVIA). Đánh giá chung Nhóm Đông dược cao cấp đạt mức tăng trưởng ấn tượng 68% so với cùng kỳ, nổi bật là Boganic Premium (tăng 70%) và Cebraton Premium (tăng 63%). Phát triển danh mục sản phẩm mới dựa trên nền tảng khoa học – công nghệ và quản lý chất lượng chặt chẽ, coi đây là động lực cốt lõi cho tăng trưởng bền vững và nâng tầm thương hiệu dược Việt. Đông dược tiếp tục giữ vai trò trụ cột trong chiến lược phát triển của Traphaco, không chỉ về doanh thu mà còn về bản sắc thương hiệu."}\n{"id": "B-A-2fc79ea5dc2225d6", "claim": "tham chiếu với mục tiêu của Việt Nam (VSDG) và các vấn đề được xem xét tại Diễn đàn chính trị cấp cao năm 2017, đối với tư cách 1 doanh nghiệp trong lĩnh vực sản xuất và kinh doanh dược phẩm, Traphaco lựa chọn và tích hợp các mục tiêu sau trong định hướng chiến lược phát triển bền vững doanh nghiệp giai đoạn 2021-2025:", "evidence": "Là điển hình tiêu biểu của phát triển bền vững tại Việt Nam, Traphaco luôn hướng tới phát triển nền kinh tế xanh với 3 mục tiêu: Hiệu quả kinh tế cao gắn với trách nhiệm xã Trong năm 2025, doanh nghiệp dược Việt Nam phải đối mặt với nhiều khó khăn như kinh tế tăng trưởng chậm; lạm phát cao ở các nền kinh tế lớn và căng thẳng địa chính trị gia tăng. Năm 2025 đánh dấu một giai đoạn nhiều biến động đối với ngành dược phẩm, khi doanh nghiệp đồng thời chịu tác động từ áp lực thị trường, thay đổi chính sách pháp luật và yêu cầu ngày càng cao về năng lực vận hành. Trước bối cảnh đó, Traphaco kiên định triển khai quản trị rủi ro như một trụ cột trọng yếu trong hệ thống quản trị doanh nghiệp, được tí"}\n{"id": "B-A-5d9f99c8cb27e7a6", "claim": "Bảo vệ, tái tạo và khuyến khích sử dụng bền vững các hệ sinh thái trên cạn, quản lý tài nguyên rừng bền vững, chống sa mạc hóa, chống xói mòn đất và mất đa dạng sinh học SDG 2.", "evidence": "5 Bảo vệ, tái tạo và khuyến khích sử dụng bền vững các hệ sinh thái trên cạn, quản lý tài nguyên rừng bền vững, chống sa mạc hóa, chống xói mòn đất và mất đa dạng sinh học. 1 Chung tay cùng cộng đồng phòng chống các dịch bệnh trong năm. Tuân thủ quy định của Nhà nước. Đóng góp, hỗ trợ các hoạt động chăm sóc sức khỏe cho người bệnh và người dân cả nước. BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Định hướng chiến lược phát triển bền vững SDG 8. SDG 9. SDG 12. Xây dựng cơ sở hạ tầng vững chắc, đẩy mạnh công nghiệp hóa rộng mở và bền vững, khuyến khích đổi mới. Tối ưu hoá năng lực và hiệu quả quản trị doanh nghiệp và chuỗi giá trị Xanh."}\n{"id": "B-A-a1c5a44034b1e5e3", "claim": "Xây dựng cơ sở hạ tầng kiên cường, thúc đẩy công nghiệp hóa toàn diện và bền vững và đổi mới thúc đẩy SDG 14.", "evidence": "SDG 3. Đảm bảo cuộc sống khỏe mạnh cho cộng đồng SDG 9. Xây dựng cơ sở hạ tầng vững chắc, đẩy mạnh công nghiệp hóa rộng mở và bền vững, khuyến khích đổi mới SDG 13. Hành động ứng phó với biến đổi khí hậu BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Định hướng chiến lược phát triển bền vững SDG 8. SDG 9. SDG 12. Xây dựng cơ sở hạ tầng vững chắc, đẩy mạnh công nghiệp hóa rộng mở và bền vững, khuyến khích đổi mới. Tối ưu hoá năng lực và hiệu quả quản trị doanh nghiệp và chuỗi giá trị Xanh."}\n{"id": "B-A-53156fb6663fec8e", "claim": "Nâng cao hiệu quả kinh doanh bằng quy mô kinh doanh và nâng cao năng suất chất lượng tại Traphaco nhằm nâng hiệu quả và gia tăng giá trị của Traphaco, trở thành Thương hiệu Xanh hàng đầu Việt Nam.", "evidence": "CHUNG CỦA Nâng cao hiệu quả kinh doanh bằng quy mô kinh doanh và nâng cao năng suất chất lượng tại Traphaco nhằm nâng hiệu quả và gia tăng giá trị của Traphaco, trở thành Thương hiệu Xanh hàng đầu Việt Nam. 3 XÃ HỘI TĂNG TRƯỞNG 5 Tiên phong trong việc hưởng ứng cuộc vận động người Việt Nam ưu tiên dùng: sử dụng nguồn nguyên liệu Việt, tri thức Việt để phục vụ chăm sóc sức khỏe. HIỆU QUẢ VÀ BỀN VỮNG Đầu tư hoạt động nghiên cứu phát triển. TABLE TRIỂN VỌNG CỔ PHIẾU TRA NĂM 2026 Sẵn sàng tăng tốc trong năm 2026, Traphaco định hướng phát triển bền vững trên nền tảng đã được củng cố, với thông điệp hành động “Tốc độ thực thi – Năng suất bứt phá”, tập trung vào các chiến lược trọng tâm: 1 Phát tri"}\n{"id": "B-A-b07a286c33031a93", "claim": "MỤC TIÊU 2 (SD12) Đảm bảo Tiêu dùng và sản xuất có trách nhiệm MỤC TIÊU 3 (SD9) Xây dựng cơ sở hạ tầng vững chắc, đẩy mạnh công nghiệp hóa rộng mở và bền vững, khuyến khích đổi mới", "evidence": "SDG 3. Đảm bảo cuộc sống khỏe mạnh cho cộng đồng SDG 9. Xây dựng cơ sở hạ tầng vững chắc, đẩy mạnh công nghiệp hóa rộng mở và bền vững, khuyến khích đổi mới SDG 13. Hành động ứng phó với biến đổi khí hậu BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Định hướng chiến lược phát triển bền vững SDG 8. SDG 9. SDG 12. Xây dựng cơ sở hạ tầng vững chắc, đẩy mạnh công nghiệp hóa rộng mở và bền vững, khuyến khích đổi mới. Tối ưu hoá năng lực và hiệu quả quản trị doanh nghiệp và chuỗi giá trị Xanh."}\n{"id": "B-A-d6347256809ae4a7", "claim": "Đảm bảo được sự bảo tồn của các hệ sinh thái, đa dạng sinh học, chống sa mạc hóa, phục hồi các vùng đất và đất bị thoái hóa.", "evidence": "5 Bảo vệ, tái tạo và khuyến khích sử dụng bền vững các hệ sinh thái trên cạn, quản lý tài nguyên rừng bền vững, chống sa mạc hóa, chống xói mòn đất và mất đa dạng sinh học. 1 Chung tay cùng cộng đồng phòng chống các dịch bệnh trong năm. Tuân thủ quy định của Nhà nước. Đóng góp, hỗ trợ các hoạt động chăm sóc sức khỏe cho người bệnh và người dân cả nước. SDG 1. Xoá nghèo dưới mọi hình thức, ở mọi nơi SDG 15. Bảo vệ, tái tạo và khuyến khích sử dụng bền vững các hệ sinh thái trên cạn, quản lý tài nguyên rừng bền vững, chống sa mạc hóa, chống xói mòn đất và mất đa dạng sinh học SDG 2. Chấm dứt nạn đói, đạt được an ninh lương thực và cải thiện dinh dưỡng và thúc đẩy nông nghiệp bền vững"}\n{"id": "B-A-6c3186fd1adb40af", "claim": "MỤC TIÊU 6 (SD13) Hành động Ứng phó với biến đổi khí hậu Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO.", "evidence": "2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. CHUNG TAY BẢO VỆ MÔI TRƯỜNG 4 Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp lý, có ý thức các nguồn năng lượng và tài nguyên thiên nhiên. GACP-WHO, nâng tổng số vùng trồng đạt chứng nhận GACP-WHO mà công ty sở hữu lên 8 dược liệu, qua đó góp phần gia tăng tính chủ động về nguồn nguyên liệu và chất lượng đầu vào. Với vai trò là đơn vị sản xuất trong hệ thống, Traphaco CNC hướng tới mục tiêu vận hành an toàn – tuân thủ – hiệu quả, đồng hành cùng chiến lược sản phẩm xanh và "}\n{"id": "B-A-75b3046b91739aea", "claim": "Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người.", "evidence": "2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. CHUNG TAY BẢO VỆ MÔI TRƯỜNG 4 Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp lý, có ý thức các nguồn năng lượng và tài nguyên thiên nhiên. Vì vậy đối với hoạt động nội bộ, Traphaco chú trọng kiểm soát việc tiêu thụ nhiên liệu, nguyên liệu trong sản xuất và xử lý chất thải tại các nhà máy – đảm bảo đạt các tiêu chuẩn môi trường cho phép; giảm thiểu việc sử dụng nguồn năng lượng không thể tái tạo và tài nguyên thiên nhiên qua đó góp phần giảm thiểu cường độ phát thải khí nhà"}\n{"id": "B-A-8324c0dc28d9887f", "claim": "3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người.", "evidence": "Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. 144 145 Vì vậy đối với hoạt động nội bộ, Traphaco chú trọng kiểm soát việc tiêu thụ nhiên liệu, nguyên liệu trong sản xuất và xử lý chất thải tại các nhà máy – đảm bảo đạt các tiêu chuẩn môi trường cho phép; giảm thiểu việc sử dụng nguồn năng lượng không thể tái tạo và tài nguyên thiên nhiên qua đó góp phần giảm thiểu cường độ phát thải khí nhà kính. Với cam kết” Con đường sức khỏe xanh”, định kỳ mỗi năm"}\n{"id": "B-A-a6fe83ded025e78f", "claim": "CHUNG CỦA Nâng cao hiệu quả kinh doanh bằng quy mô kinh doanh và nâng cao năng suất chất lượng tại Traphaco nhằm nâng hiệu quả và gia tăng giá trị của Traphaco, trở thành Thương hiệu Xanh hàng đầu Việt Nam.", "evidence": "Nâng cao hiệu quả kinh doanh bằng quy mô kinh doanh và nâng cao năng suất chất lượng tại Traphaco nhằm nâng hiệu quả và gia tăng giá trị của Traphaco, trở thành Thương hiệu Xanh hàng đầu Việt Nam. Tăng cường hiệu quả hoạt động thông qua Đầu tư công nghệ đạt chuẩn cao nhất ngành dược Việt Nam nhằm gia tăng hiệu quả sử dụng tài nguyên, và ứng dụng nhiều hơn các công nghệ sạch và thân thiện với môi trường; TABLE TRIỂN VỌNG CỔ PHIẾU TRA NĂM 2026 Sẵn sàng tăng tốc trong năm 2026, Traphaco định hướng phát triển bền vững trên nền tảng đã được củng cố, với thông điệp hành động “Tốc độ thực thi – Năng suất bứt phá”, tập trung vào các chiến lược trọng tâm: 1 Phát triển danh mục sản phẩm theo định hướn"}\n{"id": "B-A-bdf3f2327370f726", "claim": "Là điển hình tiêu biểu của phát triển bền vững tại Việt Nam, Traphaco luôn hướng tới phát triển nền kinh tế xanh với 3 mục tiêu: Hiệu quả kinh tế cao gắn với trách nhiệm xã Trong năm 2025, doanh nghiệp dược Việt Nam phải đối mặt với nhiều khó khăn như kinh tế tăng trưởng chậm;", "evidence": "TRAPHACO trưởng kinh tế trong dài hạn của doanh nghiệp kết hợp với các mục tiêu phát triển xã hội và bảo vệ môi trường, từ đó đóng góp cho sự phát triển chung của cộng đồng. Báo Tăng trưởng luôn gắn liền tăng trưởng kinh tế với bảo vệ môi trường và trách nhiệm xã hội cáo này ghi lại những hoạt động quan trọng của Traphaco và các đơn vị thành viên kinh tế của Traphaco (sau đây gọi tắt là “Traphaco”) liên quan đến các vấn đề về bền vững ba yếu tố đóng vai trò nền tảng cho thành công lâu dài của trong năm 2025. Phạm vi báo cáo: Công ty cổ phần Traphaco BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Định hướng chiến lược phát triển bền vững Căn cứ theo nội dung cụ thể của 17 mục"}\n{"id": "B-A-32455f008f38f2d5", "claim": "Kết quả hoạt động sản xuất kinh doanh ổn định, chăm lo tốt đời sống người lao động, cùng với những hoạt động chăm sóc khách hàng nổi bật, Công ty Traphaco được các tổ chức trong và ngoài nước trao tặng nhiều giải thưởng có giá trị như: Top 10 Doanh nghiệp bền vững CSI 2025,", "evidence": "BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 ĐẦU TƯ XÃ HỘI Về phúc lợi và chăm lo đời sống người lao động TRÂN TRỌNG NHỮNG GIÁ TRỊ ĐÓNG GÓP CỦA NGƯỜI LAO ĐỘNG Giải thưởng Ngôi sao CSI giai đoạn 2016-2025. Trên nền tảng kết quả hoạt động và định hướng phát triển bền vững, Traphaco tiếp tục được ghi nhận thông qua nhiều danh hiệu, giải thưởng uy tín trong nước và quốc tế, qua đó củng cố hình ảnh doanh nghiệp dược Việt Nam có thương hiệu mạnh, năng lực quản trị tốt, trách nhiệm xã hội cao và định hướng phát triển dài hạn rõ ràng:"}\n{"id": "B-A-4e6188090c02d7ae", "claim": "Gắn kết con người Traphaco với mục tiêu phát triển bền vững", "evidence": "BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Tổng quan về báo cáo Mô hình phát triển bền vững của Traphaco Cách tiếp cận với các vấn đề bền vững của Traphaco xuất phát từ mục tiêu tăng BÁO CÁO PHÁT TRIỂN BỀN VỮNG TRAPHACO - Báo cáo thường niên 2025 Định hướng chiến lược phát triển bền vững Căn cứ theo nội dung cụ thể của 17 mục tiêu phát triển bền vững của Liên hợp quốc (SDG), đồng thời tham chiếu với mục tiêu của Việt Nam (VSDG) và các vấn đề được xem xét tại Diễn đàn chính trị cấp cao năm 2017, đối với tư cách 1 doanh nghiệp trong lĩnh vực sản xuất và kinh doanh dược phẩm, Traphaco lựa chọn và tích hợp các mục tiêu sau trong định hướng chiến lược phát triển bền vững doan"}\n{"id": "B-A-53bf00ec9d00a991", "claim": "Với quan điểm lấy con người làm trọng tâm và trân trọng những đóng góp của người lao động, Traphaco cam kết cùng đội ngũ cán bộ, công nhân viên xây dựng nền tảng vững chắc cho sự phát triển ổn định, bền vững, không ngừng nâng cao vị thế và hướng tới tương lai.", "evidence": "Năm 2025 đánh dấu bước chuyển quan trọng của Traphaco trên hành trình nâng tầm danh mục sản phẩm và khẳng định vị thế doanh nghiệp dược phát triển dựa trên khoa học và chất lượng. Đây không chỉ là nền tảng cho tăng trưởng bền vững, mà còn là cam kết dài hạn của Traphaco đối với sức khỏe cộng đồng và niềm tin của người tiêu dùng Việt Nam. KHÔNG PHẢI KHẨU HIỆU Chất lượng chặt chẽ trong một chiến lược thống nhất. Trong năm 2025, Traphaco tiếp tục ưu tiên bố trí nguồn lực nhằm hoàn thiện hệ thống chế độ, chính sách đối với người lao động theo hướng công bằng, minh bạch và gắn với hiệu quả sản xuất, kinh doanh. Công ty coi việc bảo đảm thu nhập, phúc lợi và điều kiện làm việc ổn định là nền tảng "}\n{"id": "B-A-a2fd498a01e299f8", "claim": "0 9 CO2 11 11 0 2 10 Yếu tố tâm sinh lý và éc-gô-nô-my 11 11 0 Các phương tiện cấp phát bảo hộ lao động phải đảm bảo tiêu chuẩn chất lượng và đúng quy cách phù hợp với người sử dụng.", "evidence": "TABLE | | TRAPHACO - Báo cáo thường niên 2025 Khỏe & CHỦ ĐỀ: an toàn ĐƯỢC ĐỀ CẬP TRONG THỎA THUẬN CHÍNH THỨC VỚI CÔNG ĐOÀN Người sử dụng lao động và đại diện tập thể lao động cùng nhau thực hiện ký kết thỏa ước lao động tập thể, trong đó nêu rõ các điều khoản về ATVSLĐ như sau: 1 Người sử dụng lao động có trách nhiệm trang bị đầy đủ phương tiện bảo hộ lao động đảm bảo an toàn lao động, vệ sinh lao động và cải thiện điều kiện lao động cho người lao động: Người trực tiếp sản xuất được cấp mỗi năm hai bộ quần áo blu, mũ, khẩu trang, dép, xà phòng, găng tay, ... tuỳ theo đặc điểm của từng loại công việc. 2 Các phương tiện cấp phát bảo hộ lao động phải đảm bảo tiêu chuẩn chất lượng và đúng quy cá"}\n{"id": "B-A-886af0fdd6d578bb", "claim": "Thông qua các hoạt động hỗ trợ đồng bào, khách hàng và người lao động vượt qua thiên tai, bão lũ, Traphaco tiếp tục khẳng định hình ảnh một doanh nghiệp phát triển bền vững, luôn gắn hoạt động sản xuất kinh doanh với trách nhiệm xã hội và sự sẻ chia với cộng đồng trong những thời điểm khó khăn nhất.", "evidence": "Đồng thời, trước những diễn biến thiên tai, bão lũ phức tạp tại nhiều địa phương, Công ty cũng kịp thời triển khai các hoạt động hỗ trợ thiết thực dành cho đồng bào, khách hàng và người lao động bị ảnh hưởng, qua đó tiếp tục khẳng định bản sắc của một doanh nghiệp dược Việt Nam phát triển bền vững vì con người. Tổng ngân sách đóng góp cho xã hội thông qua các chương trình, sáng kiến về hỗ trợ nhân đạo, phát triển cộng đồng, bảo vệ môi trường... của Traphaco trong năm 2025 là 3.462.371.000 ĐỒNG Các hoạt động này cho thấy sự đồng hành kịp thời, linh hoạt và thực chất của Công ty, không chỉ ở tinh thần sẻ chia mà còn ở những giải pháp cụ thể nhằm góp phần giảm bớt thiệt hại, hỗ trợ khôi phục ho"}\n{"id": "B-A-d0d1cfeb3cb92d42", "claim": "Với vai trò đối tác đồng hành bảng đề Dược phẩm, Traphaco mang đến bài toán truyền thông về “Con đường sức khỏe xanh”, chia sẻ câu chuyện phát triển bền vững của doanh nghiệp, đồng thời tạo điều kiện để sinh viên tiếp cận thực tế thông qua các hoạt động tham quan vùng trồng dược liệu đạt chuẩn GACP-WHO.", "evidence": "GACP-WHO, nâng tổng số vùng trồng đạt chứng nhận GACP-WHO mà công ty sở hữu lên 8 dược liệu, qua đó góp phần gia tăng tính chủ động về nguồn nguyên liệu và chất lượng đầu vào. Với vai trò là đơn vị sản xuất trong hệ thống, Traphaco CNC hướng tới mục tiêu vận hành an toàn – tuân thủ – hiệu quả, đồng hành cùng chiến lược sản phẩm xanh và phát triển bền vững của Traphaco. 2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. CHUNG TAY BẢO VỆ MÔI TRƯỜNG 4 Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp lý, có ý thức các nguồn năng"}\n{"id": "B-A-37cbfd026d959ac3", "claim": "Quá trình liên kết ko chỉ tạo việc làm cho các hộ gia đình mà còn góp phần thay đổi mối quan hệ trong gia đình họ một cách tích cực và bền vững, giúp cho họ có cuộc sống bình đẳng, có Chuỗi hoạt động cộng đồng xã hội năm 2025 tiếp tục khẳng định định hướng phát triển bền vững của Traphaco – nơi hiệu quả kinh doanh luôn song hành cùng trách nhiệm xã hội.", "evidence": "Thông qua các hoạt động hỗ trợ đồng bào, khách hàng và người lao động vượt qua thiên tai, bão lũ, Traphaco tiếp tục khẳng định hình ảnh một doanh nghiệp phát triển bền vững, luôn gắn hoạt động sản xuất kinh doanh với trách nhiệm xã hội và sự sẻ chia với cộng đồng trong những thời điểm khó khăn nhất. 162 163 TRAPHACO NĂM 2025 TRAPHACO - Báo cáo thường niên 2025 Dấu ấn năm 2025 Ngày 19/11/2025, Traphaco được công bố nằm trong Top 10 Doanh nghiệp Bền vững tại Việt Nam (CSI) năm 2025. Đây là lần thứ 8 Traphaco nhận danh hiệu uy tín này, khẳng định vị thế tiên phong trong hành trình kinh doanh bền vững và đóng góp tích cực vào sự phát triển của xã hội."}\n{"id": "B-A-f4a7951ec1b22dc0", "claim": "Các khía cạnh môi trường trong chuỗi cung ứng: Chất thải rắn, chất thải sinh hoạt, chất thải nguy hại, tài nguyên thiên nhiên, nước thải, ô nhiễm môi trường không khí, bụi, tiếng ồn....", "evidence": "Hợp đồng rác thải sinh hoạt ký với công ty CP xử lý chất thải xây dựng và đầu tư phát triển môi trường Hà Nội. Biện pháp quản lý chất thải nguy hại: Công ty đã thực hiện lập báo cáo Sở tài nguyên môi trường Hà Nội và được cấp Giấy phép môi trường số 102/ GPMT- UBND ngày 05/7/2023. Thường xuyên thu gom, phân loại, lưu giữ chất thải trong các thùng chứa riêng biệt, có nhận biết mã CTNH 2 Sắp xếp đơn hàng để giao trên một lộ trình/ tuyến thích hợp nhằm tích kiệm nguyên liệu tiêu hao, giảm phát thải khí ra môi trường Ứng với mỗi khía cạnh là các biện pháp quản lý 3 Ưu tiên lựa chọn vật liệu làm bao bì là loại tái chế/ vật liệu thân thiện với môi trường nhằm hạn chế sự tác động tới môi trường: Ph"}\n{"id": "B-A-222ccab1600a27da", "claim": "Vì vậy đối với hoạt động nội bộ, Traphaco chú trọng kiểm soát việc tiêu thụ nhiên liệu, nguyên liệu trong sản xuất và xử lý chất thải tại các nhà máy – đảm bảo đạt các tiêu chuẩn môi trường cho phép;", "evidence": "2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. CHUNG TAY BẢO VỆ MÔI TRƯỜNG 4 Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp lý, có ý thức các nguồn năng lượng và tài nguyên thiên nhiên. Hợp đồng rác thải sinh hoạt ký với công ty CP xử lý chất thải xây dựng và đầu tư phát triển môi trường Hà Nội. Biện pháp quản lý chất thải nguy hại: Công ty đã thực hiện lập báo cáo Sở tài nguyên môi trường Hà Nội và được cấp Giấy phép môi trường số 102/ GPMT- UBND ngày 05/7/2023. Thường xuyên thu gom, phân loại, lưu giữ chất thải trong"}\n{"id": "B-A-690164a4b4206386", "claim": "giảm thiểu việc sử dụng nguồn năng lượng không thể tái tạo và tài nguyên thiên nhiên qua đó góp phần giảm thiểu cường độ phát thải khí nhà kính.", "evidence": "2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. CHUNG TAY BẢO VỆ MÔI TRƯỜNG 4 Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp lý, có ý thức các nguồn năng lượng và tài nguyên thiên nhiên. Tài Dùng nguồn năng lượng sạch (gió, ánh Sử dụng thổ nhưỡng, nước... loại tài nguyên Sử dụng, khai thác và làm ô nhiễm các Sử dụng các loại tài nguyên thiên nhiên sáng mặt trời) có thể tái tạo nguồn tài nguyên nguyên hữu hạn"}\n{"id": "B-A-131762aeae1b1252", "claim": "Với cam kết” Con đường sức khỏe xanh”, định kỳ mỗi năm", "evidence": "Tỷ lệ bệnh nghề nghiệp (occupational disease rate): 0 Không phát hiện trường hợp bệnh nghề nghiệp trong kỳ khám sức khỏe định kỳ năm 2025. Sức khỏe Sức khỏe Sức khỏe Sức khỏe Sức khỏe Trung thực Kết nối Chủ động Trách nhiệm DƯỢC PHẨM Sáng tạo Sản phẩm xanh chăm sóc sức khỏe con người Là doanh nghiệp Dược số 1 Việt Nam về tốc độ tăng trưởng doanh thu"}\n{"id": "B-A-5bf4eab91682439d", "claim": "Sử dụng nguồn năng lượng không tái tạo Khả năng gây ra tác động bất lợi Tần suất xảy ra được đánh giá trên khoảng thời gian xảy ra sự cố.", "evidence": "Hậu quả của tác động bất lợi có thể gây ra đối với môi trường 2 Tiêu thụ nguyên liệu từ các nguồn tái tạo Mức độ ô nhiễm (S) Mức độ tác động Điểm 1 Điểm 2 Điểm 3 Điểm 4 4.392 kg ~130 Lít khoảng thời gian Có thể xảy ra trên 1 năm/lần Xảy ra từ hàng quý Xảy ra hàng tuần Xảy ra từ ngày đến năm đến hàng tháng đến tuần Khí gas phục vụ nấu nướng: 2.980 kg Khí gas phục vụ nghiên cứu sản phẩm: 1.412 kg"}\n{"id": "B-A-97592f68ea77e246", "claim": "Hậu quả của tác động bất lợi có thể gây ra đối với môi trường 2 Tiêu thụ nguyên liệu từ các nguồn tái tạo Mức độ ô nhiễm (S) Mức độ tác động Điểm 1 Điểm 2 Điểm 3 Điểm 4", "evidence": "2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. CHUNG TAY BẢO VỆ MÔI TRƯỜNG 4 Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp lý, có ý thức các nguồn năng lượng và tài nguyên thiên nhiên. Vì vậy đối với hoạt động nội bộ, Traphaco chú trọng kiểm soát việc tiêu thụ nhiên liệu, nguyên liệu trong sản xuất và xử lý chất thải tại các nhà máy – đảm bảo đạt các tiêu chuẩn môi trường cho phép; giảm thiểu việc sử dụng nguồn năng lượng không thể tái tạo và tài nguyên thiên nhiên qua đó góp phần giảm thiểu cường độ phát thải khí nhà"}\n{"id": "B-A-3d7aba5e06ed0176", "claim": "loại tài nguyên Sử dụng, khai thác và làm ô nhiễm các Sử dụng các loại tài nguyên thiên nhiên sáng mặt trời) có thể tái tạo nguồn tài nguyên nguyên hữu hạn", "evidence": "Vì vậy đối với hoạt động nội bộ, Traphaco chú trọng kiểm soát việc tiêu thụ nhiên liệu, nguyên liệu trong sản xuất và xử lý chất thải tại các nhà máy – đảm bảo đạt các tiêu chuẩn môi trường cho phép; giảm thiểu việc sử dụng nguồn năng lượng không thể tái tạo và tài nguyên thiên nhiên qua đó góp phần giảm thiểu cường độ phát thải khí nhà kính. Với cam kết” Con đường sức khỏe xanh”, định kỳ mỗi năm 2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 3 Giảm thiểu các hoạt động gây phát sinh chất thải và ô nhiễm môi trường, góp phần cải tạo môi trường và sức khỏe của con người. CHUNG TAY BẢO VỆ MÔI TRƯỜNG 4 Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp "}\n{"id": "B-A-10f78a35677eabcb", "claim": "SOP về ứng cứu sự cố tràn đổ nước thải SOP Hướng dẫn xử lý tràn đổ hóa chất và rò rỉ vi sinh vật Phòng chống bão lụt Chất thải và chất thải nguy hại Năm 2025 Công ty không có sự cố môi trường", "evidence": "TABLE SOP về ứng cứu sự cố tràn đổ nước thải | | | SOP Hướng dẫn xử lý tràn đổ hóa chất và rò rỉ vi sinh vật | | | Phòng chống bão lụt TABLE | | TRAPHACO - Báo cáo thường niên 2025 HẠN CHẾ TỐI ĐA Ban an toàn công ty đã xác định các nguy cơ gây ra sự cố môi trường trong các công đoạn sản xuất, SỰ CỐ MÔI TRƯỜNG kinh doanh: Tràn đổ nước thải, tràn đổ hóa chất, rò rỉ vi sinh vật, lũ lụt... Với mỗi tình huống cụ thể có các phương án phòng ngừa khác nhau, được quy định cụ thể trong “Quy Phòng ngừa, ứng phó và khắc phục sự cố môi trình ứng phó tình trạng khẩn cấp” trong đó xác định rõ một số tình huống khẩn cấp. trường (SCMT) là một chế định quan trọng được quy định ở Mục 3, Chương X của Luật SOP v"}\n{"id": "B-A-98a59ea0dcfeab53", "claim": "2 Nguyên phụ liệu thuốc thành phẩm hủy 29,375kg Thiêu đốt Công ty CP xử lý, tái chế chất thải công nghiệp Hòa Bình 3 Chất thải nguy hại 638,5kg Làm sạch, Thiêu đốt, chôn lấp.", "evidence": "STT Tên chất thải Phòng lượng (kg) Phương pháp xử lý Đơn vị xử lý TUÂN THỦ CÁC QUY ĐỊNH VỀ MÔI TRƯỜNG 1 Chất thải sinh hoạt 60,000 kg Thiêu đốt Công ty CP xử lý chất thải xây dựng và đầu tư phát triển môi trường Hà Nội Hệ thống hóa các văn bản luật liên quan được cập nhật liên tục lên mạng nội bộ của công ty và thông báo sự ảnh hưởng của văn bản đến các bộ phận liên quan. Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy định toàn công ty. Hợp đồng rác thải tái chế ký với Công ty CP xử lý, tái chế chất thải công nghiệp Hòa Bình."}\n{"id": "B-A-ef48c0d7c7c37855", "claim": "Biện pháp quản lý rác thải sinh hoạt và tái chế:", "evidence": "Hợp đồng rác thải sinh hoạt ký với công ty CP xử lý chất thải xây dựng và đầu tư phát triển môi trường Hà Nội. Biện pháp quản lý chất thải nguy hại: Công ty đã thực hiện lập báo cáo Sở tài nguyên môi trường Hà Nội và được cấp Giấy phép môi trường số 102/ GPMT- UBND ngày 05/7/2023. Thường xuyên thu gom, phân loại, lưu giữ chất thải trong các thùng chứa riêng biệt, có nhận biết mã CTNH Hệ thống hóa các văn bản luật liên quan được cập nhật liên tục lên mạng nội bộ của công ty và thông báo sự ảnh hưởng của văn bản đến các bộ phận liên quan. Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy định toàn công ty. Hợp đồng rác thải tái chế ký với Công ty CP xử lý, tái chế"}\n{"id": "B-A-a6bc603492097df2", "claim": "Phân loại tại nguồn, phổ biển và thực hiện quy định phân loại chất thải tại mỗi bộ phận, phòng ban.", "evidence": "Giao Bộ phận chuyên trách thực hiện nhiệm vụ theo dõi và quản lý chất thải nguy hại theo quy trình. Nhà thầu vào công ty làm việc được phổ biến và tuân thủ theo quy trình quản lý chất thải của công ty. 178 179 2 Sắp xếp đơn hàng để giao trên một lộ trình/ tuyến thích hợp nhằm tích kiệm nguyên liệu tiêu hao, giảm phát thải khí ra môi trường Ứng với mỗi khía cạnh là các biện pháp quản lý 3 Ưu tiên lựa chọn vật liệu làm bao bì là loại tái chế/ vật liệu thân thiện với môi trường nhằm hạn chế sự tác động tới môi trường: Phân loại chất thải thông thường và nguy hại tại nguồn thải"}\n{"id": "B-A-b8f70e54d7d8b77c", "claim": "Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy định toàn công ty.", "evidence": "Giao Bộ phận chuyên trách thực hiện nhiệm vụ theo dõi và quản lý chất thải nguy hại theo quy trình. Nhà thầu vào công ty làm việc được phổ biến và tuân thủ theo quy trình quản lý chất thải của công ty. 178 179 An toàn, Sức khỏe Công ty có quy trình đánh giá hệ thống nước thải và cán bộ chuyên trách theo dõi, vận hành hệ thống xử lý nước thải hàng ngày, kiểm tra chất lượng nước thải sau xử lý và có biện pháp cải tiến kịp thời. Có sổ ghi chép theo dõi hàng ngày việc vận hành hệ thống. PHỤ LỤC và Môi trường Hệ thống xử lý nước thải: Hệ thống thu gom, xử lý nước thải được xây dựng theo công nghệ sinh học, công suất thiết kế 30 m3/ngày đêm."}\n{"id": "B-A-2d60f3d75d2bce5c", "claim": "Hợp đồng rác thải sinh hoạt ký với công ty CP xử lý chất thải xây dựng và đầu tư phát triển môi trường Hà Nội.", "evidence": "STT Tên chất thải Phòng lượng (kg) Phương pháp xử lý Đơn vị xử lý TUÂN THỦ CÁC QUY ĐỊNH VỀ MÔI TRƯỜNG 1 Chất thải sinh hoạt 60,000 kg Thiêu đốt Công ty CP xử lý chất thải xây dựng và đầu tư phát triển môi trường Hà Nội Hệ thống hóa các văn bản luật liên quan được cập nhật liên tục lên mạng nội bộ của công ty và thông báo sự ảnh hưởng của văn bản đến các bộ phận liên quan. Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy định toàn công ty. Hợp đồng rác thải tái chế ký với Công ty CP xử lý, tái chế chất thải công nghiệp Hòa Bình."}\n{"id": "B-A-bb0139bf3431d0f1", "claim": "Biện pháp quản lý chất thải nguy hại: Công ty đã thực hiện lập báo cáo Sở tài nguyên môi trường Hà Nội và được cấp Giấy phép môi trường số 102/ GPMT- UBND ngày 05/7/2023.", "evidence": "TC, Nghiền Công ty cổ phần môi trường đô thị và công nghiệp Bắc Sơn Công ty tuân thủ các quy định pháp luật về môi trường và thực hiện theo hướng dẫn của cơ quan quản lý môi trường. Biện pháp quản lý rác thải sinh hoạt và tái chế: 2 Sắp xếp đơn hàng để giao trên một lộ trình/ tuyến thích hợp nhằm tích kiệm nguyên liệu tiêu hao, giảm phát thải khí ra môi trường Ứng với mỗi khía cạnh là các biện pháp quản lý 3 Ưu tiên lựa chọn vật liệu làm bao bì là loại tái chế/ vật liệu thân thiện với môi trường nhằm hạn chế sự tác động tới môi trường: Phân loại chất thải thông thường và nguy hại tại nguồn thải"}\n{"id": "B-A-5dc9bc2ba4a7321a", "claim": "Giao Bộ phận chuyên trách thực hiện nhiệm vụ theo dõi và quản lý chất thải nguy hại theo quy trình.", "evidence": "An toàn, Sức khỏe Công ty có quy trình đánh giá hệ thống nước thải và cán bộ chuyên trách theo dõi, vận hành hệ thống xử lý nước thải hàng ngày, kiểm tra chất lượng nước thải sau xử lý và có biện pháp cải tiến kịp thời. Có sổ ghi chép theo dõi hàng ngày việc vận hành hệ thống. PHỤ LỤC và Môi trường Hệ thống xử lý nước thải: Hệ thống thu gom, xử lý nước thải được xây dựng theo công nghệ sinh học, công suất thiết kế 30 m3/ngày đêm. Hệ thống hóa các văn bản luật liên quan được cập nhật liên tục lên mạng nội bộ của công ty và thông báo sự ảnh hưởng của văn bản đến các bộ phận liên quan. Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy định toàn công ty. Hợp đồng rá"}\n{"id": "B-A-e44082a68aa6610c", "claim": "Nhà thầu vào công ty làm việc được phổ biến và tuân thủ theo quy trình quản lý chất thải của công ty.", "evidence": "1 Nguồn tiếp nhận nước thải: Nước thải sau xử lý xả ra hệ thống thoát nước chung của khu vực thuộc Phường Yên Sở, Hà Nội. 2 Nhà thầu phải đảm bảo nhân viên làm việc tại Công ty đủ năng lực, nhận thức và tuân thủ các quy định tại Công ty. Tiêu chuẩn áp dụng với nước thải: QCTĐHN 02:2014/BTNMT: Quy chuẩn kỹ thuật về Nước thải Công nghiệp trên địa bàn Thủ đô Hà Nội; ĐỐI VỚI NHÀ THẦU KHI LÀM VIỆC TẠI CÔNG TY CỔ PHẦN TRAPHACO BM75/01 BH/SĐ: 22/07/16 Chế độ xả thải: Liên tục Phương thức xả thải: Tự chảy Nhà thầu phải tuân thủ các quy định về An toàn vệ sinh lao động, môi trường."}\n{"id": "B-A-fe8e2e7d9e4bcb3d", "claim": "Từ khi thành lập đến nay, Tập đoàn đã liên tục mở rộng hệ sinh thái thép từ thượng nguồn đến hạ nguồn, nâng cao năng lực sản xuất, không ngừng tập trung nghiên cứu nâng cao chất lượng sản phẩm, cho ra các mác sản phẩm mới ngày càng chuyên sâu, phức tạp, đáp ứng rộng hơn các loại nhu cầu về thép cho hạ tầng đất nước.", "evidence": "Việc đầu tư nâng gấp đôi công suất nhằm đáp ứng nhu cầu ngày càng cao của thị trường trong nước và mở rộng thị trường xuất khẩu. Với ưu thế chủ động được phôi nguyên liệu đầu vào do chính Tập đoàn sản xuất, Hòa Phát thực hiện khép kín chuỗi sản xuất, kiểm soát chất lượng tối đa và truy xuất được nguồn gốc sản phẩm. Dự kiến, Công ty sẽ hoàn thành lắp đặt các dây chuyền mới và ra sản phẩm vào quý 3/2025. QUẢN TRỊ DOANH NGHIỆP Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chúng tôi không ngừng theo đuổi chất lượng vượt trội gắn liền với đổi mới bền vững, thông qua việc cải tiến công nghệ và quy trình sản xuất để nâng cao hiệu suất sản phẩm, nghiên cứu và mở rộng dải sản phẩm, nâng"}\n{"id": "B-A-3e90e361c6507c57", "claim": "Tôi rất vinh dự giới thiệu Báo cáo phát triển bền vững năm 2025 của Tập đoàn, như một minh chứng cho nỗ lực thấu hiểu sâu sắc và ghi nhận những mối quan tâm của các bên liên quan.", "evidence": "Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, đảm bảo hài hòa lợi ích của các bên liên quan trên cùng một con thuyền, hướng tới sự phát triển bền vững. Đặc biệt, Tập đoàn Hòa Phát đã xây dựng được mối quan hệ đối tác bền vững, lâu dài, tin tưởng như người một nhà với các đại lý bán hàng song hành cùng Tập đoàn từ những ngày đầu thành lập. Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt đ"}\n{"id": "B-A-7a8e9f3e5708fe6b", "claim": "Theo đó, xanh hóa sản xuất, phát triển con người, quản trị và đổi mới là ba ưu tiên hàng đầu.", "evidence": "HÒA PHÁT ĐẠT ĐƯỢC SỰ TIN YÊU CỦA KHÁCH HÀNG Trụ cột XANH HÓA SẢN XUẤT PHÁT TRIỂN CON NGƯỜI QUẢN TRỊ VÀ ĐỔI MỚI Chuyển đổi quy trình sản xuất thép theo hướng ít carbon, sử dụng tài nguyên hiệu quả và vận hành PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 CHUYỂN ĐỔI SỐ TRONG SẢN XUẤT VÀ QUẢN TRỊ DOANH NGHIỆP PHƯƠNG PHÁP QUẢN LÝ PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI E S G MÔI TRƯỜNG: Ứng dụng công nghệ số giúp giám sát và tối ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào;"}\n{"id": "B-A-6d276d2bd4386298", "claim": "Chúng tôi nhận thức về trách nhiệm của mình trong việc tham gia lộ trình giảm phát thải quốc gia và đang tích cực thực hành các giải pháp phù hợp với điều kiện kinh doanh, đặc biệt trong lĩnh vực thép, để đồng hành cùng Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050.", "evidence": "BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI [GRI 302-1, 302-2, 305-1, 305-2, 305-4] Nhận thức rõ Biến đổi Khí hậu là thách thức toàn cầu, và hưởng ứng cam kết Net Zero 2050 của Việt Nam tại COP26, Tậ"}\n{"id": "B-A-e9ca4e260fdf873f", "claim": "Tập đoàn hướng tới từng bước chuyển đổi mô hình sản xuất và quản trị, gắn liền với cải tiến công nghệ, tối ưu hiệu quả sử dụng tài nguyên, và giảm thiểu phát thải ra môi trường.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 NGUYÊN VẬT LIỆU: CÔNG NGHỆ: CHUYỂN DỊCH THEO HƯỚNG TĂNG TỶ TRỌNG NGUYÊN VẬT LIỆU PHÁT THẢI THẤP HƯỚNG TỚI CÔNG NGHỆ SẢN XUẤT THÉP XANH Tập đoàn Hòa Phát cam kết tuân thủ đầy đủ các quy định của pháp luật về môi trường như Luật bảo vệ môi trường năm 2020, các Nghị định và Thông tư hướng dẫn có liên quan, thông qua việc triển khai hệ thống quản lý môi trường hiệu quả bao gồm: Hành động giảm thiểu và thích ứng với biến đổi khí hậu HỆ THỐNG: Đầu tư công nghệ hiện đại Hòa Phát ứng dụng công nghệ tiên tiến để giảm thiểu tối đa các tác động bất lợi đến môi trường."}\n{"id": "B-A-b21548daca48fd96", "claim": "Đến nay, cả hai khu liên hợp lớn nhất của Tập đoàn tại Dung Quất và Hải Dương đều có mức độ tự chủ điện năng rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia.", "evidence": "Sản lượng phát điện tự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. Tổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và Dung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện hiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. Nguồn điện xanh này được tích hợp vào hệ thống cấp điện nội bộ, thay thế một phần điện từ lưới quốc gia. Đây là giải pháp sạch, không phát thải tại chỗ, góp phần lớn vào mục tiêu trung hòa carbon trong dài hạn. Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai các giải pháp chủ chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng cao"}\n{"id": "B-A-9ab349ae5bbcddcc", "claim": "Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 TƯƠNG LAI NGÀNH THÉP CỦA CHÚNG TÔI Tập đoàn Hòa Phát là tập đoàn đa ngành hoạt động trong nhiều lĩnh vực bao gồm Thép, Nông nghiệp, Bất động sản, Điện máy gia dụng. Quản lý môi trường Xét theo lĩnh vực hoạt động, ngành sản xuất thép là lĩnh vực phát thải lớn nhất, đóng góp chủ yếu vào tổng phát thải của toàn tập đoàn. Nhóm nông nghiệp, bất động sản và điện máy gia dụng có mức phát thải thấp hơn đáng kể. Nhóm nguồn phát thải trực tiếp từ quá trình sản xuất chiếm tỷ trọng rõ rệt, phản ánh đặc thù ngành nghề và quy mô sản xuất của tập đoàn."}\n{"id": "B-A-510ec5c5a12bf0da", "claim": "Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế.", "evidence": "Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh tế ; nghiên cứu và tìm kiếm các giải pháp điều chỉnh cơ cấu tiêu thụ nguyên liệu và qua đó giảm phát sinh chất thải. Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng các hệ thống tuần hoàn khép kín, tái chế nước và xử lý triệt để để tuân thủ quy chuẩn xả thải. QUẢN LÝ MÔI TRƯỜNG 5.2 Tái chế, tái sử dụng chất thải rắn thành nguyên vật liệu quay lại quá trình sản xuất Các giải pháp tái chế, tái sử dụng nội bộ tập trung vào việc phân loại và xử lý các chất thải ngay tại nguồn phát sinh. Theo đó, các chất thải rắn chứa sắt sẽ được th"}\n{"id": "B-A-2069f2b05f47b750", "claim": "ESG Hòa Phát tích hợp các mục tiêu phát triển bền vững (SDGs) vào chiến lược kinh doanh, phù hợp với định hướng phát triển xanh của Chính phủ Việt Nam và cam kết trung hòa carbon đến năm 2050.", "evidence": "BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG Định hướng chiến lược về Phát triển bền vững. Phê duyệt "}\n{"id": "B-A-eb6e80cb57f20a1f", "claim": "Chúng tôi tin rằng với tinh thần sáng tạo, quyết tâm và trách nhiệm, Hòa Phát sẽ không chỉ duy trì vị thế tiên phong trong ngành công nghiệp thép, mà còn đóng góp tích cực cho sự phát triển thịnh vượng và bền vững của đất nước.", "evidence": "Hội đồng Quản trị định hướng Hòa Phát sẽ luôn ở trong tâm thế tiến về phía trước. Tập đoàn cam kết giai đoạn 2025 - 2030 sẽ duy trì mức tăng trưởng không dưới 15% mỗi năm, đóng góp tích cực vào sự phát triển chung của nền kinh tế Việt Nam. Thay mặt Hội đồng Quản trị, tôi xin trân trọng cảm ơn sự tín nhiệm và ủng hộ của Quý vị cổ đông, cảm ơn sự hợp tác tích cực của khách hàng, đối tác. TABLE KỶ LỤC VỀ SỐ NỘP NGÂN SÁCH NHÀ NƯỚC Với quy mô hoạt động ngày càng lớn, Tập đoàn Hòa Phát đóng góp tích cực vào sự phát triển kinh tế xã hội của cả nước. Tổng nộp ngân sách Nhà nước trong năm 2024 đạt gần 13.500 tỷ đồng, cao nhất trong lịch sử 32 năm hoạt động và tăng 48% so với 2023. Theo báo Tuổi trẻ, "}\n{"id": "B-A-eba7c73d2de9da60", "claim": "MÔI TRƯỜNG nhóm thép và nông nghiệp đạt trên 99% Điểm nổi bật về phát triển bền vững năm 2025 Đóng góp gần 13.000 tỷ đồng vào ngân sách Nhà nước Số giờ đào tạo trung bình cho mỗi nhân Giải thưởng và vinh danh năm 2025 viên là 38,52 giờ mỗi năm Thành viên của các Hiệp hội ngành nghề Đóng góp hơn 120 tỷ đồng cho các chương trình phúc lợi xã hội.", "evidence": "TABLE | 3.1 ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025 Hoàn thành kiểm kê khí nhà kính theo tiêu 90% Tự chủ nhu cầu điện chuẩn ISO 14064-1:2018 cho toàn Tập tại Khu liên hợp Gang thép Hòa Phát đoàn và được BSI thẩm định số liệu . Dung Quất 98% Tái sử dụng nước Tái chế, tái sử dụng chất thải rắn MÔI TRƯỜNG tại Khu liên hợp Gang thép Hòa Phát 99% nhóm thép và nông nghiệp đạt trên Dung Quất Số giờ đào tạo trung bình cho mỗi nhân 13.000 tỷ đồng Đóng góp gần vào 38,52 giờ ngân sách Nhà nước viên là mỗi năm 120 tỷ đồng Đóng góp hơn cho các chương trình phúc lợi xã hội. XÃ HỘI 22% thành viên Tỷ lệ phụ nữ ở các cấp quản lý trong toàn Hội đồng Quản trị là thành viên độc lập. 35,85% Tập đoàn đạt vào năm"}\n{"id": "B-A-8e66afd106994238", "claim": "Hoàn thành kiểm kê khí nhà kính theo tiêu 90%", "evidence": "QĐ-TTg ban hành danh mục lĩnh vực, cơ sở phát thải khí nhà kính phải thực hiện kiểm kê khí nhà kính và nộp báo cáo kiểm kê khí nhà kính cấp cơ sở về cơ quan có thẩm quyền từ tháng 3/2025. Đến tháng 01/2025, BSI đã trao giấy kiểm tra, xác nhận, công bố khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 cho các công ty thành viên của Tập đoàn Hòa Phát tham gia kiểm kê. Việc hoàn thành kiểm kê khí nhà kính là bước đi quan trọng trong chiến lược phát triển bền vững của Tập đoàn. HÒA PHÁT HOÀN THÀNH KIỂM KÊ KHÍ NHÀ KÍNH TOÀN TẬP ĐOÀN BÁO CÁO THƯỜNG NIÊN - 2024 Ngày 11/1/2025, BSI - Tổ chức chứng nhận quốc tế hàng đầu của Vương quốc Anh đã trao giấy kiểm tra, xác nhận, công bố khí nhà kính theo tiêu ch"}\n{"id": "B-A-971f092f641cd3c5", "claim": "Thông qua việc tham gia các hiệp hội này, Tập đoàn tham gia đối thoại chính sách, chia sẻ tri thức và hợp tác trong các sáng kiến hướng tới bảo vệ môi trường, nâng cao an toàn lao động và thúc đẩy kinh tế tuần hoàn trong ngành thép.", "evidence": "Báo cáo đầy đủ định kỳ hằng năm cho các cơ quan quản lý về thuế, môi trường, y tế và an toàn lao động. Tham gia vào các tổ chức hiệp hội, như Hiệp hội Thép Việt Nam. Tham gia hội nghị, hội thảo chuyên ngành do các Bộ, Ban, Ngành tổ chức Thúc đẩy hợp tác, đổi mới công nghệ và vận động chính Thúc đẩy kết nối doanh nghiệp, đối thoại chính sách Đóng góp vào việc xây dựng các văn bản pháp luật liên đẩy sử dụng nguyên liệu hợp ESG trong hoạt động sách nhằm nâng cao năng lực cơ khí chế tạo và sản xuất và phát triển kinh tế bền vững; hỗ trợ hội viên thông qua đào tạo, xúc tiến thương quan đến lĩnh vực nông nghiệp, thúc đẩy các hoạt động chuyên ngành về chăn nuôi, thú y, trao đổi kỹ thuật và thúc đẩy"}\n{"id": "B-A-c1abfbd17623f291", "claim": "VIỆT NAM (VSA) NAM (VCCI) (VAMI) (HUBA) (AHAV) Hỗ trợ sản xuất thép bền vững, giảm phát thải và thúc Thúc đẩy quản trị doanh nghiệp có trách nhiệm và tích", "evidence": "TABLE HIỆP HỘI THÉP VIỆT NAM (VSA) Hỗ trợ sản xuất thép bền vững, giảm phát thải và thúc đẩy sử dụng nguyên liệu tuần hoàn. | PHÒNG THƯƠNG MẠI VÀ CÔNG NGHIỆP VIỆT NAM (VCCI) Thúc đẩy quản trị doanh nghiệp có trách nhiệm và tích hợp ESG trong hoạt động kinh doanh. | HIỆP HỘI DOANH NGHIỆP CƠ KHÍ VIỆT NAM (VAMI) Thúc đẩy hợp tác, đổi mới công nghệ và vận động chính sách nhằm nâng cao năng lực cơ khí chế tạo và sản xuất trong nước. | HIỆP HỘI DOANH NGHIỆP TP. HỒ CHÍ MINH (HUBA) Thúc đẩy kết nối doanh nghiệp, đối thoại chính sách và phát triển kinh tế bền vững; hỗ trợ hội viên thông qua đào tạo, xúc tiến thương mại và tư vấn chính sách với các cơ quan quản lý. | HỘI CHĂN NUÔI VIỆT NAM (AHAV) Đóng"}\n{"id": "B-A-d484554de38985e1", "claim": "Hỗ trợ sản xuất thép bền vững, giảm phát thải và thúc đẩy sử dụng nguyên liệu tuần hoàn.", "evidence": "Thúc đẩy hợp tác, đổi mới công nghệ và vận động chính Thúc đẩy kết nối doanh nghiệp, đối thoại chính sách Đóng góp vào việc xây dựng các văn bản pháp luật liên đẩy sử dụng nguyên liệu hợp ESG trong hoạt động sách nhằm nâng cao năng lực cơ khí chế tạo và sản xuất và phát triển kinh tế bền vững; hỗ trợ hội viên thông qua đào tạo, xúc tiến thương quan đến lĩnh vực nông nghiệp, thúc đẩy các hoạt động chuyên ngành về chăn nuôi, thú y, trao đổi kỹ thuật và thúc đẩy công nghệ chăn nuôi, xây dựng chuỗi liên kết tuần hoàn. BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính"}\n{"id": "B-A-7eb8099cc4d314cf", "claim": "Sản phẩm THÁNG 12: Phân kỳ 1, dự án Khu Thép Hòa Phát Dung ray đường sắt cao tốc đầu tiên dự kiến ra lò vào năm 2027, giúp Hòa Phát trở thành doanh nghiệp đầu tiên và duy nhất ở Đông Phát Hưng Yên, đánh dấu bước phát triển mới tối ưu hệ sinh thái thép Hòa Quất, triển khai Khu Liên nóng/năm, tổng vốn đầu tư dự kiến 85.000", "evidence": "QUÝ 1/2022 Khởi công dự án Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 2, công suất 5,6 triệu tấn thép cuộn cán nóng/năm, tổng vốn đầu tư dự kiến 85.000 tỷ đồng. NĂM 2024 THÁNG 12: Phân kỳ 1, dự án Khu liên hợp sản xuất gang thép TABLE | THÁNG 2/2016 THÁNG 1/2021 NĂM 2025 • Thành lập Công ty CP Phát triển Nông Lò cao số 4 – KLH gang thép Hòa Phát THÁNG 9: Dung Quất được đưa vào hoạt động, đánh nghiệp Hòa Phát, quản lý, chi phối hoạt động lò cao số 2 của dự án NĂM 2019 dấu việc hoàn thành toàn bộ dự án, nâng NĂM 2023 của tất cả các Công ty trong nhóm nông Dung Quất 2 đi vào hoạt tổng công suất thép thô của Hòa Phát lên nghiệp (gồm thức ăn chăn nuôi, chăn nuôi). Hai lò cao đầu tiên • TH"}\n{"id": "B-A-2dc3c6110256763d", "claim": "Phát Hưng Yên, đánh tối ưu hệ sinh thái thép Hòa Phân kỳ 1, dự án Khu", "evidence": "Sản phẩm THÁNG 12: Phân kỳ 1, dự án Khu Thép Hòa Phát Dung ray đường sắt cao tốc đầu tiên dự kiến ra lò vào năm 2027, giúp Hòa Phát trở thành doanh nghiệp đầu tiên và duy nhất ở Đông Phát Hưng Yên, đánh dấu bước phát triển mới tối ưu hệ sinh thái thép Hòa Quất, triển khai Khu Liên nóng/năm, tổng vốn đầu tư dự kiến 85.000 Hòa Phát lên 8,5 triệu tấn/ năm, lớn nhất khu vực Đông Nam Á. THÁNG 11/2020 Tập đoàn Hòa Phát bắt đầu cung cấp sản phẩm thép cuộn cán nóng thương mại ra thị trường. Sản phẩm giúp tối ưu hệ sinh thái thép Hòa Phát."}\n{"id": "B-A-deb391b4014efb34", "claim": "Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, đảm bảo hài hòa lợi ích của các bên liên quan trên cùng một con thuyền, hướng tới sự phát triển bền vững.", "evidence": "GIÁ TRỊ CỐT LÕI: Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng phát triển. Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, đảm bảo hài hòa lợi ích của các bên liên quan trên cùng một con thuyền, hướng tới sự phát triển bền vững. Đặc biệt, Tập đoàn Hòa Phát đã xây dựng được mối quan hệ đối tác bền vững, lâu dài, tin tưởng như người một nhà với các đại lý bán hàng song hành cùng Tập đoàn từ những ngày đầu thành lập. TRÁCH NHIỆM XÃ HỘI – CỘNG ĐỒNG CHẶNG ĐƯỜNG NỐI YÊU THƯƠNG: MẸ ĐỠ ĐẦU BÁO CÁO THƯỜNG NIÊN - 2024 Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng Phát triển. Điều này th"}\n{"id": "B-A-8342bdb722d60c13", "claim": "2 6 Điểm nổi bật về phát triển bền vững năm 2025 qua cách chúng tôi lựa chọn nguồn cung, tổ chức sản xuất, hợp tác cùng đối tác, chăm lo cho người 6.", "evidence": "3 6 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT PHÁT TRIỂN CON NGƯỜI 6.1 | Sức khoẻ và an toàn lao động 3.1 | Điểm nổi bật về phát triển bền vững năm 2025 6.2 | Quản trị nguồn nhân lực 3.2 | Giải thưởng và vinh danh năm 2025 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12"}\n{"id": "B-A-75a0a80cdd94a369", "claim": "Thép, với đặc tính bền vững, chịu lực cao, và dễ dàng tái chế, là xương sống của mọi nền kinh tế và kiến tạo nên môi trường sống của chúng ta.", "evidence": "Là một trong những tập đoàn công nghiệp hàng đầu Việt Nam, Hòa Phát coi quản trị tốt và đạo đức kinh doanh không chỉ là nền tảng của sự phát triển bền vững, mà còn là yếu tố cốt lõi để củng cố niềm tin của cổ đông, khách hàng, đối tác và xã hội. Tập đoàn coi đạo đức kinh doanh là nền tảng của quản trị bền vững, đóng vai trò then chốt trong việc kiểm soát rủi ro, bảo vệ uy tín doanh nghiệp và bảo đảm tuân thủ các tiêu chuẩn ESG trong bối cảnh ngành thép chịu áp lực ngày càng lớn từ thị trường, chuỗi cung ứng và các cơ chế điều chỉnh carbon. GIỚI TÍNH QUỐC TỊCH 100% NAM 100% VIỆT NAM QUẢN LÝ CHẤT LƯỢNG 7.2 Ngoài ra, bộ phận R&D của chúng tôi tại các nhà máy thép liên tục thực hiện nghiên cứu đ"}\n{"id": "B-A-ad46f3d47e9fd5f9", "claim": "Xây dựng và hạ tầng 52% Phương tiện vận tải khác Điểm nổi bật về phát triển bền vững năm 2025", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 10"}\n{"id": "B-A-69dc4cd69a6abf9a", "claim": "Sản phẩm kim loại Hoạt động kinh doanh tại Tập đoàn Đồ gia dụng 10% 2% Thiết bị cơ khí 16% Chuyển đổi năng lượng: là nguyên liệu quan trọng trong quá trình chuyển đổi năng lượng tái tạo (tua-bin gió, điện mặt trời), điện khí hóa, giao thông công cộng và nền kinh tế hydro.", "evidence": "Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo. Bên cạnh đó. Tập đoàn Hòa Phát nghiên cứu triển khai lắp đặt hệ thống điện mặt trời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy. Các nguồn nhiệt dư sinh ra trong quá trình sản xuất được thu hồi để phát điện. Đến nay, cả hai khu liên hợp lớn nhất của Tập đoàn tại Dung Quất và Hải Dương đều có mức độ tự chủ điện năng rất"}\n{"id": "B-A-600eb0c8547c4e30", "claim": "Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF).", "evidence": "Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn. Lượng phát thải carbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai đoạn nào của vòng đời. Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu thành thép để tạo vòng đời tiếp theo. Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát t"}\n{"id": "B-A-da3f6b4729ad809c", "claim": "Vòng đời của thép đi qua hai giai đoạn: gang lỏng, sau đó tiếp tục được tinh luyện trong lò thổi oxy nhằm loại bỏ tạp chất và điều chỉnh hàm lượng carbon để trở thành thép thành phẩm.", "evidence": "Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn. Lượng phát thải carbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai đoạn nào của vòng đời. Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu thành thép để tạo vòng đời tiếp theo. Giải thưởng và vinh danh năm 2025 thay vì trở thành rác thải, chúng được thu gom, phân tách và đưa vào lò điện để nấu Sản xuất thép chảy lại thành thép lỏng, tạo ra các sản phẩm thép mới.. Giai đoạn này gắn với công Thàn"}\n{"id": "B-A-596e682d4215a56c", "claim": "HÒA PHÁT trò nền tảng để cung cấp nguồn thép chất lượng cao cho các công trình hạ tầng và máy móc, tuy nhiên nó tiêu tốn một lượng lớn năng lượng và giải phóng nhiều khí thải CO2 do quá trình đốt cháy nhiên liệu hóa thạch.", "evidence": "Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật"}\n{"id": "B-A-c507da891b4b7862", "claim": "Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép", "evidence": "Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn. Lượng phát thải carbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai đoạn nào của vòng đời. Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu thành thép để tạo vòng đời tiếp theo. Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – tái chế thép). Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ"}\n{"id": "B-A-fc62e0cc280256b5", "claim": "Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép.", "evidence": "Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – tái chế thép). Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ than và giảm phát thải. Theo nghiên cứu của Hiệp hội Thép Thế giới (2021), việc tái chế một tấn thép phế có thể giúp giảm khoảng 1,5 tấn khí CO2. Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn. Lượng phát thải carbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai đoạn nào của vòng đời. Phát thải carbon phát sinh nhiều nhất ở gia"}\n{"id": "B-A-8300b77461440a89", "claim": "Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu.", "evidence": "Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF). Trong các lò cao làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế, BÁO CÁO PHÁT TRIỂN BỀN VỮNG công nghiệp, quặng sắt được hoàn nguyên và nấu chảy ở nhiệt độ cực cao để tạo ra Thép, với đặc tính bền vững, chịu lực cao, và dễ dàng tái chế, là xương sống của mọi nền kinh tế và kiến tạo nên môi trường sống của chúng ta. Thép không chỉ là vật liệu mà còn là nền tảng vật chất định hình nên mọi mặt của xã hội văn minh. Xây dựng và hạ tầng 52% Phương tiện vận tải khác Điểm nổi bật về phát triển bền vững"}\n{"id": "B-A-7607ca0788742c05", "claim": "Quy trình này tạo thành một vòng lặp kín vô tận, biến thép trở thành một trong những vật liệu cốt lõi của nền kinh tế tuần hoàn và phát triển bền vững.", "evidence": "Trong đó, ngành Thép giữ vai trò cốt lõi và là trụ cột chính trong mô hình kinh doanh. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT VAI TRÒ CỦA THÉP Vai trò của thép trong đời sống hiện đại là vô cùng thiết yếu và không thể thay thế. Đây chính là mắt xích quan trọng trong mô hình kinh tế tuần hoàn, góp phần thực hiện mục tiêu sản xuất thép xanh và bền vững. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu"}\n{"id": "B-A-9bd2adebd14e4eb7", "claim": "Hoạt động kinh doanh tại Tập đoàn Phế liệu thép sau tiêu dùng 100% có thể tái chế Tái sử dụng & tái sản xuất", "evidence": "Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – tái chế thép). Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ than và giảm phát thải. Theo nghiên cứu của Hiệp hội Thép Thế giới (2021), việc tái chế một tấn thép phế có thể giúp giảm khoảng 1,5 tấn khí CO2. Thành viên của các Hiệp hội ngành nghề Phát đang chủ động gia tăng dần tỷ lệ sử dụng thép phế liệu trong cả hai quy trình sản xuất. Trong tầm nhìn dài hạn, khi nguồn cung thép phế liệu trở nên dồi dào và dễ tiếp cận hơn, Hòa Phát định hướng đẩy mạnh hơn nữa tỷ trọng thép tái chế trong cơ cấu sản Hoạt động kinh doanh tại Tập đoàn xu"}\n{"id": "B-A-a5b32854f70a5fa2", "claim": "Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn.", "evidence": "Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF). Trong các lò cao làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế, BÁO CÁO PHÁT TRIỂN BỀN VỮNG công nghiệp, quặng sắt được hoàn nguyên và nấu chảy ở nhiệt độ cực cao để tạo ra Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu. Khi cá"}\n{"id": "B-A-da87a809c80e5738", "claim": "Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu thành thép để tạo vòng đời tiếp theo.", "evidence": "Ngược lại, EAF có mức phát thải thấp hơn, nhưng phụ thuộc vào nguồn nguyên liệu như thép phế, quặng sắt đã hoàn nguyên – những nguyên liệu có giá thành cao và nguồn cung hạn chế, phù hợp với các quốc gia phát triển, nơi ngành thép đã đi qua nhiều vòng đời tái chế. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Hiện nay, các biện pháp giảm phát thải trong sản xuất thép từ quặng Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – tái chế thép). Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ than và giảm phát thải. Theo nghiên cứu của Hiệp hội Thép Thế giới (2021), việc tái chế một tấn thép phế có thể g"}\n{"id": "B-A-ce4d7747a28fe4c2", "claim": "Hướng tới mục tiêu phát triển bền vững và giảm dấu chân carbon, Hòa", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG Định hướng chiến lược về Phát triển bền vững. Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít "}\n{"id": "B-A-ac6c8c99043c1d91", "claim": "Trong tầm nhìn dài hạn, khi nguồn cung thép phế liệu trở nên dồi dào và dễ tiếp cận hơn, Hòa Phát định hướng đẩy mạnh hơn nữa tỷ trọng thép tái chế trong cơ cấu sản Hoạt động kinh doanh tại Tập đoàn xuất để tối ưu hóa vòng đời thứ cấp.", "evidence": "Quy trình này tạo thành một vòng lặp kín vô tận, biến thép trở thành một trong những vật liệu cốt lõi của nền kinh tế tuần hoàn và phát triển bền vững. Hoạt động kinh doanh tại Tập đoàn Phế liệu thép sau tiêu dùng 100% có thể tái chế Tái sử dụng & tái sản xuất Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – tái chế thép). Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ than và giảm phát thải. Theo nghiên cứu của Hiệp hội Thép Thế giới (2021), việc tái chế một tấn thép phế có thể giúp giảm khoảng 1,5 tấn khí CO2."}\n{"id": "B-A-0ecc933d1a3b6a79", "claim": "Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững", "evidence": "MỤC LỤC BÁO CÁO PHÁT TRIỂN BỀN VỮNG NĂM 2025 TỔNG QUAN VỀ PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT CÁC LĨNH VỰC TRỌNG YẾU VỀ PHÁT TRIỂN BỀN VỮNG CÔNG TY CỔ PHẦN TẬP ĐOÀN HÒA PHÁT BÁO CÁO CỦA BAN GIÁM ĐỐC BÁO CÁO THƯỜNG NIÊN - 2024 PHÊ DUYỆT BÁO CÁO TÀI CHÍNH HỢP NHẤT Chúng tôi phê duyệt báo cáo tài chính hợp nhất cho năm kết thúc ngày 31 tháng 12 năm 2024 đính kèm."}\n{"id": "B-A-cedfb347c7282d21", "claim": "Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan.", "evidence": "QUẢN TRỊ NGUỒN PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT NHÂN LỰC Chúng tôi cam kết đảm bảo minh bạch, trách nhiệm giải trình và liêm chính trong mọi cấp độ tổ chức. Hệ thống quản trị được thiết kế để hỗ trợ tăng trưởng bền vững dài hạn và củng cố niềm tin của các bên liên quan. Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Khung quản trị bền vững của Tập đoàn được minh họa như sau: Gắn kết với các bên liên quan HỘI ĐỒNG QUẢN TRỊ"}\n{"id": "B-A-c339bc93609c7fc1", "claim": "Ban ESG được thành lập với sự tham gia của Ban Tổng Giám đốc, Ban Giám đốc các công ty thành viên cùng các ban chức năng như ban Tài chính, Pháp chế, Nhân sự, Truyền thông của Tập đoàn nhằm điều phối các hoạt động ESG và tích hợp các mục tiêu phát triển bền vững vào hoạt động vận hành hàng ngày của doanh nghiệp.", "evidence": "Đối với các vấn đề phát sinh từ người lao động, tổ chức tín dụng, cơ quan quản lý nhà nước hoặc khách hàng, đối tác kinh doanh, các thông tin này được truyền đạt đến Ban Giám đốc từng công ty thành viên để xem xét và phản hồi phù hợp. Các phòng, ban của Tập đoàn Hòa Phát cũng hỗ trợ và phối hợp với Ban Giám đốc công ty thành viên để phản hồi các bên liên quan trong phạm vi chuyên môn của phòng ban đó. Quản trị Doanh nghiệp và Quản lý rủi ro Phối hợp với Bộ phận Kiểm toán nội bộ, đưa ra các khuyến Tập đoàn Hòa Phát duy trì cơ chế trao đổi thường xuyên giữa Hội đồng Quản trị và Ban Tổng Giám đốc, giữa Ban Tổng Giám đốc và các công ty thành viên và phòng ban của Tập đoàn Hòa Phát nhằm đảm bảo c"}\n{"id": "B-A-d08082b724db6473", "claim": "Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG Gắn kết với các bên liên quan BƯỚC 4 tại 15 doanh nghiệp lớn cùng ngành trên thế giới, các mối quan tâm về phát triển bền vững đã được xác định.", "evidence": "Mức độ tác động đến doanh nghiệp được xác định dựa trên ảnh hưởng của các vấn đề ESG đối với hoạt động kinh doanh, tài chính và danh tiếng của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan kết hợp với đóng góp mà Hòa Phát mang lại cho xã hội và môi trường. Cách tiếp cận này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then chốt nhất. GẮN KẾT VỚI CÁC BÊN LIÊN QUAN 4.4 CHÍNH PHỦ VÀ CƠ QUAN QUẢN LÝ VẤN ĐỀ QUAN TÂM PHẢN HỒI CỦA TẬP ĐOÀN & KẾT QUẢ THỰC HIỆN 2025 PHƯƠNG THỨC TƯƠNG TÁC BÁO CÁO PHÁT TRIỂN BỀN VỮNG"}\n{"id": "B-A-60a8b92ce13a669d", "claim": "Các chủ đề có mức độ ưu tiên khác nhau sẽ có những tác động và định hướng vững (SASB), cùng các vấn đề trọng yếu phổ biến trong ngành thép và nông nghiệp toàn quản lý phù hợp trong chiến lược phát triển bền vững của Hòa Phát.", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN"}\n{"id": "B-A-00c9f19c32f65860", "claim": "Nhóm nghiên cứu đã sắp xếp các mối quan tâm ESG lại thành 12 chủ đề liên quan đến hoạt động của Tập đoàn để đưa vào bước ưu tiên tiếp theo thông qua tham vấn các BƯỚC 5 bên liên quan.", "evidence": "Mức độ tác động đến doanh nghiệp được xác định dựa trên ảnh hưởng của các vấn đề ESG đối với hoạt động kinh doanh, tài chính và danh tiếng của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan kết hợp với đóng góp mà Hòa Phát mang lại cho xã hội và môi trường. Cách tiếp cận này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then chốt nhất. Nhận diện các vấn đề trọng yếu BƯỚC 1 liên quan bên ngoài giúp xác định mức độ quan tâm của họ đối với từng chủ đề trọng Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát yếu. Toàn bộ phản hồi sau đó được lượng hóa thông qua hệ thống điểm số và xếp hạng"}\n{"id": "B-A-64f871f4b3ccb179", "claim": "Quản trị rủi ro Quản trị phát triển bền vững 9.", "evidence": "QUẢN TRỊ DOANH NGHIỆP VÀ QUẢN LÝ RỦI RO 7.1 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Quản trị Doanh nghiệp và Quản lý rủi ro Quản trị Phát triển bền vững 2-14 Vai trò của cơ quan quản trị cao nhất trong Báo cáo Phát triển Quản trị Phát triển bền vững bền vững 2-15 Xung đột lợi ích Quản trị Doanh nghiệp & Quản lý rủi ro"}\n{"id": "B-A-039f621a84daa3ad", "claim": "Xây dựng môi trường làm việc an toàn, Tăng cường quản trị minh bạch, Mục tiêu công bằng và thúc đẩy sự phát triển tích hợp ESG vào chiến lược, và thúc đẩy đổi mới sáng tạo có trách nhiệm với môi trường của con người Hành động giảm thiểu Quản lý môi trường Sức khoẻ và an toàn lao động", "evidence": "THÀNH VIÊN CỦA CÁC HIỆP HỘI NGÀNH NGHỀ 3.3 Tập đoàn Hòa Phát tích cực tham gia nhiều hiệp hội ngành nghề và tổ chức chuyên môn nhằm thúc đẩy thực hành kinh doanh có trách nhiệm, thúc đẩy đổi mới sáng tạo và đóng góp vào sự phát triển công nghiệp bền vững tại Việt Nam cũng như trên toàn cầu. Thông qua việc tham gia các hiệp hội này, Tập đoàn tham gia đối thoại chính sách, chia sẻ tri thức và hợp tác trong các sáng kiến hướng tới bảo vệ môi trường, nâng cao an toàn lao động và thúc đẩy kinh tế tuần hoàn trong ngành thép. Tập đoàn Hòa Phát không chỉ chú trọng vào hiệu quả sản xuất – kinh doanh mà còn mở rộng đóng góp ra toàn bộ chuỗi giá trị. Điều đó thể hiện 2. MÔI TRƯỜNG LÀM VIỆC AN TOÀN, LÀN"}\n{"id": "B-A-e8f79ea6a5c87c42", "claim": "Quản lý nguồn nhân lực Quản trị doanh nghiệp Quản lý chất lượng Chủ đề trọng yếu hàng đầu & Thích ứng BĐKH Quản trị phát triển bền vững", "evidence": "HÒA PHÁT TRỌNG YẾU VỚI CÁC BÊN LIÊN QUAN 4 3. Quản trị doanh nghiệp 8 4. Quản lý môi trường 10 5 5. Quản lý chất lượng 6 6. Quản trị nguồn nhân lực QUẢN LÝ CHẤT LƯỢNG 7.2 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Quản trị Doanh nghiệp và Quản lý rủi ro Quản lý chất lượng Phát triển đổi mới sáng tạo"}\n{"id": "B-A-bf0ae26b57d81933", "claim": "Các chủ đề quan trọng Phát triển đổi Quản lý chuỗi Đa dạng sinh học Nhân quyền Phát triển cộng đồng Quản lý Nhận diện các vấn đề trọng yếu mới sáng tạo cung ứng rủi ro khác Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát", "evidence": "NHÓM CHỦ ĐỀ ƯU TIÊN THẤP Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát 10. Quản lý chuỗi cung ứng 11. Phát triển cộng đồng 12. Đa dạng sinh học MÔI TRƯỜNG Gắn kết với các bên liên quan 9 NHÓM CHỦ ĐỀ ƯU TIÊN TRUNG BÌNH 12 11 7. Nhân quyền 8. Quản trị rủi ro Quản trị phát triển bền vững 9. Phát triển đổi mới sáng tạo Nhận diện các vấn đề trọng yếu"}\n{"id": "B-A-3dae6793a3d826ea", "claim": "Hòa Phát hướng tới duy trì năng lực cạnh tranh bền vững trong quá trình chuyển đổi hướng tới trung hòa carbon, đồng thời tiếp tục khẳng định cam kết về chất lượng sản phẩm dẫn đầu thị trường.", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 CHỨNG NHẬN CHẤT LƯỢNG cung cấp sản phẩm đáp ứng đầy đủ các quy định và tiêu chuẩn kỹ thuật quốc tế, giúp thủ tục thông quan diễn ra thuận lợi, đồng thời nâng cao năng lực cạnh tranh của sản phẩm trên thị trường toàn cầu. Nhằm đáp ứng yêu cầu của thị trường trong nước và quốc tế, Tập đoàn Hòa Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo. Bên cạnh đó. Tập đoàn Hòa Phá"}\n{"id": "B-A-4f0f0d1cca18d858", "claim": "Trong năm 2025, Hòa Phát đã xây dựng Chiến lược Phát triển Bền vững để định hướng hoạt động kinh doanh theo các xu thế dài hạn toàn cầu về ứng phó biến đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao năng lực cạnh tranh và khả năng thích ứng của Tập đoàn trong ngành thép đang biến đổi mạnh mẽ.", "evidence": "2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt khe và áp lực cạnh tranh về chi phí, Tập đoàn Hòa Phát xác định Chuyển đổi số là một trong những trụ cột chiến lược nhằm nâng cao năng lực cạnh tranh dài hạn, tối ưu hiệu quả sản xuất – quản trị và hỗ trợ thực hiện các mục tiêu phát triển bền vững. Đây chính là mắt xích quan trọng trong mô hình kinh tế tuần hoàn, góp phần thực hiện mục tiêu sản xuất thép xanh và bền vững. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu"}\n{"id": "B-A-3234f473c79b293c", "claim": "Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh tế ;", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 Tái chế, tái sử dụng chất thải rắn thành nguyên vật liệu quay lại quá trình sản xuất Các giải pháp tái chế, tái sử dụng nội bộ tập trung vào việc phân loại và xử lý các chất thải ngay tại nguồn phát sinh. Theo đó, các chất thải rắn chứa sắt sẽ được thu gom và xử lý, trở thành nguyên vật liệu đưa ngược lại quy trình sản xuất thép. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, "}\n{"id": "B-A-46192f6729107406", "claim": "nghiên cứu và tìm kiếm các giải pháp điều chỉnh cơ cấu tiêu thụ nguyên liệu và qua đó giảm phát sinh chất thải.", "evidence": "Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – tái chế thép). Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ than và giảm phát thải. Theo nghiên cứu của Hiệp hội Thép Thế giới (2021), việc tái chế một tấn thép phế có thể giúp giảm khoảng 1,5 tấn khí CO2. 103-5 Giảm tiêu thụ năng lượng Hành động giảm thiểu và thích ứng với biến đổi khí hậu 102-4 Mục tiêu và tiến độ giảm phát thải khí nhà kính Đây là báo cáo năm đầu tiên của chúng tôi, lộ trình giảm phát thải chi tiết vẫn đang được nghiên cứu xây dựng 102-5 Phát thải KNK Phạm vi 1 Kết quả kiểm kê phát thải khí nhà kính năm 2025"}\n{"id": "B-A-b53ef06e51401e03", "claim": "Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng các hệ thống tuần hoàn khép kín, tái chế nước và xử lý triệt để để tuân thủ quy chuẩn xả thải.", "evidence": "Dù lượng nước cần thiết cho sản xuất thép là rất cao, các Khu liên hợp Gang Thép của Hòa Phát đã và đang áp dụng công nghệ hiện đại, chủ động triển khai nhiều biện pháp nhằm bảo tồn nguồn nước để giảm thiểu việc lấy nước mới, giảm thiểu phần lớn áp lực khai thác nước đối với nguồn nước khu vực. Nước chứa cặn bẩn, tạp chất trong Khu liên hợp được xử lý thông qua hệ thống xử lý nước trung tâm và tái chế 100%. Các khu liên hợp hoàn toàn không có nước thải công nghiệp xả ra môi trường. Những nỗ lực này nhằm giảm sự phụ thuộc vào nguồn nước ngọt, đặc biệt là nguồn nước khai thác từ cộng đồng, đồng thời góp phần thực hiện các mục tiêu bảo vệ môi trường rộng hơn của Tập đoàn. Chất lượng nước thải s"}\n{"id": "B-A-a888054953f12a3c", "claim": "Những phản hồi nhận được là cơ sở quan trọng giúp Hòa Phát điều chỉnh kế hoạch hoạt động, củng cố quan hệ hợp tác và xác định các mục tiêu phát triển bền vững phù hợp trong từng giai đoạn.", "evidence": "Nhận diện các vấn đề trọng yếu BƯỚC 1 liên quan bên ngoài giúp xác định mức độ quan tâm của họ đối với từng chủ đề trọng Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát yếu. Toàn bộ phản hồi sau đó được lượng hóa thông qua hệ thống điểm số và xếp hạng Mức độ tác động đến doanh nghiệp được xác định dựa trên ảnh hưởng của các vấn đề ESG đối với hoạt động kinh doanh, tài chính và danh tiếng của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan kết hợp với đóng góp mà Hòa Phát mang lại cho xã hội và môi trường. Cách tiếp cận này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then chốt nhất."}\n{"id": "B-A-92488e0e32191ad1", "claim": "Tăng trưởng bền vững và hiệu quả lợi nhuận Quản trị doanh nghiệp minh bạch Quản trị rủi ro hiệu quả Công bố thông tin chính xác, kịp thời Là một công ty đại chúng, đảm bảo hiệu quả kinh doanh và quyền lợi hợp pháp của cổ đông/nhà đầu tư là mục tiêu được ưu tiên hàng đầu của Tập đoàn.", "evidence": "Duy trì nguyên tắc minh bạch và công bằng trong công bố thông tin, đảm bảo tuân thủ đầy đủ, chính xác và kịp thời theo quy định pháp luật. Tất cả thông tin liên quan đến sản xuất, kinh doanh, dự án trọng điểm và hoạt động quản trị doanh nghiệp đều được công bố thường xuyên trên website chính thức của Tập đoàn. Đại hội đồng cổ đông thường niên. Website công bố các thông tin pháp lý, tài chính, hoạt động kinh doanh, hoạt động xã hội.... Đảm bảo việc thực hiện các báo cáo liên quan Phát triển bền vững đầy đủ, chính xác, kịp thời PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Các yếu tố bền vững được tích hợp vào quá trình thảo luận và xây dựng chiến lược, ra quyết định đầu tư và giám sát quản"}\n{"id": "B-A-afdee4b0ae10a8c2", "claim": "An toàn và sức khỏe người tiêu dùng (đặc biệt với sản phẩm nông nghiệp, thực phẩm) Giá bán hợp lý Bao bì thân thiện môi trường Thông tin sản phẩm rõ ràng được dán nhãn trên bao bì sản phẩm", "evidence": "Sản phẩm thép là sản phẩm cơ bản dùng cho xây dựng và chế tạo thiết bị phục vụ cho đời sống con người, không phải là sản phẩm sử dụng trực tiếp như các loại thực phẩm. Tuy nhiên với ý thức trách nhiệm đối với người tiêu dùng và xã hội, Hòa Phát vẫn tiến hành rà soát các yếu tổ rủi ro có ảnh hưởng đến sức khỏe và an toàn của người tiêu dùng trên các công đoạn: Thiết kế, sản xuất, chứng nhận, lưu trữ và cung ứng sản phẩm. Nông nghiệp: sản phẩm sạch và đảm bảo dinh dưỡng với nguyên tắc cốt lõi là bảo vệ sức khỏe và an toàn của người tiêu dùng;"}\n{"id": "B-A-a3f85609b90abe4a", "claim": "Điện máy gia dụng: Các tính năng bảo vệ sức khỏe, tiết kiệm năng lượng, hậu mãi chu đáo Giá cả: Giá bán các sản phẩm của Tập đoàn hợp lý theo cung cầu thị trường, tạo khả năng tiếp cận đến đông đảo người tiêu dùng Cập nhật thông tin sản phẩm, chương trình, phản hồi nhanh chóng thông qua website & mạng xã hội.", "evidence": "Đa dạng hóa danh mục sản phẩm, đáp ứng nhiều loại nhu cầu của khách hàng và thị trường Giao hàng đúng tiến độ, đúng chủng loại, giá cả phù hợp với cung cầu thị trường, chế độ hậu mãi tốt. Cập nhật thông tin sản phẩm, chương trình, phản hồi nhanh chóng thông qua website & mạng xã hội. Các sản phẩm điện máy gia dụng được bảo quản và vận chuyển trong điều kiện phù hợp, bao bì chắc chắn đảm bảo an toàn cho người lao động và sản phẩm. Dịch vụ hậu mãi toàn quốc sử dụng linh kiện chính hãng, duy trì độ an toàn như ban đầu. Công ty cũng thu thập phản hồi khách hàng để cải tiến thiết kế, nâng cao độ bền và hiệu quả bảo vệ sức khỏe người tiêu dùng."}\n{"id": "B-A-949b6ecfceffe47a", "claim": "Quản trị doanh nghiệp tốt Tuân thủ các quy định pháp luật và minh bạch về Thuế, Môi trường, Lao động, An toàn và Sức khỏe nghề nghiệp Đóng góp cho lộ trình và cam kết quốc gia về giảm phát thải khí nhà kính", "evidence": "Duy trì cơ chế kê khai và nộp thuế minh bạch và tuân thủ pháp luật. Hòa Phát đóng góp đáng kể vào ngân sách Nhà nước mỗi năm với gần 13,000 tỷ đồng trong năm 2025, nằm trong nhóm các doanh nghiệp tư nhân hàng đầu về nghĩa vụ thuế. Tuân thủ nghiêm túc các quy định pháp luật trong nước Giám sát và báo cáo – Hiệu quả của các biện pháp quản lý rủi ro được đánh giá định kỳ, đồng thời điều chỉnh kịp thời để thích ứng với tình hình thực tế. Tập đoàn luôn tuân thủ nghiêm ngặt các quy định của pháp luật về thuế, thực hiện kê khai, nộp thuế đầy đủ, minh bạch và đúng hạn trên toàn hệ thống, khẳng định vai trò là một trong những doanh nghiệp đóng góp ngân sách nhà nước lớn nhất Việt Nam, tích cực tạo ng"}\n{"id": "B-A-7639eece2a85dc7f", "claim": "Thực hiện kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 và báo cáo đầy đủ theo quy định Hợp tác và phối hợp với chính quyền địa phương.", "evidence": "ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025 3.1 Tự chủ 90% nhu cầu điện tại Khu liên hợp Gang thép Hòa Phát Dung Quất Hoàn thành kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 cho toàn Tập đoàn và được BSI thẩm định số liệu. HOÀN THÀNH KIỂM KÊ KHÍ NHÀ KÍNH THEO ISO 14064-1:2018 VÀ ISO 14067:2018 TÁI CƠ CẤU ĐỘI TÀU, MỞ RỘNG VẬN TẢI NỘI ĐỊA Năm 2024, Hòa Phát tập trung tái cơ cấu đội tàu biển, điều chỉnh quy mô tàu để phù hợp với nhu cầu vận chuyển của Khu liên hợp Gang thép Hòa Phát Dung Quất, đặc biệt khi giai đoạn 2 của dự án đi vào hoạt động."}\n{"id": "B-A-067feef455620325", "claim": "Tập đoàn đang triển khai đồng bộ các sáng kiến và hành động, đặc biệt trong lĩnh vực sản xuất thép, nhằm tiết kiệm năng lượng và giảm phát thải khí nhà kính phù hợp với điều kiện kinh doanh.", "evidence": "Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn. Trong đó, hai Khu liên hợp Gang thép Hòa Phát Hải Dương và Hòa Phát Dung Quất — chiếm khoảng 94% tổng công suất thép thô toàn Tập đoàn — là đơn vị trọng điểm trong công tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải. XANH HÓA SẢN XUẤT QUẢN TRỊ VÀ ĐỔI MỚI Trụ cột Xanh hóa Sản xuất hướng tới hài hòa giữa tăng trưởng công nghiệp và bảo vệ môi trường, giảm thiểu biến đổi khí"}\n{"id": "B-A-1763c9b70fe9d902", "claim": "Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn.", "evidence": "Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phá"}\n{"id": "B-A-5c488a73c99c5baa", "claim": "Trong đó, hai Khu liên hợp Gang thép Hòa Phát Hải Dương và Hòa Phát Dung Quất — chiếm khoảng 94% tổng công suất thép thô toàn Tập đoàn — là đơn vị trọng điểm trong công tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải.", "evidence": "Công ty trong nhóm nông nghiệp (gồm thức ăn chăn nuôi, chăn nuôi). Hoàn thành đầu tư giai đoạn 3 – Khu liên hợp gang thép Hòa Phát Hải Dương, nâng công suất thép xây dựng Hòa Phát lên hơn 2 triệu tấn/năm. THÁNG 2/2017 Thành lập Công ty CP Thép Hòa Phát Dung Quất, triển khai Khu Liên hợp sản xuất gang thép Hòa Phát Dung ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Tập đoàn lập kế hoạch triển khai các dự án đề xuất tăng cường phối phế liệu cùng gang lỏng trong lò thổi để tăng tỷ trọng nguyên vật liệu phát thải thấp. Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao than cốc và các nhiên liệu trên một tấn thép"}\n{"id": "B-A-9bfe2228fdd68025", "claim": "Phục vụ quản lý nội bộ: Ngoài đáp ứng yêu cầu của pháp luật, các công ty thành viên không nằm trong danh sách bắt buộc phải kiểm kê KNK của Nghị định 06/2022/NĐ-CP cũng được BSI đào tạo để chủ động thực hiện kiểm kê, phục vụ cho nhu cầu quản lý phát thải chung của cả Tập đoàn.", "evidence": "Chúng tôi cam kết nỗ lực đồng hành cùng Chính phủ Việt Nam hướng tới mục tiêu Net Zero 2050. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Tuân thủ quy định pháp luật: Tất cả các công ty thành viên của Tập đoàn thuộc danh mục phải kiểm kê phát thải theo Nghị định 06/2022/NĐ-CP đã tiến hành kiểm kê KNK theo chuẩn ISO 14064 và được BSI chứng nhận kết quả kiểm kê, hoàn thành và nộp báo cáo kiểm kê KNK đầy đủ cho các cơ quan chức năng từ năm 2024. PHẦN 5 XANH HÓA SẢN XUẤT PHƯƠNG PHÁP QUẢN LÝ Trong năm 2024, bộ phận Kiểm toán nội bộ thuộc HĐQT quản lý đã phối hợp với các tuyến kiểm soát thực hiện tốt các nhiệm vụ và hoàn thành các mục tiêu đề ra theo đúng quy chế, kế hoạch kiểm toán năm cũng như quy định pháp"}\n{"id": "B-A-601e678b1b4377a9", "claim": "Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn.", "evidence": "SẢN XUẤT Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn, do vậy kỳ báo cáo từ ngày 01/01/2025 đến 31/12/2025 được lựa chọn làm năm cơ sở vì đây là kỳ báo cáo đầu tiên. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Những cải tiến này đã góp phần nâng cao đáng kể hiệu suất năng lượng, giảm phụ thuộc vào nguồn điện bên ngoài và đóng góp tích cực vào mục tiêu khử carbon dài hạn của Tập đoàn. Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát"}\n{"id": "B-A-a7cf98474cf4da07", "claim": "Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau:", "evidence": "Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám sát hiệu quả sử dụng năng lượng, theo dõi kiểm kê phát thải và thúc đẩy các sáng kiến giảm phát thải. Ban họp định kỳ hàng quý nhằm đánh giá tiến độ thực hiện mục tiêu, chia sẻ thông tin về các dự án tiết kiệm năng lượng, truyền thông các chủ đề trọng yếu, đồng thời tham mưu cho"}\n{"id": "B-A-39e899b27630cc26", "claim": "Ban họp định kỳ hàng quý nhằm đánh giá tiến độ thực hiện mục tiêu, chia sẻ thông tin về các dự án tiết kiệm năng lượng, truyền thông các chủ đề trọng yếu, đồng thời tham mưu cho", "evidence": "Bên cạnh triển khai các dự án phần mềm, Tập đoàn Hòa Phát chú trọng truyền thông và đào tạo nhằm nâng cao nhận thức và kỹ năng số cho nhân viên: PHẦN 6 PHÁT TRIỂN BỀN VỮNG Salesforce CRM: Giúp đội ngũ kinh doanh chuẩn hóa quy trình bán hàng, tối ưu hoạt động và tăng cường hợp tác với đại lý, nhà phân phối, nâng cao hiệu quả kinh doanh và giảm chi phí vận hành. Chuyên trang truyền thông nội bộ về Chuyển đổi số đã đăng tải các bài viết cập nhật thông tin dự án, chia sẻ tư duy tiên tiến và phương pháp chuyển đổi số, thu hút sự quan tâm của hàng ngàn cán bộ nhân viên. Chuyên trang truyền thông nội bộ về Chuyển đổi số: được duy trì thường xuyên, cập nhật kịp thời tiến độ các dự án, đồng thời chia"}\n{"id": "B-A-2285774a9edf6d37", "claim": "Hằng năm, Ban xây dựng kế hoạch giảm phát thải, theo dõi tiến độ triển khai và đề xuất các giải pháp chiến lược nhằm nâng cao hiệu quả năng lượng và giảm thiểu tác động môi trường.", "evidence": "Tập đoàn Hòa Phát cam kết tuân thủ đầy đủ các quy định của pháp luật về môi trường như Luật bảo vệ môi trường năm 2020, các Nghị định và Thông tư hướng dẫn có liên quan, thông qua việc triển khai hệ thống quản lý môi trường hiệu quả bao gồm: Hành động giảm thiểu và thích ứng với biến đổi khí hậu HỆ THỐNG: Đầu tư công nghệ hiện đại Hòa Phát ứng dụng công nghệ tiên tiến để giảm thiểu tối đa các tác động bất lợi đến môi trường. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là"}\n{"id": "B-A-43643173918263ce", "claim": "Báo cáo được thiết lập dựa trên các yêu cầu của tiêu chuẩn ISO 14064-1, theo các hướng dẫn của Ban liên chính phủ về biến đổi khí hậu (IPCC Guidelines) và đáp ứng các yêu cầu về giảm phát thải khí nhà kính của Cơ quan quản lý được đề cập trong Nghị định 06/2022/ND-CP, Nghị định 119/2024/ND-CP.", "evidence": "Phục vụ quản lý nội bộ: Ngoài đáp ứng yêu cầu của pháp luật, các công ty thành viên không nằm trong danh sách bắt buộc phải kiểm kê KNK của Nghị định 06/2022/NĐ-CP cũng được BSI đào tạo để chủ động thực hiện kiểm kê, phục vụ cho nhu cầu quản lý phát thải chung của cả Tập đoàn. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Thực hiện kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 và báo cáo đầy đủ theo quy định Hợp tác và phối hợp với chính quyền địa phương. Làm việc trực tiếp với các cơ quan và tổ chức quản lý nhà nước."}\n{"id": "B-A-6eb95685c55ec1af", "claim": "Theo đó, Tập đoàn sẽ xem xét các Công ty mà Tập đoàn có góp vốn đầu tư để thu thập thông tin khí nhà kính và đưa vào báo cáo này.", "evidence": "(Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt động này cũng góp phần phản ánh một cách hiệu quả các kết quả mà Tập đoàn đã đạt được và các giải pháp mà Tập đoàn sẽ triển khai trong tương lai. Thu hồi nhiệt và khí thải từ các lò luyện cốc, lò cao và lò chuyển để phát điện và cải thiện hiệu suất sản xuất. Ứng dụng công nghệ dập coke khô, vừa tận dụng nhiệt năng để phát điện, vừa giảm thiểu tác động môi trường. Tập đoàn Hòa Phát duy trì cơ chế trao đổi thường xuyên giữa Hội đồng Quản trị và Ban Tổng Giám đốc, giữa Ban Tổng Giám đốc và các công ty thành viên và phòng ban của Tập đoàn Hòa Phá"}\n{"id": "B-A-49ba0de06edee2d7", "claim": "Lượng khí nhà kính tổng hợp dựa trên lượng khí nhà kính của các Công ty góp vốn, có tính đến tỉ lệ phần trăm vốn góp.", "evidence": "BAN TỔNG GIÁM ĐỐC BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Tham mưu, đề xuất, triển khai thực hiện các công việc liên quan đến quản lý năng lượng và khí nhà kính. TẠI CẤP TẬP ĐOÀN TẠI CẤP CÔNG TY CON BAN 6 PHẦN PHÁT TRIỂN BỀN VỮNG 1 HÒA PHÁT HOÀN THÀNH KIỂM KÊ KHÍ NHÀ KÍNH TOÀN TẬP ĐOÀN 2 SÁNG KIẾN CẢI TIẾN KỸ THUẬT GÓP PHẦN TIẾT KIỆM HÀNG TRĂM TỶ ĐỒNG 3 TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP"}\n{"id": "B-A-c90876cb8b899406", "claim": "(Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt động này cũng góp phần phản ánh một cách hiệu quả các kết quả mà Tập đoàn đã đạt được và các giải pháp mà Tập đoàn sẽ triển khai trong tương lai.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 Với mục đích phản ánh một cách chính xác và khách quan nhất những tác động biến đổi khí hậu xuất phát từ các hoạt động trong phạm vi kinh doanh của mình cũng như trình bày một cách minh bạch các kết quả đã đạt được và những giải pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp). Bên cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài Các nguồn nhiệt dư sinh ra trong quá trình sản xuất được thu hồi để phát điện. Đến nay, cả hai khu liên hợp lớn nhất của Tập đoàn tại Dung Quất và Hải Dương đều có mức độ tự c"}\n{"id": "B-A-73aaca264c29e5cb", "claim": "SẢN XUẤT Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn, do vậy kỳ báo cáo từ ngày 01/01/2025 đến 31/12/2025 được lựa chọn làm năm cơ sở vì đây là kỳ báo cáo đầu tiên.", "evidence": "Phạm vi. thời gian và giới hạn báo cáo 2-4 Trình bày lại thông tin Đây là báo cáo công bố lần đầu tiên 2-5 Đảm bảo số liệu từ Đơn vị bên ngoài Số liệu về phát thải KNK của toàn tập đoàn đã được xác nhận bởi BSI. 01/01/2025 đến ngày 31/12/2025, với các thông tin công bố được tham chiếu trong bảng chỉ mục nội dung GRI. Báo cáo cũng phản ánh các tiêu chuẩn của Hội đồng Chuẩn mực Kế toán Bền vững (SASB) dành cho các nhà sản xuất sắt và thép. Báo cáo này được công bố định kỳ hằng năm."}\n{"id": "B-A-54e348a76f7faeae", "claim": "Hành động giảm thiểu và thích ứng với biến đổi khí hậu Những cải tiến này đã góp phần nâng cao đáng kể hiệu suất năng lượng, giảm phụ thuộc vào nguồn điện bên ngoài và đóng góp tích cực vào mục tiêu khử carbon dài hạn của Tập đoàn.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 Với mục đích phản ánh một cách chính xác và khách quan nhất những tác động biến đổi khí hậu xuất phát từ các hoạt động trong phạm vi kinh doanh của mình cũng như trình bày một cách minh bạch các kết quả đã đạt được và những giải pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp). Bên cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài PHẦN 5 XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu HÀNH ĐỘNG GIẢM PHÁT THẢI KHÍ NHÀ KÍNH & TỐI ƯU NĂNG LƯỢNG Quản lý môi trường TĂNG SỬ DỤNG THÉP "}\n{"id": "B-A-4c3f8204e304b2d8", "claim": "Thành tựu này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon.", "evidence": "Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo. Bên cạnh đó. Tập đoàn Hòa Phát nghiên cứu triển khai lắp đặt hệ thống điện mặt trời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy. Cam kết này nhấn mạnh sự hài lòng của khách hàng, tuân thủ quy định, hiệu quả hoạt động và cải tiến liên tục để nâng cao độ tin cậy và hiệu suất của sản phẩm. Quản lý chất lượng Phát triển đổ"}\n{"id": "B-A-4e7008411ce1d86d", "claim": "Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải.", "evidence": "Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: Quản trị Doanh nghiệp và Quản lý rủi ro XÃ HỘI: Chuẩn hóa quy trình và tự động hóa giúp nâng cao năng suất lao động, giảm áp lực thủ tục hành chính, cải thiện môi trường làm việc và tăng tính minh bạch trong đánh giá hiệu quả công việc. TẦM NHÌN & ĐỊNH HƯỚNG CHIẾN LƯỢC CHUYỂN ĐỔI SỐ Tập đoàn định hướng xây dựng mô hình doanh nghiệp sản xuất công nghiệp chất lượng dẫn đầu, vận hành trên nền tảng dữ liệu tập trung, quy trình chuẩn hóa và hệ thống tự động hóa thông minh."}\n{"id": "B-A-650f6dcac772f240", "claim": "LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp mái tại các cơ sở sản xuất.", "evidence": "Các nguồn nhiệt dư sinh ra trong quá trình sản xuất được thu hồi để phát điện. Đến nay, cả hai khu liên hợp lớn nhất của Tập đoàn tại Dung Quất và Hải Dương đều có mức độ tự chủ điện năng rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tớ"}\n{"id": "B-A-38d92e11a963e68c", "claim": "Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo.", "evidence": "Thành tựu này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám sát hiệu quả sử dụng năng lượng, theo dõi kiểm kê phát thải và thúc đẩy các sáng kiến giảm phát thải. Ban họp định kỳ "}\n{"id": "B-A-8a53d3e0fdfc2fd3", "claim": "Tập đoàn Hòa Phát nghiên cứu triển khai lắp đặt hệ thống điện mặt trời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy.", "evidence": "Các nguồn nhiệt dư sinh ra trong quá trình sản xuất được thu hồi để phát điện. Đến nay, cả hai khu liên hợp lớn nhất của Tập đoàn tại Dung Quất và Hải Dương đều có mức độ tự chủ điện năng rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp mái tại các cơ sở sản xuất."}\n{"id": "B-A-008237d7adb0735e", "claim": "Đây là giải pháp sạch, không phát thải tại chỗ, góp phần lớn vào mục tiêu trung hòa carbon trong dài hạn.", "evidence": "Quất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng mà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng đến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt dư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên liệu hóa thạch”. Trong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, tự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. SẢN XUẤT Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn, do vậy kỳ báo cáo từ ngày 01/01/2025 đến 31/12/2025 được lựa chọn làm năm cơ sở vì đây là kỳ báo cáo đầu tiên. Hành động giảm thiểu và thích ứ"}\n{"id": "B-A-145c1f36a04d93e2", "claim": "Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai các giải pháp chủ chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng cao hiệu quả vận hành và giảm phát thải.", "evidence": "ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Tập đoàn lập kế hoạch triển khai các dự án đề xuất tăng cường phối phế liệu cùng gang lỏng trong lò thổi để tăng tỷ trọng nguyên vật liệu phát thải thấp. Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao than cốc và các nhiên liệu trên một tấn thép, đồng thời tiết kiệm năng lượng trong toàn bộ chuỗi sản xuất. Quất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng mà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng đến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt dư trong quá trình s"}\n{"id": "B-A-ffc060149c606389", "claim": "Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời hoặc điện gió.", "evidence": "Do sự khan hiếm trong nguồn cung nguyên liệu sạch, hiện nay, hơn 70% sản lượng thép toàn cầu vẫn đang được sản xuất theo công nghệ lò cao truyền thống (BOF) (World Bank). Công nghệ này giúp sản xuất thép từ quặng sắt nguyên khai với độ tinh khiết và chất lượng cao, song cũng phát thải CO2 nhiều hơn so với lò điện hồ quang (EAF). Đặc biệt, các Khu liên hợp sản xuất gang thép ứng dụng đồng thời nhiều hệ thống lọc bụi phù hợp với từng loại đối tượng bụi, được lắp đặt tại từng công đoạn như lò thiêu kết, lò cao, lò tinh luyện, nồi hơi nhiệt dư và dây chuyền cán thép giúp hiệu suất xử lý đạt trên 99% ngay tại nguồn phát sinh và không phát tán ra không khí và tận dụng bụi như một nguồn tài nguyên "}\n{"id": "B-A-0e882b5a07d83141", "claim": "Ngược lại, EAF có mức phát thải thấp hơn, nhưng phụ thuộc vào nguồn nguyên liệu như thép phế, quặng sắt đã hoàn nguyên – những nguyên liệu có giá thành cao và nguồn cung hạn chế, phù hợp với các quốc gia phát triển, nơi ngành thép đã đi qua nhiều vòng đời tái chế.", "evidence": "Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn. Lượng phát thải carbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai đoạn nào của vòng đời. Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu thành thép để tạo vòng đời tiếp theo. Luyện thép bằng lò điện hồ quang (EAF) là công nghệ sử dụng thép phế liệu làm nguyên liệu chính. Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp"}\n{"id": "B-A-4646a1edaa9a73e3", "claim": "Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – tái chế thép).", "evidence": "Thành viên của các Hiệp hội ngành nghề Phát đang chủ động gia tăng dần tỷ lệ sử dụng thép phế liệu trong cả hai quy trình sản xuất. Trong tầm nhìn dài hạn, khi nguồn cung thép phế liệu trở nên dồi dào và dễ tiếp cận hơn, Hòa Phát định hướng đẩy mạnh hơn nữa tỷ trọng thép tái chế trong cơ cấu sản Hoạt động kinh doanh tại Tập đoàn xuất để tối ưu hóa vòng đời thứ cấp. Quy trình này tạo thành một vòng lặp kín vô tận, biến thép trở thành một trong những vật liệu cốt lõi của nền kinh tế tuần hoàn và phát triển bền vững. Hoạt động kinh doanh tại Tập đoàn Phế liệu thép sau tiêu dùng 100% có thể tái chế Tái sử dụng & tái sản xuất"}\n{"id": "B-A-81820fee46a933b3", "claim": "Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ than và giảm phát thải.", "evidence": "Tập đoàn đang tăng cường tỷ lệ sử dụng thép phế và HBI trong lò thổi. Biện pháp này giúp giảm mạnh nhu cầu hoàn nguyên quặng, giảm tiêu hao than cốc, tận dụng nhiệt lượng và giảm phát thải khí nhà kính trong toàn bộ chuỗi sản xuất, đồng thời gia tăng năng lực sản xuất thép. Đến năm 2030, các công ty thép của tập đoàn Hòa Phát dự kiến tăng sử dụng thép phế đạt 30%. Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh tế ; nghiên cứu và tìm kiếm các giải pháp điều chỉnh cơ cấu tiêu thụ nguyên liệu và qua đó giảm phát sinh chất thải. Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng cá"}\n{"id": "B-A-ad63a5f4116e4aa5", "claim": "Theo nghiên cứu của Hiệp hội Thép Thế giới (2021), việc tái chế một tấn thép phế có thể giúp giảm khoảng 1,5 tấn khí CO2.", "evidence": "17.48 Cường độ tiêu thụ năng lượng trên một tấn thép thô (GJ/tấn) – Bình quân thế giới 2024 Hành động giảm thiểu và thích ứng với biến đổi khí hậu TỶ LỆ TỰ CHỦ ĐIỆN NĂNG CỦA KLH GANG THÉP Quy trình này tạo thành một vòng lặp kín vô tận, biến thép trở thành một trong những vật liệu cốt lõi của nền kinh tế tuần hoàn và phát triển bền vững. Hoạt động kinh doanh tại Tập đoàn Phế liệu thép sau tiêu dùng 100% có thể tái chế Tái sử dụng & tái sản xuất"}\n{"id": "B-A-7f6a379803495a59", "claim": "Biện pháp này giúp giảm mạnh nhu cầu hoàn nguyên quặng, giảm tiêu hao than cốc, tận dụng nhiệt lượng và giảm phát thải khí nhà kính trong toàn bộ chuỗi sản xuất, đồng thời gia tăng năng lực sản xuất thép.", "evidence": "ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Tập đoàn lập kế hoạch triển khai các dự án đề xuất tăng cường phối phế liệu cùng gang lỏng trong lò thổi để tăng tỷ trọng nguyên vật liệu phát thải thấp. Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao than cốc và các nhiên liệu trên một tấn thép, đồng thời tiết kiệm năng lượng trong toàn bộ chuỗi sản xuất. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HA"}\n{"id": "B-A-98032aba5c523104", "claim": "Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao than cốc và các nhiên liệu trên một tấn thép, đồng thời tiết kiệm năng lượng trong toàn bộ chuỗi sản xuất.", "evidence": "Tập đoàn đang tăng cường tỷ lệ sử dụng thép phế và HBI trong lò thổi. Biện pháp này giúp giảm mạnh nhu cầu hoàn nguyên quặng, giảm tiêu hao than cốc, tận dụng nhiệt lượng và giảm phát thải khí nhà kính trong toàn bộ chuỗi sản xuất, đồng thời gia tăng năng lực sản xuất thép. Đến năm 2030, các công ty thép của tập đoàn Hòa Phát dự kiến tăng sử dụng thép phế đạt 30%. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải "}\n{"id": "B-A-67ed1eb1d48ef297", "claim": "Bên cạnh đó, các công ty thép tiếp tục nghiên cứu, tìm kiếm nguồn cung cấp quặng và nhiên liệu sạch hơn, hướng tới chuyển dịch dần từ than coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao.", "evidence": "(vòng đời sơ cấp – sản xuất thép mới) trên thế giới chủ yếu tập trung vào việc sử dụng các loại nguyên liệu đầu vào có hàm lượng carbon thấp làm chất khử quặng sắt như khí tự nhiên hoặc Hydrogen xanh (sản xuất từ nguồn điện sạch) để tạo ra sắt hoàn nguyên trực tiếp (DRI và HBI), thay thế cho việc hoàn nguyên bằng nhiên liệu hóa thạch truyền thống như than. Hạn chế khiến cho biện pháp này chưa thể áp dụng rộng rãi nằm ở sự khan hiếm nguồn cung các loại quặng sắt có hàm lượng sắt cao %Fe>65% cũng như nguồn khí tự nhiên và hydrogen sạch. Ngược lại, EAF có mức phát thải thấp hơn, nhưng phụ thuộc vào nguồn nguyên liệu như thép phế, quặng sắt đã hoàn nguyên – những nguyên liệu có giá thành cao và "}\n{"id": "B-A-c21723a4c2d50e0e", "claim": "Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch.", "evidence": "HÒA PHÁT trò nền tảng để cung cấp nguồn thép chất lượng cao cho các công trình hạ tầng và máy móc, tuy nhiên nó tiêu tốn một lượng lớn năng lượng và giải phóng nhiều khí thải CO2 do quá trình đốt cháy nhiên liệu hóa thạch. Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Luyện thép bằng lò điện hồ quang (EAF) là công nghệ sử dụng thép phế liệu làm nguyên liệu chính. Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời hoặc điện gió. Tuy nhiên, việc chuyển dịch công nghệ phụ thuộc cao vào giải quyết bài toán nguyên vật liệu."}\n{"id": "B-A-df0802d1054adb62", "claim": "Qua đó, lượng CO2 sinh ra trong quá trình cháy và hoàn nguyên cũng giảm tương ứng.", "evidence": "HÒA PHÁT trò nền tảng để cung cấp nguồn thép chất lượng cao cho các công trình hạ tầng và máy móc, tuy nhiên nó tiêu tốn một lượng lớn năng lượng và giải phóng nhiều khí thải CO2 do quá trình đốt cháy nhiên liệu hóa thạch. Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống"}\n{"id": "B-A-58c29218b3598262", "claim": "Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ưu hóa chi phí vận hành mà còn giảm thiểu đáng kể phát thải nhiệt và khí thải ra môi trường.", "evidence": "Điểm ưu việt của mô hình này nằm ở việc năng lượng được sản xuất và tiêu thụ tại chỗ, loại bỏ hoàn toàn hao hụt trong quá trình truyền tải xa và tạo ra một lớp “lá chắn nhiệt” tự nhiên giúp hạ nhiệt độ mái tôn đáng kể, từ đó giảm điện năng tiêu thụ cho các thiết bị làm mát bên dưới. Việc tối ưu hóa diện tích mái xưởng sẵn có không chỉ giúp Hòa Phát chủ động nguồn điện, giảm tải cho lưới điện quốc gia mà còn là minh chứng thực tế cho cam kết phát triển bền vững. Quan trọng hơn, biogas tạo ra khí Mê-tan CH4 có thể được sử dụng làm nguồn năng lượng tái tạo để đun nấu hoặc phát điện, giúp giảm chi phí vận hành và đồng thời giảm phát thải khí nhà kính mạnh ra môi trường. Tại các trang trại chăn n"}\n{"id": "B-A-48ad4edd9019a288", "claim": "Đây chính là mắt xích quan trọng trong mô hình kinh tế tuần hoàn, góp phần thực hiện mục tiêu sản xuất thép xanh và bền vững.", "evidence": "Chất thải rắn trong mô hình sản xuất thép tuần hoàn không còn được xem là gánh nặng chôn lấp, đổ bỏ mà trở thành tài nguyên được tận dụng tối đa và mang lại hiệu quả kinh tế. Hiện nay, tỷ lệ tái chế chất thải rắn của nhóm thép đạt hơn 99%. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA Quy trình này tạo thành một vòng lặp kín vô tận, biến thép trở thành một trong những vật liệu cốt lõi của nền kinh tế tuần hoàn và phát triển bền vững. Hoạt động kinh doanh tại Tập đoàn Phế liệu thép sau tiêu dùng 100% có thể tái chế Tái sử dụng & tái sản xuất"}\n{"id": "B-A-11e1de27b24fb33b", "claim": "Việc tối ưu hóa diện tích mái xưởng sẵn có không chỉ giúp Hòa Phát chủ động nguồn điện, giảm tải cho lưới điện quốc gia mà còn là minh chứng thực tế cho cam kết phát triển bền vững.", "evidence": "Tất cả nguồn hơi này hội tụ về hệ thống quản lý năng lượng tập trung để làm quay máy phát điện công suất lớn. Chu trình khép kín này giúp chuyển hóa toàn bộ phụ phẩm phát sinh thành điện năng sạch, phục vụ trực tiếp cho các hoạt động sản xuất tại nhà máy. Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ưu hóa chi phí vận hành mà còn giảm thiểu đáng kể phát thải nhiệt và khí thải ra môi trường. Nguồn điện xanh này được tích hợp vào hệ thống cấp điện nội bộ, thay thế một phần điện từ lưới quốc gia. Đây là giải pháp sạch, không phát thải tại chỗ, góp phần lớn vào mục tiêu trung hòa carbon trong dài hạn. Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai cá"}\n{"id": "B-A-f594ca02a550a195", "claim": "Đây là bước đi then chốt nhằm giảm dấu chân carbon trên mỗi đơn vị sản phẩm, trực tiếp nâng cao năng lực cạnh tranh cho thép và các sản phẩm công nghiệp của Tập đoàn trên thị trường quốc tế, đặc biệt là tại các khu vực khắt khe về tiêu chuẩn môi trường.", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 CHỨNG NHẬN CHẤT LƯỢNG cung cấp sản phẩm đáp ứng đầy đủ các quy định và tiêu chuẩn kỹ thuật quốc tế, giúp thủ tục thông quan diễn ra thuận lợi, đồng thời nâng cao năng lực cạnh tranh của sản phẩm trên thị trường toàn cầu. Nhằm đáp ứng yêu cầu của thị trường trong nước và quốc tế, Tập đoàn Hòa 2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt khe và áp lực cạnh tranh về chi phí, Tập đoàn Hòa Phát xác định Chuyển đổi số là một trong những trụ cột chiến lược nhằm nâng cao năng lực cạnh tranh dài hạn, tối ưu hiệu quả sản xuất – quản trị và hỗ trợ thực hiện các mục tiêu ph"}\n{"id": "B-A-2efc3c090e2a1854", "claim": "Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) trong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu...", "evidence": "Quan trọng hơn, biogas tạo ra khí Mê-tan CH4 có thể được sử dụng làm nguồn năng lượng tái tạo để đun nấu hoặc phát điện, giúp giảm chi phí vận hành và đồng thời giảm phát thải khí nhà kính mạnh ra môi trường. Tại các trang trại chăn nuôi heo, chất lượng không khí được kiểm soát bằng kết hợp nhiều biện pháp: BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 KIỂM SOÁT KHẨU PHẦN ĂN CỦA VẬT NUÔI Nhiệt độ luôn nằm ở mức ổn định từ 18 - 25oC nhờ hệ thống điều khiển tự động, đàn vật nuôi ít bị mệt, giảm stress nhiệt giúp tiêu hóa hiệu quả và giảm phát thải khí Amoniac trong chất thải. Các khí thải trong chuồng được xử lý bằng chế phẩm vi sinh thay vì phát tán không kiểm soát như trong chuồng hở. Hệ thống cũng cho p"}\n{"id": "B-A-25130ffaa51d500c", "claim": "Những giải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu mà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an toàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp bền vững và thích ứng khí hậu.", "evidence": "2025 1. Giảm thiểu và Thích ứng với biến đổi khí hậu 3 7 2. Sức khỏe & An toàn lao động Thấp Trung bình Cao PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024"}\n{"id": "B-A-afe638d512f5d282", "claim": "Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp.", "evidence": "Sản xuất phân bón và hợp chất nitơ. Chăn nuôi lợn, chăn nuôi gia cầm, và hoạt động dịch vụ chăn nuôi. Chế biến, bảo quản thịt và các sản phẩm từ thịt. Sản xuất thức ăn gia súc, gia cầm và thủy sản. Nông nghiệp: chăn nuôi lợn, bò, gia cầm; hoạt động dịch vụ chăn nuôi; chế biến, bảo quản thịt và các sản phẩm từ thịt; sản xuất thức ăn gia súc, gia cầm"}\n{"id": "B-A-2b621e5d0fc6f6f2", "claim": "Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực.", "evidence": "Quản lý môi trường Xét theo lĩnh vực hoạt động, ngành sản xuất thép là lĩnh vực phát thải lớn nhất, đóng góp chủ yếu vào tổng phát thải của toàn tập đoàn. Nhóm nông nghiệp, bất động sản và điện máy gia dụng có mức phát thải thấp hơn đáng kể. Nhóm nguồn phát thải trực tiếp từ quá trình sản xuất chiếm tỷ trọng rõ rệt, phản ánh đặc thù ngành nghề và quy mô sản xuất của tập đoàn. TỔNG PHÁT THẢI KNK TOÀN TẬP ĐOÀN 2025 (Theo phạm vi) BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Phạm vi 1 PHẦN 5 XANH HÓA SẢN XUẤT Phạm vi 2 Như vậy, việc kiểm soát và giảm thiểu phát thải khí nhà kính cần tập trung vào nhóm nguồn phát thải trực tiếp trong sản xuất và lĩnh vực thép – lĩnh vực đóng vai trò chủ lực trong hoạt động "}\n{"id": "B-A-f3e879e43b891045", "claim": "Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 PHẦN 5 XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu HÀNH ĐỘNG GIẢM PHÁT THẢI KHÍ NHÀ KÍNH & TỐI ƯU NĂNG LƯỢNG Quản lý môi trường TĂNG SỬ DỤNG THÉP PHẾ LIỆU"}\n{"id": "B-A-a9508c1b5ddda690", "claim": "Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất.", "evidence": "Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. Quản lý môi trường Xét theo lĩnh vực hoạt động, ngành sản xuất thép là lĩnh vực phát thải lớn nhất, đóng góp chủ yếu vào tổng phát thải của toàn tập đoàn. Nhóm nông nghiệp, bất động sản và điện máy gia dụng có mức phát t"}\n{"id": "B-A-ccdecaad212a562d", "claim": "Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài.", "evidence": "Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. Các nguồn nhiệt dư sinh ra trong quá trình sản xuất được thu hồi để phát điện. Đến nay, cả hai khu liên hợp lớn nhất của Tập đoàn tại Dung Quất và Hải Dương đều có mức độ tự chủ điện năng rất cao, góp phần giảm phát thải"}\n{"id": "B-A-d297bd0fe81a4b9c", "claim": "Quản lý môi trường Xét theo lĩnh vực hoạt động, ngành sản xuất thép là lĩnh vực phát thải lớn nhất, đóng góp chủ yếu vào tổng phát thải của toàn tập đoàn.", "evidence": "TỔNG PHÁT THẢI KNK TOÀN TẬP ĐOÀN 2025 (Theo phạm vi) BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Phạm vi 1 PHẦN 5 XANH HÓA SẢN XUẤT Phạm vi 2 Như vậy, việc kiểm soát và giảm thiểu phát thải khí nhà kính cần tập trung vào nhóm nguồn phát thải trực tiếp trong sản xuất và lĩnh vực thép – lĩnh vực đóng vai trò chủ lực trong hoạt động của Tập đoàn Hòa Phát. Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trìn"}\n{"id": "B-A-06dc02df178a98d1", "claim": "Nhóm nông nghiệp, bất động sản và điện máy gia dụng có mức phát thải thấp hơn đáng kể.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 CƠ CẤU NGÀNH BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 THÉP NÔNG NGHIỆP BẤT ĐỘNG SẢN ĐIỆN MÁY GIA DỤNG CÙNG CÁC CÔNG TY THÀNH VIÊN Điểm nổi bật về phát triển bền vững năm 2025 THÉP NÔNG NGHIỆP BẤT ĐỘNG SẢN ĐIỆN MÁY GIA DỤNG Giải thưởng và vinh danh năm 2025 TP. Đà Nẵng"}\n{"id": "B-A-1922bfd99a84abed", "claim": "Nhóm nguồn phát thải trực tiếp từ quá trình sản xuất chiếm tỷ trọng rõ rệt, phản ánh đặc thù ngành nghề và quy mô sản xuất của tập đoàn.", "evidence": "Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận "}\n{"id": "B-A-4b72b9f1164044af", "claim": "Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải.", "evidence": "23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. HÒA PHÁT trò nền tảng để cung cấp nguồn thép chất lượng cao cho các công trình hạ tầng và máy móc, tuy nhiên nó tiêu tốn một lượng lớn năng lượng và giải phóng nhiều khí thải CO2 do quá trình đốt cháy nhiên liệu hóa thạch. Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép"}\n{"id": "B-A-c1a4820eee0dd3b4", "claim": "Từ tháng 9 năm 2025, Hòa Phát Dung Quất 2 đi vào vận hành, Tổng sản lượng thép thô tăng 24%, tương ứng với mức tăng phát thải của toàn tập đoàn.", "evidence": "TABLE | THÁNG 2/2016 THÁNG 1/2021 NĂM 2025 • Thành lập Công ty CP Phát triển Nông Lò cao số 4 – KLH gang thép Hòa Phát THÁNG 9: Dung Quất được đưa vào hoạt động, đánh nghiệp Hòa Phát, quản lý, chi phối hoạt động lò cao số 2 của dự án NĂM 2019 dấu việc hoàn thành toàn bộ dự án, nâng NĂM 2023 của tất cả các Công ty trong nhóm nông Dung Quất 2 đi vào hoạt tổng công suất thép thô của Hòa Phát lên nghiệp (gồm thức ăn chăn nuôi, chăn nuôi). Hai lò cao đầu tiên • THÁNG 8: Hòa Phát động, đánh dấu hoàn 8,5 triệu tấn/năm, lớn nhất khu vực Đông của Dự án Hòa Phát thành đồng bộ dự án Khu • Hoàn thành đầu tư giai đoạn 3 – Khu liên chính thức cung cấp Nam Á. Dung Quất chính thức liên hợp sản xuất gang hợp"}\n{"id": "B-A-24ff5f2fdcf5b883", "claim": "Công nghệ sản xuất điện năng này đã tăng tính cạnh tranh cho sản phẩm, góp phần bảo vệ môi trường, giảm lượng phát thải khí nhà kính.", "evidence": "Phát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng hàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày 05/02/2025. Việc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng lượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm môi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia. Chuyển đổi số tại tập đoàn Hòa Phát năm 2024 PHẦN 5 XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu HÀNH ĐỘNG GIẢM PHÁT THẢI KHÍ NHÀ KÍNH & TỐI ƯU NĂNG LƯỢNG Quản lý môi trường TĂNG SỬ DỤNG THÉP PHẾ LIỆU"}\n{"id": "B-A-a9ee34023c56da8b", "claim": "Công tác quản lý môi trường của Tập đoàn tập trung vào các khía cạnh trọng yếu gồm quản lý và sử dụng nước, quản lý chất thải và nước thải, kiểm soát chất lượng không khí nhằm giảm thiểu tác động môi trường, tuân thủ quy định pháp luật và hướng tới tăng trưởng xanh.", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 KIỂM SOÁT CHẤT LƯỢNG KHÔNG KHÍ KIỂM SOÁT CHẤT LƯỢNG KHÔNG KHÍ TRONG SẢN XUẤT THÉP Thực hiện theo tiêu chuẩn hệ thống quản lý môi trường ISO 14001, Tập đoàn Hòa Phát đặt mục tiêu giảm thiểu tác động không khí thông qua ứng dụng các công nghệ kiểm soát tiên tiến. QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường,"}\n{"id": "B-A-e5c83b777c8716f5", "claim": "Tập đoàn dành tỷ trọng lớn trong chi phí đầu tư tại các khu liên hợp và nhà máy, trang trại cho các hạng mục về bảo vệ môi trường, xử lý khí thải, nước thải, chất thải rắn, tích hợp trong hệ thống máy móc thiết bị, dây chuyền sản xuất.", "evidence": "Chất thải khác Hành động giảm thiểu và thích ứng với biến đổi khí hậu Quản lý môi trường Chất thải khác (Chất thải sinh hoạt, CÁC LOẠI Bụi thải từ các hệ thống xử lý bụi, khí thải Tại các nhà máy thép, hệ thống xử lý khí thải khép kín được đầu tư đồng bộ để kiểm soát chất lượng không khí từ đầu vào đến đầu ra của quá trình sản xuất, đảm bảo kiểm soát chặt chẽ các thông số về bụi, khí SOx, NOx và CO trước khi thải ra môi trường. Khí thải sau xử lý luôn được quan trắc tự động liên tục (CEMS) và truyền dữ liệu trực tiếp về Sở Nông nghiệp và Môi trường theo quy định pháp luật. Việc thu gom và tái sử dụng bụi trong quá trình luyện thép được thực hiện triệt để."}\n{"id": "B-A-02e7cd1ece325735", "claim": "Quá trình này giúp giảm đáng kể tải lượng ô nhiễm và loại bỏ mùi hôi của chất thải trước khi xả ra môi trường.", "evidence": "Bằng cách tránh cho vật nuôi ăn protein dư thừa, bài tiết dưới dạng urê được giảm đáng kể. Nhờ đó, quá trình chuyển hóa thành Amoniac loại khí độc gây ô nhiễm không khí và ảnh hưởng xấu đến sức khỏe hô hấp – được giảm thiểu ngay tại chuồng nuôi, giúp tiết kiệm chi phí thức ăn và cải thiện chất lượng môi trường. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Tất cả nguồn hơi này hội tụ về hệ thống quản lý năng lượng tập trung để làm quay máy phát điện công suất lớn. Chu trình khép kín này giúp chuyển hóa toàn bộ phụ phẩm phát sinh thành điện năng sạch, phục vụ trực tiếp cho các hoạt động sản xuất tại nhà máy. Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ư"}\n{"id": "B-A-c175b312231962b7", "claim": "Quan trọng hơn, biogas tạo ra khí Mê-tan CH4 có thể được sử dụng làm nguồn năng lượng tái tạo để đun nấu hoặc phát điện, giúp giảm chi phí vận hành và đồng thời giảm phát thải khí nhà kính mạnh ra môi trường.", "evidence": "Tất cả nguồn hơi này hội tụ về hệ thống quản lý năng lượng tập trung để làm quay máy phát điện công suất lớn. Chu trình khép kín này giúp chuyển hóa toàn bộ phụ phẩm phát sinh thành điện năng sạch, phục vụ trực tiếp cho các hoạt động sản xuất tại nhà máy. Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ưu hóa chi phí vận hành mà còn giảm thiểu đáng kể phát thải nhiệt và khí thải ra môi trường. Tấm làm mát sử dụng chất liệu gỗ bền bỉ và thấm hút tốt hơn, hướng đến người dùng mong muốn giải pháp làm mát tiết kiệm điện, thân thiện với môi trường. Quy trình sản xuất tại nhà máy cũng được cải tiến để hạn chế khí thải gây hiệu ứng nhà kính, trong đó dây chuyền sơn tự độn"}\n{"id": "B-A-37ed12d0f055dd90", "claim": "Nhiệt độ luôn nằm ở mức ổn định từ 18 - 25oC nhờ hệ thống điều khiển tự động, đàn vật nuôi ít bị mệt, giảm stress nhiệt giúp tiêu hóa hiệu quả và giảm phát thải khí Amoniac trong chất thải.", "evidence": "Phòng ấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm tối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết lập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng trưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của biến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải qua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) trong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Bằng cách tránh cho vật nuôi ăn protein dư thừa, bài tiết dưới dạng urê được giảm đáng kể. Nhờ đó, quá trình chuyển hóa thành Amoniac loại khí độ"}\n{"id": "B-A-2357fa095b3eeecb", "claim": "Việc vệ sinh, thu gom chất thải được thực hiện thường xuyên giúp trang trại vận hành ổn định, sạch sẽ và thân thiện với môi trường.", "evidence": "Các trang trại vận hành theo mô hình khép kín, đảm bảo vệ sinh và có hệ thống xử lý chất thải đồng bộ. Đội ngũ R&D xây dựng chương trình dinh dưỡng và thú y chuyên biệt, cùng với việc thực hiện truy xuất nguồn gốc đầy đủ, nhằm cung cấp sản phẩm thịt heo an toàn, chất lượng cao, đáp ứng tiêu chuẩn vệ sinh an toàn thực phẩm. 83 Tập đoàn dành tỷ trọng lớn trong chi phí đầu tư tại các khu liên hợp và nhà máy, trang trại cho các hạng mục về bảo vệ môi trường, xử lý khí thải, nước thải, chất thải rắn, tích hợp trong hệ thống máy móc thiết bị, dây chuyền sản xuất. Quản lý môi trường CON NGƯỜI: Nâng cao nhận thức và năng lực Tập đoàn thường xuyên triển khai các chương trình đào tạo về quy định pháp "}\n{"id": "B-A-1706deb2c53528a5", "claim": "Việc xây dựng hầm biogas là giải pháp kinh tế tuần hoàn để xử lý chất thải đầu ra.", "evidence": "Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh tế ; nghiên cứu và tìm kiếm các giải pháp điều chỉnh cơ cấ"}\n{"id": "B-A-efa452189b3135ba", "claim": "Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường,", "evidence": "Công tác quản lý môi trường của Tập đoàn tập trung vào các khía cạnh trọng yếu gồm quản lý và sử dụng nước, quản lý chất thải và nước thải, kiểm soát chất lượng không khí nhằm giảm thiểu tác động môi trường, tuân thủ quy định pháp luật và hướng tới tăng trưởng xanh. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT TUÂN THỦ PHÁP LUẬT VỀ MÔI TRƯỜNG Lượng nước thải Quản lý nước 303-5 Lượng nước tiêu thụ Quản lý nước QUẢN LÝ CHẤT THẢI RẮN GRI 3 – Chủ đề trọng yếu 2021 3-3 Quản lý chủ đề trọng yếu Quản lý chất thải rắn"}\n{"id": "B-A-33dc30efb635fcfe", "claim": "Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn.", "evidence": "yêu cầu vận hành và nghĩa vụ pháp lý. Hệ thống EHS được triển khai phù hợp với mô hình hoạt động tại các công ty thành viên của Tập đoàn, thể hiện cam kết phát triển có trách nhiệm và bền vững dựa trên nền tảng minh bạch và cải tiến liên tục dựa trên các nguyên tắc cốt lõi sau: AN TOÀN LÀ TRÊN HẾT: Nhận diện và loại bỏ mọi rủi ro tiềm ẩn trước khi xảy ra sự cố, đảm bảo môi trường làm việc an toàn cho toàn bộ người lao động. PHẦN 5 XANH HÓA SẢN XUẤT TUÂN THỦ VÀ ĐẠO ĐỨC: Thực hiện nghiêm túc các quy định của pháp luật, quy chế nội bộ và quy tắc đạo đức kinh doanh của Công ty; Quản trị doanh nghiệp tốt Tuân thủ các quy định pháp luật và minh bạch về Thuế, Môi trường, Lao động, An toàn và Sức kh"}\n{"id": "B-A-201de7021378f57a", "claim": "Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn.", "evidence": "Nhà máy chế biến tinh quặng sắt Minh Sơn đã chủ động nghiên cứu, thay đổi công nghệ để phù hợp với tính chất của quặng nguyên khai theo phân kỳ khai thác của mỏ sắt Sàng Thần nhằm sản xuất ra tinh quặng sắt có chất lượng tốt, phù hợp với công nghệ sản xuất thép hiện nay Khu liên hợp sản xuất gang thép Hòa Phát Hải Dương. 46 Dung Quất 1 và 2. Công nghệ Lò đáy quay giúp tách bụi lò cao thành quặng sắt hoàn nguyên trực tiếp (DRI), là nguồn quặng đầu vào quay trở lại sản xuất thép và bột oxit kẽm là sản phẩm phụ có giá trị cao, có thể bán cho các nhà máy luyện kẽm, ứng dụng mạ kẽm hoặc ngành hóa chất. Hệ thống đồng thời thu hồi nhiệt hơi để phát điện. (Nghìn tấn) Tổng khối lượng chất thải rắn ph"}\n{"id": "B-A-8bda85e6af05003b", "claim": "với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn.", "evidence": "Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý Quản lý chất thải rắn GRI 306: Waste 2020 306-1 Phát sinh chất thải và các tác động đáng kể liên quan đến chất thải Quản lý chất thải rắn 306-2 Quản lý các tác động đáng kể liên quan đến chất thải Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra"}\n{"id": "B-A-e63473c5de57bcb4", "claim": "Chất thải rắn trong mô hình sản xuất thép tuần hoàn không còn được xem là gánh nặng chôn lấp, đổ bỏ mà trở thành tài nguyên được tận dụng tối đa và mang lại hiệu quả kinh tế.", "evidence": "Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng. Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. Trụ cột Quản trị và Đổi mới là kim chỉ nam đảm bảo sự minh bạch, trách nhiệm và hiệu quả trong mọi hoạt động của Tập đoàn. 104,6 57,8 55% Cộng 7.085 6.982 99% Năm 2025, tổng lượng chất thải rắn phát sinh của Tập đoàn là 7.085 nghì"}\n{"id": "B-A-7b78e97b9a7a9da4", "claim": "Hiện nay, tỷ lệ tái chế chất thải rắn của nhóm thép đạt hơn 99%.", "evidence": "Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý Quản lý chất thải rắn Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải."}\n{"id": "B-A-7f7a25d69452ad60", "claim": "Chất thải khác Hành động giảm thiểu và thích ứng với biến đổi khí hậu Quản lý môi trường Chất thải khác (Chất thải sinh hoạt, CÁC LOẠI Bụi thải từ các hệ thống xử lý bụi, khí thải", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 PHẦN 5 XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu HÀNH ĐỘNG GIẢM PHÁT THẢI KHÍ NHÀ KÍNH & TỐI ƯU NĂNG LƯỢNG Quản lý môi trường TĂNG SỬ DỤNG THÉP PHẾ LIỆU"}\n{"id": "B-A-398be4f8ea6dbf47", "claim": "Theo đó, các chất thải rắn chứa sắt sẽ được thu gom và xử lý, trở thành nguyên vật liệu đưa ngược lại quy trình sản xuất thép.", "evidence": "Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý Quản lý chất thải rắn Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh tế ; nghiên cứu và tìm kiếm các giải pháp điều chỉnh cơ cấu tiêu thụ nguyên liệu và qua đó giảm phát sinh chất thải. Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng các hệ thống tuần hoàn khép kín, tái chế nước và xử lý triệt để để tuân thủ quy chuẩn xả thải."}\n{"id": "B-A-dc9c4e3447e7a555", "claim": "Việc tối ưu hiệu quả sử dụng sắt trước hết giúp giảm tiêu hao nguyên vật liệu thô, đồng thời giải quyết vấn đề chất thải rắn.", "evidence": "Tập đoàn hướng tới từng bước chuyển đổi mô hình sản xuất và quản trị, gắn liền với cải tiến công nghệ, tối ưu hiệu quả sử dụng tài nguyên, và giảm thiểu phát thải ra môi trường. Với Khu liên hợp Gang thép tại Dung Quất (chiếm 75% công suất thép thô toàn tập đoàn), Hòa Phát đã áp dụng hệ thống ERP để kiểm soát nguyên liệu từ đầu vào đến đầu ra với các chỉ số đo cụ thể, có thể trích xuất báo cáo bất cứ lúc nào theo thời gian thực giúp giám sát chặt chẽ để kiểm soát tiêu hao nguyên vật liệu, tối ưu hóa hiệu suất năng lượng. ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Tập đoàn lập kế hoạch triển khai các dự án đề xuất tăng cường phối phế liệu cùng gang lỏng trong lò "}\n{"id": "B-A-e68a11fa21c95395", "claim": "Chất thải nguy hại được xử lý đúng quy định Các chất thải nguy hại (mỡ, giẻ lau dính dầu, hóa chất) đều được lưu giữ an toàn, dán nhãn đầy đủ và chuyển giao cho đơn vị có giấy phép xử lý theo đúng quy định pháp luật.", "evidence": "Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại) Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại)"}\n{"id": "B-A-b37eea29cb336963", "claim": "Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo.", "evidence": "Quan trọng hơn, biogas tạo ra khí Mê-tan CH4 có thể được sử dụng làm nguồn năng lượng tái tạo để đun nấu hoặc phát điện, giúp giảm chi phí vận hành và đồng thời giảm phát thải khí nhà kính mạnh ra môi trường. Tại các trang trại chăn nuôi heo, chất lượng không khí được kiểm soát bằng kết hợp nhiều biện pháp: BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 KIỂM SOÁT KHẨU PHẦN ĂN CỦA VẬT NUÔI QUẢN LÝ MÔI TRƯỜNG 5.2 KIỂM SOÁT CHẤT LƯỢNG KHÔNG KHÍ TRONG CHĂN NUÔI kiện yếm khí để các vi sinh vật phân hủy chất hữu cơ. Quá trình này giúp giảm đáng kể tải lượng ô nhiễm và loại bỏ mùi hôi của chất thải trước khi xả ra môi trường."}\n{"id": "B-A-94d9b57abbddc59c", "claim": "Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%.", "evidence": "Chất thải rắn trong mô hình sản xuất thép tuần hoàn không còn được xem là gánh nặng chôn lấp, đổ bỏ mà trở thành tài nguyên được tận dụng tối đa và mang lại hiệu quả kinh tế. Hiện nay, tỷ lệ tái chế chất thải rắn của nhóm thép đạt hơn 99%. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường,"}\n{"id": "B-A-21118d24552f1e51", "claim": "Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi tài nguyên.", "evidence": "Chính sách này đã trở thành động lực quan trọng để gia tăng sự gắn kết nhân viên, khuyến khích cải tiến liên tục và nâng cao hiệu quả công việc. Nhờ vậy, nhiều sáng kiến đổi mới đã được áp dụng, góp phần tối ưu hóa hoạt động sản xuất – kinh doanh của Tập đoàn. NGHỈ SỐ NGƯỜI ĐỦ ĐIỀU KIỆN TIÊU CHÍ TỶ LỆ Từ khi thành lập đến nay, Tập đoàn đã liên tục mở rộng hệ sinh thái thép từ thượng nguồn đến hạ nguồn, nâng cao năng lực sản xuất, không ngừng tập trung nghiên cứu nâng cao chất lượng sản phẩm, cho ra các mác sản phẩm mới ngày càng chuyên sâu, phức tạp, đáp ứng rộng hơn các loại nhu cầu về thép cho hạ tầng đất nước. Năm 2025, Hòa Phát tập trung toàn bộ nguồn lực để đưa dự án Khu liên hợp gang t"}\n{"id": "B-A-033a7cd333b74fda", "claim": "Hành động giảm thiểu và thích ứng với biến đổi khí hậu Chuyển đổi các chất thải có khối lượng lớn thành sản phẩm phụ", "evidence": "PHẦN 5 XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu HÀNH ĐỘNG GIẢM PHÁT THẢI KHÍ NHÀ KÍNH & TỐI ƯU NĂNG LƯỢNG Quản lý môi trường TĂNG SỬ DỤNG THÉP PHẾ LIỆU HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024"}\n{"id": "B-A-360ec4290cc78f87", "claim": "Tổng khối lượng chất thải phát sinh 7,085 Tổng khối lượng được sử dụng cho các mục đích khác thay vì bị vứt bỏ/tiêu hủy 6,982 Tổng khối lượng chất thải được xử lý thải bỏ", "evidence": "Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát sinh Tổng khối lượng chất thải được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.981.878 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại)"}\n{"id": "B-A-bda493d526f8be80", "claim": "Việc áp dụng EPD sẽ hỗ trợ Hòa Phát trong mua sắm xanh, phát triển thép carbon thấp, đồng thời nâng cao năng lực cạnh tranh trên thị trường trong nước và quốc tế.", "evidence": "Đây là bước đi then chốt nhằm giảm dấu chân carbon trên mỗi đơn vị sản phẩm, trực tiếp nâng cao năng lực cạnh tranh cho thép và các sản phẩm công nghiệp của Tập đoàn trên thị trường quốc tế, đặc biệt là tại các khu vực khắt khe về tiêu chuẩn môi trường. Quản lý môi trường TRIỂN KHAI ĐIỆN MẶT TRỜI ÁP MÁI TẠI CÁC NHÀ MÁY 38 Tuy nhiên, lĩnh vực Gang thép của Hòa Phát vẫn phải đối mặt với không ít thách thức đến từ cả thị trường nội địa và thị trường quốc tế. Báo cáo của Thành viên HĐQT Độc lập Đánh giá của HĐQT về hoạt động của Ban Tổng Giám đốc Thứ nhất, thép giá rẻ của Trung Quốc và Ấn Độ tạo áp lực cạnh tranh lớn cho các doanh nghiệp thép trong nước nói chung, Hòa Phát nói riêng."}\n{"id": "B-A-cdfa9479b0d1451b", "claim": "104,6 57,8 55% Cộng 7.085 6.982 99% Năm 2025, tổng lượng chất thải rắn phát sinh của Tập đoàn là 7.085 nghìn tấn, trong đó chất thải nguy hại chiếm 3%, chất thải không nguy hại chiếm 97%.", "evidence": "7,085 Chất thải không nguy hại 6,838 Chất thải nguy hại 247 LƯỢNG CHẤT THẢI THEO CÁC HƯỚNG XỬ LÝ NĂM 2025 CHỈ TIÊU NĂM 2025 (Nghìn tấn) Dự án không chỉ góp phần giảm chất thải rắn và tăng hiệu quả tái chế nội bộ, mà còn thể hiện định hướng của Hòa Phát trong kinh tế tuần hoàn và sử dụng tài nguyên bền vững. Triệu lít N/A Chất thải nguy hại Tấn 246.867 306-3 Chất thải phát sinh Chất thải không nguy hại Tấn 6.838.336 306-3 Chất thải phát sinh Chất thải khác"}\n{"id": "B-A-3d451ebe866942f0", "claim": "Quản lý môi trường Trong tổng số hơn 7 triệu tấn chất thải rắn phát sinh, có đến gần 6,98 triệu tấn đã được thu hồi và đưa quay trở lại chu trình sản xuất hoặc chuyển đổi mục đích sử dụng.", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 Tái chế, tái sử dụng chất thải rắn thành nguyên vật liệu quay lại quá trình sản xuất Các giải pháp tái chế, tái sử dụng nội bộ tập trung vào việc phân loại và xử lý các chất thải ngay tại nguồn phát sinh. Theo đó, các chất thải rắn chứa sắt sẽ được thu gom và xử lý, trở thành nguyên vật liệu đưa ngược lại quy trình sản xuất thép. Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát sinh Tổng khối lượng chất thải được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.981.878 306-4 Chất thải được chuyển giao khỏi quy trình xử lý"}\n{"id": "B-A-7acf996114679082", "claim": "Con số này cho thấy chất thải không còn là gánh nặng môi trường mà đã thực sự trở thành nguồn tài nguyên đầu vào quan trọng, giúp Tập đoàn tối ưu hóa chi phí và giảm thiểu tác động tiêu cực đến hệ sinh thái.", "evidence": "Tất cả nguồn hơi này hội tụ về hệ thống quản lý năng lượng tập trung để làm quay máy phát điện công suất lớn. Chu trình khép kín này giúp chuyển hóa toàn bộ phụ phẩm phát sinh thành điện năng sạch, phục vụ trực tiếp cho các hoạt động sản xuất tại nhà máy. Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ưu hóa chi phí vận hành mà còn giảm thiểu đáng kể phát thải nhiệt và khí thải ra môi trường. QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách"}\n{"id": "B-A-4cc20f396e90d6ab", "claim": "Những thành quả này phản ánh nỗ lực không ngừng của Tập đoàn trong việc mở rộng cơ hội lãnh đạo cho phụ nữ, đồng thời đặt nền móng cho sự tiến bộ bền vững hơn trong các lĩnh vực vốn được coi là “nam giới chiếm ưu thế”.", "evidence": "phục vụ quản lý tốt hơn. Theo Ban lãnh đạo Tập đoàn, điều này sẽ giúp cho Ban điều hành của Tập đoàn tối ưu hóa hoạt động, sâu sát các lĩnh vực kinh doanh hơn, đáp ứng tốt yêu cầu phát triển của Hòa Phát trong tương lai. Hòa Phát có chủ trương trẻ hóa đội ngũ cán bộ lãnh đạo, quản lý. Việc đào tạo nhân sự quản lý kế cận trong độ tuổi 30-35 là hoạt động được lãnh đạo Tập đoàn rất quan tâm chú trọng, trong đó ưu tiên những cán bộ gắn bó lâu dài, thấu hiểu văn hóa, cách thức vận hành của Hòa Phát. Là một trong những tập đoàn công nghiệp hàng đầu Việt Nam, Hòa Phát coi quản trị tốt và đạo đức kinh doanh không chỉ là nền tảng của sự phát triển bền vững, mà còn là yếu tố cốt lõi để củng cố niềm ti"}\n{"id": "B-A-5e9b2e879ae6326e", "claim": "Đào tạo lập Báo cáo kiểm kê khí nhà kính ...", "evidence": "QĐ-TTg ban hành danh mục lĩnh vực, cơ sở phát thải khí nhà kính phải thực hiện kiểm kê khí nhà kính và nộp báo cáo kiểm kê khí nhà kính cấp cơ sở về cơ quan có thẩm quyền từ tháng 3/2025. Đến tháng 01/2025, BSI đã trao giấy kiểm tra, xác nhận, công bố khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 cho các công ty thành viên của Tập đoàn Hòa Phát tham gia kiểm kê. Việc hoàn thành kiểm kê khí nhà kính là bước đi quan trọng trong chiến lược phát triển bền vững của Tập đoàn. HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024"}\n{"id": "B-A-02a4e78913dbbc31", "claim": "Nỗ lực này thể hiện tinh thần trách nhiệm xã hội và cam kết vì cộng đồng của Tập đoàn Hòa Phát trong hành trình phát triển bền vững.", "evidence": "TRÁCH NHIỆM XÃ HỘI – CỘNG ĐỒNG CHẶNG ĐƯỜNG NỐI YÊU THƯƠNG: MẸ ĐỠ ĐẦU BÁO CÁO THƯỜNG NIÊN - 2024 Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng Phát triển. Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, hướng tới sự phát triển bền vững. 4 PHÁT TRIỂN SẢN PHẨM MỚI 5 CHUYỂN ĐỔI SỐ TẠI TẬP ĐOÀN HÒA PHÁT NĂM 2024 6 TRÁCH NHIỆM XÃ HỘI - CỘNG ĐỒNG"}\n{"id": "B-A-486b8ea09669ba4d", "claim": "Thông qua những chương trình trọng điểm, bền vững và giàu tính nhân văn, Hòa Phát khẳng định cam kết phát triển cộng đồng toàn diện, để mỗi bước tiến trong kinh doanh luôn gắn liền với giá trị xã hội được sẻ chia và lan tỏa.", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 24 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 23"}\n{"id": "B-A-f7f9228a3da88f3c", "claim": "Các khoản đầu tư xã hội năm 2025 thể hiện cam kết lâu dài của Hòa Phát trong việc tạo ra giá trị bền vững cho cộng đồng.", "evidence": "PHÁT TRIỂN CỘNG ĐỒNG 6.4 GRI 204, 411, 413] Với triết lý “Hòa hợp cùng phát triển”, Hòa Phát tích hợp mục tiêu phát triển kinh tế với trách nhiệm xã hội, hướng đến tăng trưởng bền vững và tạo giá trị chung cho các bên liên quan. Mọi hoạt động đều mang đến mục tiêu tạo ra giá trị lâu dài cho xã hội, lan tỏa tinh thần sẻ chia và đóng góp thiết thực cho đời sống người dân tại những địa phương nơi HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 4. GẮN KẾT CỘNG ĐỒNG Chúng tôi tạo cơ hội việc làm tại địa phương và đóng góp cho sự phát triển xã hội, môi trường của cộng đồng nơi Tập đoàn hoạt động."}\n{"id": "B-A-12825d29f5f1202f", "claim": "Là một trong những tập đoàn công nghiệp hàng đầu Việt Nam, Hòa Phát coi quản trị tốt và đạo đức kinh doanh không chỉ là nền tảng của sự phát triển bền vững, mà còn là yếu tố cốt lõi để củng cố niềm tin của cổ đông, khách hàng, đối tác và xã hội.", "evidence": "GIÁ TRỊ CỐT LÕI: Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng phát triển. Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, đảm bảo hài hòa lợi ích của các bên liên quan trên cùng một con thuyền, hướng tới sự phát triển bền vững. Đặc biệt, Tập đoàn Hòa Phát đã xây dựng được mối quan hệ đối tác bền vững, lâu dài, tin tưởng như người một nhà với các đại lý bán hàng song hành cùng Tập đoàn từ những ngày đầu thành lập. VỀ TẬP ĐOÀN HÒA PHÁT SỨ MỆNH Cung cấp sản phẩm dẫn đầu, góp phần nâng cao chất lượng cuộc sống, đạt được sự tin yêu của khách hàng. Điểm nổi bật về phát triển bền vững năm 2025 GIÁ T"}\n{"id": "B-A-1ce15c0d836614b4", "claim": "Trong thời gian tới, Tập đoàn sẽ xây dựng tiêu chí đánh giá rõ ràng hơn nhằm chuẩn hóa quy trình quản lý rủi ro, đặc biệt là đối với các rủi ro liên quan đến phát triển bền vững.", "evidence": "CÁC RỦI RO TRONG QUÁ TRÌNH HOẠT ĐỘNG BÁO CÁO THƯỜNG NIÊN - 2024 Với đặc thù của doanh nghiệp sản xuất đa ngành, đa lĩnh vực, quy mô hoạt động rộng khắp lãnh thổ Việt Nam, Tập đoàn Hòa Phát không thể tránh khỏi những rủi ro phát sinh trong quá trình hoạt động. Để phòng ngừa hiệu quả các rủi ro này, Tập đoàn đề ra những biện pháp cụ thể với từng nhóm rủi ro trọng yếu, đặc biệt là trong phòng ngừa rủi ro về biến động giá nguyên nhiên liệu, chính sách, pháp lý và nhân sự. Do đó, Tập đoàn luôn ưu tiên các chương trình đào tạo nhằm phát triển trình độ chuyên môn và kỹ năng cho người lao động. Gắn kết với các bên liên quan Hòa Phát đặt mục tiêu trở thành một nhà sản xuất công nghiệp với chất lượng "}\n{"id": "B-A-262279ee4841429b", "claim": "Cam kết này không chỉ nhằm mục đích tối đa hóa sự hài lòng của khách hàng mà còn bảo vệ người tiêu dùng cuối cùng bằng cách cung cấp những sản phẩm thép chất lượng cao, an toàn, góp phần vào sự bền vững của các công trình xây dựng và cơ sở hạ tầng quốc gia.", "evidence": "Cuối cùng, các trang trại vận hành mô hình khép kín với hệ thống xử lý chất thải đồng bộ và truy xuất nguồn gốc đầy đủ, củng cố cam kết cung cấp sản phẩm an toàn, chất lượng cao, bền vững, bảo vệ sức khỏe người tiêu dùng CHĂN NUÔI LỢN Tập đoàn áp dụng quy trình An toàn sinh học (ATSH) nghiêm ngặt và tuân thủ các tiêu chuẩn như VietGAHP và tiêu chuẩn Châu Âu (EU). Đàn heo được tiêm phòng định kỳ bằng vaccine đã được phê duyệt, tuyệt đối không sử dụng kháng sinh cấm hay chất kích thích tăng trưởng. BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện"}\n{"id": "B-A-73c21d1f75562890", "claim": "Các trang trại vận hành theo mô hình khép kín, đảm bảo vệ sinh và có hệ thống xử lý chất thải đồng bộ.", "evidence": "Cuối cùng, các trang trại vận hành mô hình khép kín với hệ thống xử lý chất thải đồng bộ và truy xuất nguồn gốc đầy đủ, củng cố cam kết cung cấp sản phẩm an toàn, chất lượng cao, bền vững, bảo vệ sức khỏe người tiêu dùng CHĂN NUÔI LỢN Tập đoàn áp dụng quy trình An toàn sinh học (ATSH) nghiêm ngặt và tuân thủ các tiêu chuẩn như VietGAHP và tiêu chuẩn Châu Âu (EU). Đàn heo được tiêm phòng định kỳ bằng vaccine đã được phê duyệt, tuyệt đối không sử dụng kháng sinh cấm hay chất kích thích tăng trưởng. Chất thải khác Hành động giảm thiểu và thích ứng với biến đổi khí hậu Quản lý môi trường Chất thải khác (Chất thải sinh hoạt, CÁC LOẠI Bụi thải từ các hệ thống xử lý bụi, khí thải"}\n{"id": "B-A-4b33d9479d600546", "claim": "Dòng máy lọc nước HyperS được phát triển theo hướng tối ưu năng lượng và thân thiện môi trường, nổi bật với công nghệ làm lạnh Block sử dụng gas R600a không gây hại tầng ozone, hệ điện cực titanium phủ Platinum nâng cao độ bền và cụm lõi Ultra H8.0 tối ưu lưu lượng.", "evidence": "Công nghệ xanh và quy trình sản xuất tối ưu hóa được áp dụng trên nhiều dòng sản phẩm của Điện máy Gia dụng Hòa Phát. Máy lọc nước Hòa Phát sử dụng công nghệ làm lạnh bằng BLOCK với Gas R600a, giúp tiết kiệm điện năng và giảm thiểu tác động tới tầng ozone. Song song đó, ngành hàng tủ đông đánh dấu bước tiến mới với dòng sản phẩm ứng dụng công nghệ thép ZAM. Tháng 01/2025, Hòa Phát giới thiệu bộ sưu tập máy lọc nước Hydrogen ion kiềm HyperS, dòng sản phẩm tích hợp công nghệ điện phân có màng ngăn với điện cực Titan phủ Platinum, tăng gấp đôi hàm lượng hydrogen trong nước, mở ra giải pháp nước sạch chất lượng cao cho người tiêu dùng. Công nghệ điện phân trong sản phẩm giúp cân bằng độ pH, trun"}\n{"id": "B-A-d8c48fab5acc501d", "claim": "tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành;", "evidence": "23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Nâng cao trải nghiệm khách hàng Giảm tiêu hao tài nguyên và phát thải gián tiếp Tăng khả năng thích ứng với biến động thị trường Chiến lược này đồng thời đóng góp trực tiếp vào 3 trụ cột ESG:"}\n{"id": "B-A-d318b2cd4fdef14c", "claim": "đồng thời hỗ trợ cảnh báo sớm khi các chỉ số môi trường vượt ngưỡng và nâng cao tính minh bạch, tuân thủ trong quản lý phát thải.", "evidence": "Công tác quản lý môi trường của Tập đoàn tập trung vào các khía cạnh trọng yếu gồm quản lý và sử dụng nước, quản lý chất thải và nước thải, kiểm soát chất lượng không khí nhằm giảm thiểu tác động môi trường, tuân thủ quy định pháp luật và hướng tới tăng trưởng xanh. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT TUÂN THỦ PHÁP LUẬT VỀ MÔI TRƯỜNG Các công ty thành viên của Tập đoàn đã bố trí nhân sự chuyên trách thực hiện kiểm tra định kỳ việc vận hành máy móc, thiết bị tại nhà máy, đảm bảo tuân thủ các ngưỡng cho phép tại các nguồn thải. Trong năm 2025, các đơn vị thành viên của Hòa Phát không phát sinh vi phạm liên quan đến bảo vệ môi trường. 42"}\n{"id": "B-A-e4cb4e9d03c932cc", "claim": "2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt khe và áp lực cạnh tranh về chi phí, Tập đoàn Hòa Phát xác định Chuyển đổi số là một trong những trụ cột chiến lược nhằm nâng cao năng lực cạnh tranh dài hạn, tối ưu hiệu quả sản xuất – quản trị và hỗ trợ thực hiện các mục tiêu phát triển bền vững.", "evidence": "Trong năm 2025, Hòa Phát đã xây dựng Chiến lược Phát triển Bền vững để định hướng hoạt động kinh doanh theo các xu thế dài hạn toàn cầu về ứng phó biến đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao năng lực cạnh tranh và khả năng thích ứng của Tập đoàn trong ngành thép đang biến đổi mạnh mẽ. Trọng tâm chiến lược tập trung vào ba trụ cột ESG chính: Xanh hóa Sản xuất (Greening Our Production), Phát triển Con người (Caring for Hòa Phát hướng tới duy trì năng lực cạnh tranh bền vững trong quá trình chuyển đổi hướng tới trung hòa carbon, đồng thời tiếp tục khẳng định cam kết về chất lượng sản phẩm dẫn đầu thị trường. Bên cạnh đó, chúng tôi dành sự quan tâm sâu sắc,"}\n{"id": "B-A-625307e2dbbaa64a", "claim": "Nâng cao trải nghiệm khách hàng Giảm tiêu hao tài nguyên và phát thải gián tiếp Tăng khả năng thích ứng với biến động thị trường Chiến lược này đồng thời đóng góp trực tiếp vào 3 trụ cột ESG:", "evidence": "Trong năm 2025, Hòa Phát đã xây dựng Chiến lược Phát triển Bền vững để định hướng hoạt động kinh doanh theo các xu thế dài hạn toàn cầu về ứng phó biến đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao năng lực cạnh tranh và khả năng thích ứng của Tập đoàn trong ngành thép đang biến đổi mạnh mẽ. Trọng tâm chiến lược tập trung vào ba trụ cột ESG chính: Xanh hóa Sản xuất (Greening Our Production), Phát triển Con người (Caring for Các sản phẩm mới tập trung vào ứng dụng công nghệ tiên tiến, nâng cao trải nghiệm người dùng và tính bền vững. Điện máy Gia dụng Hòa Phát đã thể hiện tầm nhìn chiến lược với định hướng “Tiến vào kỷ nguyên xanh - Sống khỏe bền vững” trong nă"}\n{"id": "B-A-ee86a3216182f219", "claim": "Ứng dụng đã góp phần nâng cao trải nghiệm người dùng và từng bước hoàn thiện hệ sinh thái số tại công ty.", "evidence": "Quan hệ Cổ đông Bước sang năm 2025, Hòa Phát tiếp tục đẩy mạnh cơ cấu tổ chức, phân công phân nhiệm, kiểm soát công việc tại trang trại và phòng ban. Mục tiêu trọng tâm hướng đến nâng cao tăng trọng trong chăn nuôi, cải thiện chất lượng thịt, đẩy mạnh cung ứng bò thịt ra thị trường với giá cả và chất lượng tốt nhất, từ đó nâng cao hiệu quả kinh doanh của công ty. 52 Năm 2025 đánh dấu giai đoạn mở rộng triển khai và nâng cao hiệu quả sử dụng các hệ thống đã đầu tư, đồng thời từng bước hình thành nền tảng quản trị số toàn Tập đoàn. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 DỰ ÁN CHUYỂN ĐỔI SỐ TẠI CÁC CÔNG TY THÀNH VIÊN"}\n{"id": "B-A-a1d3e928df3e26e6", "claim": "Với định hướng rõ ràng và phương pháp triển khai có kiểm soát, Tập đoàn đang tiến từng bước vững chắc trên lộ trình xây dựng doanh nghiệp sản xuất công nghiệp vận hành hiện đại, minh bạch và bền vững.", "evidence": "Rủi ro vận hành: phát sinh trong quá trình sản xuất và kinh doanh. Tác động tiềm ẩn và biện pháp ứng phó cho từng nhóm rủi ro được xác định, triển khai trong hệ thống quản lý tổng thể, và công bố trong Báo cáo thường niên 2025. Trong thời gian tới, Tập đoàn sẽ xây dựng tiêu chí đánh giá rõ ràng hơn nhằm chuẩn hóa quy trình quản lý rủi ro, đặc biệt là đối với các rủi ro liên quan đến phát triển bền vững. Quản trị Doanh nghiệp và Quản lý rủi ro XÃ HỘI: Chuẩn hóa quy trình và tự động hóa giúp nâng cao năng suất lao động, giảm áp lực thủ tục hành chính, cải thiện môi trường làm việc và tăng tính minh bạch trong đánh giá hiệu quả công việc. TẦM NHÌN & ĐỊNH HƯỚNG CHIẾN LƯỢC CHUYỂN ĐỔI SỐ Tập đoàn "}\n{"id": "B-A-862b29dacc984eaa", "claim": "Chiến lược, chính sách và thực tiễn 2-24 Triển khai cam kết chính sách Được trình bày theo các chủ đề trọng yếu về phát triển bền vững trong ba trụ cột chính 2-25 Các quy trình khắc phục tác động tiêu cực", "evidence": "2-21 Tỷ lệ tổng thù lao hàng năm Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-22 Tuyên bố về chiến lược phát triển bền vững Thông điệp từ Tổng Giám Đốc 2-23 Cam kết chính sách Được trình bày theo các chủ đề trọng yếu về phát triển bền vững trong ba trụ cột chính CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN"}\n{"id": "B-A-acb1ae31f130d78e", "claim": "GRI 102 – Biến đổi khí hậu 2025 102-6 Phát thải KNK Phạm vi 1 Kết quả kiểm kê phát thải khí nhà kính năm 2025 102-8 Cường độ phát thải Do các nhà máy sử dụng công nghệ sản xuất thép khác nhau và có nhiều loại sản phẩm khác nhau, chúng tôi đang sắp xếp nguồn lực phù hợp để thu thập và tính toán chỉ tiêu này cho các kỳ báo cáo sau.", "evidence": "103-5 Giảm tiêu thụ năng lượng Hành động giảm thiểu và thích ứng với biến đổi khí hậu 102-4 Mục tiêu và tiến độ giảm phát thải khí nhà kính Đây là báo cáo năm đầu tiên của chúng tôi, lộ trình giảm phát thải chi tiết vẫn đang được nghiên cứu xây dựng 102-5 Phát thải KNK Phạm vi 1 Kết quả kiểm kê phát thải khí nhà kính năm 2025 TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính "}\n{"id": "B-A-fa5140aa5584f619", "claim": "Năng lượng tiêu thụ là năng lượng tái tạo tại các % 0,03% địa điểm", "evidence": "GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo tại các địa điểm GRI 3 – Chủ đề trọng yếu 2021 103-1 Chính sách và cam kết năng lượng Phương pháp quản lý 103-2 Tiêu thụ năng lượng và năng lượng tự tạo ra trong tổ chức Kết quả tiêu thụ năng lượng năm 2025 GRI 103 – Năng lượng 2025"}\n{"id": "B-A-e4473d240f16ca3d", "claim": "ESG was integrated across investment teams and the ESG Engine was used to make portfolio decisions.", "evidence": "The order states DWS lacked formalized controls to verify use of the ESG Engine and documentation of material ESG factors. The SEC order found inconsistent documented compliance and inadequate implementation of ESG integration policy requirements."}\n{"id": "B-A-722b39d4308b14aa", "claim": "Join us in creating a more sustainable future.", "evidence": "The court found 15 of 19 assessed environmental statements misleading, including vague claims and overly positive presentation of SAF and reforestation. The annual report says KLM continued the Fly Responsibly campaign and set targets for further CO2 and noise reduction."}\n{"id": "B-A-a3c043c3d78a029b", "claim": "được mối quan hệ đối tác bền vững, lâu dài, tin tưởng như người một", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 TẦM NHÌN Trở thành Tập đoàn sản xuất công nghiệp với chất lượng dẫn đầu, trong đó Thép là lĩnh vực cốt lõi. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT SỨ MỆNH Cung cấp sản phẩm dẫn đầu, góp phần nâng cao chất lượng cuộc sống, đạt được sự tin yêu của khách hàng. Điểm nổi bật về phát triển bền vững năm 2025 GIÁ TRỊ CỐT LÕI Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng phát triển. Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, đảm bảo hài hòa lợi ích của các bên liên quan trên cùng một con thuyền, hướng tới sự phát triển "}\n{"id": "B-A-863c8a143dfe627b", "claim": "TỪ CƠ CHẾ ĐIỀU CHỈNH BIÊN GIỚI CARBON (CBAM)", "evidence": "triển Các rủi ro trong Cơ chế điều chỉnh biên giới Carbon (CBAM) do Liên minh châu Âu (EU) ban hành cũng là thách thức không nhỏ khi Hòa Phát đang xuất khẩu vào thị trường này. Cơ chế này đặt ra yêu cầu các nhà nhập khẩu tại EU phải mua chứng chỉ CBAM tương ứng với lượng khí thải các-bon phát sinh trong quá trình sản xuất hàng hóa nhập khẩu, bao gồm các sản phẩm thép. CBAM sắp kết thúc giai đoạn chuyển tiếp vào cuối năm 2025 và dự kiến sẽ chính thức áp dụng vào năm 2026. Việc áp dụng chính thức CBAM từ đầu năm 2026 sẽ làm gia tăng đáng kể chi phí xuất khẩu do phải mua chứng chỉ phát thải các-bon. Điều này không chỉ ảnh hưởng đến lợi nhuận mà còn tạo áp lực lớn trong việc duy trì thị phần tại"}\n{"id": "B-A-62668a33ef44f525", "claim": "QĐ-TTg ban hành danh mục lĩnh vực, cơ sở phát thải khí nhà kính phải thực hiện", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-b61d6f87e3f7e36a", "claim": "trong chiến lược phát triển bền vững của Tập đoàn.", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 24 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 23"}\n{"id": "B-A-7bdf4e2c4a1b2ad4", "claim": "Cơ chế điều chỉnh biên giới Carbon (CBAM) do Liên minh châu Âu (EU) ban hành", "evidence": "triển Các rủi ro trong Cơ chế điều chỉnh biên giới Carbon (CBAM) do Liên minh châu Âu (EU) ban hành cũng là thách thức không nhỏ khi Hòa Phát đang xuất khẩu vào thị trường này. Cơ chế này đặt ra yêu cầu các nhà nhập khẩu tại EU phải mua chứng chỉ CBAM tương ứng với lượng khí thải các-bon phát sinh trong quá trình sản xuất hàng hóa nhập khẩu, bao gồm các sản phẩm thép. CBAM sắp kết thúc giai đoạn chuyển tiếp vào cuối năm 2025 và dự kiến sẽ chính thức áp dụng vào năm 2026. Việc áp dụng chính thức CBAM từ đầu năm 2026 sẽ làm gia tăng đáng kể chi phí xuất khẩu do phải mua chứng chỉ phát thải các-bon. Điều này không chỉ ảnh hưởng đến lợi nhuận mà còn tạo áp lực lớn trong việc duy trì thị phần tại"}\n{"id": "B-A-dfb13bd1a70b24c3", "claim": "khẩu do phải mua chứng chỉ phát thải các-bon.", "evidence": "m kê khí nhà kính là năng lượng, giao thông vận tải, xây dựng, công nghiệp, nông nghiệp, và xử lý chất thải. Là một tập đoàn đa ngành, Tập đoàn Hòa Phát có hơn 10 Công ty thành viên nằm trong danh mục các doanh nghiệp phải thực hiện kiểm kê khí nhà kính bắt buộc. Thông tin cơ bản Thông tin cổ phiếu Ngành nghề kinh doanh Sơ đồ tổ chức Đồng thời, Tập đoàn Hòa Phát nghiên cứu về cách tính và phân bổ hạn ngạch nêu tại dự thảo nghị định của Chính phủ và tìm hiểu về thị trường tín chỉ các-bon để xây dựng các bước đi thích hợp nhằm đảm bảo tuân thủ quy định pháp luật và tối ưu chi phí. Tiếp đó, tháng 01/2025, Thủ tướng Chính phủ tiếp tục ban hành Quyết định số 232/QĐ-TTg Phê duyệt Đề án thành lập v"}\n{"id": "B-A-b9bb060dc1a90963", "claim": "(4) Lạm phát trên thế giới đã hạ nhiệt nhưng chưa bền vững.", "evidence": "thép lớn nhất trên thế giới. Thành tựu này củng cố vị thế của Việt Nam là quốc gia số 1 Đông Nam Á và thứ 11 trên thế giới về năng lực sản xuất thép thô, khẳng định vai trò tiên phong của Hòa Phát trong ngành công nghiệp quốc gia và nâng tầm vị thế ngành luyện kim của Việt Nam trong khu vực và quốc tế. Sự phát triển của Hòa Phát không chỉ được đo lường bằng sản lượng hay quy mô. Với triết lý “Hòa hợp cùng Phát triển”, Hòa Phát xác định rằng tăng trưởng kinh tế phải song hành cùng phát triển bền vững. Tôi rất vinh dự giới thiệu Báo cáo phát triển bền vững năm 2025 của Tập đoàn, như một minh chứng cho nỗ lực thấu hiểu sâu sắc và ghi nhận những mối quan tâm của các bên liên quan. Thông qua đánh"}\n{"id": "B-A-6009ec5b62d66ad2", "claim": "thép hàm lượng carbon thấp sản xuất các sản phẩm như vỏ đồ hộp, đồ gia dụng,", "evidence": "óa thạch. • Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu. Khi các sản phẩm từ thép như vỏ ô tô, khung nhà hay thiết bị điện tử...hết thời hạn sử dụng, Giải thưởng và vinh danh năm 2025 thay vì trở thành rác thải, chúng được thu gom, phân tách và đưa vào lò điện để nấu Sản xuất thép chảy lại thành thép lỏng, tạo ra các sản phẩm thép mới.. Giai đoạn này gắn với công "}\n{"id": "B-A-bc75fd6734db7ec5", "claim": "này đã góp phần bảo vệ môi trường, giảm lượng phát thải CO2, tự chủ tới 90%", "evidence": "TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP BÁO CÁO THƯỜNG NIÊN - 2024 Năm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang thép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 tỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện cho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – Công ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế từ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất. Bốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng. Năm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau đúc có nhiệt độ 750-900 độ C sẽ được chuyển n"}\n{"id": "B-A-1e64de091c98461b", "claim": "đã góp phần bảo vệ môi trường, giảm lượng phát thải CO2, tự chủ tới 90%", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-a27eb115351398d2", "claim": "ở xã hội quy mô 31 ha tại KCN Yên Mỹ II, hướng tới mục tiêu phát triển bền vững", "evidence": "Lĩnh vực bất động sản hoàn thành vượt chỉ tiêu doanh thu và lợi nhuận sau thuế so với kế hoạch được giao. Năm 2024, doanh thu lĩnh vực bất động sản tăng 123%, lợi nhuận sau thuế tăng 118% so với năm 2023. Các khu công nghiệp Hòa Phát đã bàn giao được hơn 57,3 ha đất, lấp đầy gần như toàn bộ diện tích đã đầu tư hạ tầng kỹ thuật. TỶ TRỌNG LỢI NHUẬN SAU THUẾ TỶ TRỌNG DOANH THU BÁN HÀNG NĂM 2024 NĂM 2024 BÁO CÁO THƯỜNG NIÊN - 2024 Bất động sản 6% Bất động sản 2% Nông nghiệp Nông nghiệp Tổng quỹ đất khu công nghiệp (KCN) của Hòa Phát được quy hoạch hiện đạt gần 1.500 ha, tập trung tại các tỉnh Hưng Yên, Hà Nam và Bắc Giang. Tập đoàn hiện sở hữu và vận hành 4 KCN, bao gồm: KCN Phố Nối A có quy mô "}\n{"id": "B-A-63e72fae68f5f0a0", "claim": "Với mục tiêu phát triển bền vững và đảm bảo an sinh xã hội, Tập đoàn đang tích", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-39f1e9aca546aa6c", "claim": "đã hoàn thành kiểm kê phát thải khí nhà kính cấp Tập đoàn với 18 đơn vị thuộc 5", "evidence": "KIỂM KÊ KHÍ NHÀ KÍNH VÀ RỦI RO TỪ CƠ CHẾ ĐIỀU CHỈNH BIÊN GIỚI CARBON (CBAM) BIỆN PHÁP BÁO CÁO THƯỜNG NIÊN - 2024 Tập đoàn Hòa Phát đã phối hợp với BSI – Tổ chức chứng nhận quốc tế hàng đầu của Vương quốc Anh để đào tạo kiểm kê khí nhà kính cho các công ty thành viên trong Tập đoàn. Tháng 8/2024, Thủ tướng Chính phủ đã chính thức ký Quyết định số 13/2024/ QĐ-TTg ban hành danh mục lĩnh vực, cơ sở phát thải khí nhà kính phải thực hiện kiểm kê khí nhà kính và nộp báo cáo kiểm kê khí nhà kính cấp cơ sở về cơ quan có thẩm quyền từ tháng 3/2025. Đến tháng 01/2025, BSI đã trao giấy kiểm tra, xác nhận, công bố khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 cho các công ty thành viên của Tập đoàn Hòa P"}\n{"id": "B-A-a6507fd6f8d7453f", "claim": "kính là bước đi quan trọng trong chiến lược phát triển bền vững của Tập đoàn.", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 24 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 23"}\n{"id": "B-A-3ccef6e65e213d78", "claim": "chương trình chống biến đổi khí hậu cũng như các mục tiêu phát triển bền vững", "evidence": "PHÁT TRIỂN CỘNG ĐỒNG 6.4 GRI 204, 411, 413] Với triết lý “Hòa hợp cùng phát triển”, Hòa Phát tích hợp mục tiêu phát triển kinh tế với trách nhiệm xã hội, hướng đến tăng trưởng bền vững và tạo giá trị chung cho các bên liên quan. Mọi hoạt động đều mang đến mục tiêu tạo ra giá trị lâu dài cho xã hội, lan tỏa tinh thần sẻ chia và đóng góp thiết thực cho đời sống người dân tại những địa phương nơi Hòa Phát hoạt động. CÁC YẾU TỐ CHÍNH TRONG PHƯƠNG PHÁP TIẾP CẬN CỦA HÒA PHÁT BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 HỖ TRỢ CÓ TRỌNG ĐIỂM: Các chương trình được thiết kế để tập trung giúp đỡ những nhóm đối tượng yếu thế, dễ bị tổn thương trong xã hội để nguồn lực được sử dụng hiệu quả và tạo ra tác động rõ rệ"}\n{"id": "B-A-8d92775d56faf798", "claim": "toán và báo cáo phát thải, loại bỏ khí nhà kính của một doanh nghiệp hay một tổ", "evidence": "quả kiểm kê phát thải khí nhà kính năm 2025 GRI 102 – Biến đổi khí hậu 2025 102-6 Phát thải KNK Phạm vi 1 Kết quả kiểm kê phát thải khí nhà kính năm 2025 102-8 Cường độ phát thải Do các nhà máy sử dụng công nghệ sản xuất thép khác nhau và có nhiều loại sản phẩm khác nhau, chúng tôi đang sắp xếp nguồn lực phù hợp để thu thập và tính toán chỉ tiêu này cho các kỳ báo cáo sau. KIỂM KÊ KHÍ NHÀ KÍNH VÀ RỦI RO TỪ CƠ CHẾ ĐIỀU CHỈNH BIÊN GIỚI CARBON (CBAM) BIỆN PHÁP BÁO CÁO THƯỜNG NIÊN - 2024 Tập đoàn Hòa Phát đã phối hợp với BSI – Tổ chức chứng nhận quốc tế hàng đầu của Vương quốc Anh để đào tạo kiểm kê khí nhà kính cho các công ty thành viên trong Tập đoàn. Tháng 8/2024, Thủ tướng Chính phủ đã chín"}\n{"id": "B-A-096bc47f8c7b0387", "claim": "để tối ưu sản xuất, tiết kiệm năng lượng, giảm phát thải khí nhà kính tại tất cả các", "evidence": "kính tại tất cả các khu liên hợp sản xuất gang thép. Hòa Phát đã tính toán kỹ lưỡng và dành tới 30% vốn đầu tư cho các hạng mục môi trường. Hàng loạt các giải pháp và công nghệ mới nhất được Hòa Phát ứng dụng để tối ưu sản xuất cũng như tiết kiệm năng lượng, tự chủ điện năng cho sản xuất với sản lượng phát điện hàng năm đạt hàng tỷ kWh. 82 TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP BÁO CÁO THƯỜNG NIÊN - 2024 Năm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang thép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 tỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện cho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – Công ty"}\n{"id": "B-A-37ad4ac3e6023dd1", "claim": "như góp phần giảm phát thải như: Nghiên cứu, ứng dụng, phối hợp và thực hiện", "evidence": "rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng. Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. Trụ cột Quản trị và Đổi "}\n{"id": "B-A-863eacd8d0e426f9", "claim": "Đề tài Chế tạo đồ gá kẹp để thao tác dễ dàng khi thực hiện sửa chữa các trục phủ", "evidence": "CÔNG TY CỔ PHẦN TẬP ĐOÀN HÒA PHÁT THUYẾT MINH BÁO CÁO TÀI CHÍNH HỢP NHẤT CHO NĂM KẾT THÚC NGÀY 31 THÁNG 12 NĂM 2024 (TIẾP THEO) Mẫu B 09 – DN/HN (Ban hành theo Thông tư số 202/2014/TT-BTC ngày 22/12/2014 của Bộ Tài chính) Hàng tồn kho được sản xuất và để bán Đối với thành phẩm và sản phẩm sản xuất dở dang, giá gốc bao gồm nguyên vật liệu, chi phí nhân công trực tiếp và các chi phí sản xuất chung đã được phân bổ. Tập đoàn áp dụng phương pháp kê khai thường xuyên để hạch toán hàng tồn kho. PHẦN 7 BCTC HỢP NHẤT ĐÃ KIỂM TOÁN Bất động sản xây dựng để bán trong tương lai Giá gốc được tính theo phương pháp đích danh và bao gồm chi phí đất, chi phí xây dựng cơ sở hạ tầng, các chi phí trực tiếp khác "}\n{"id": "B-A-ebba2fe135c5500c", "claim": "giảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.", "evidence": "ản xuất mang lại hiệu quả về kinh tế cũng như tiết kiệm nguồn tài nguyên nước. Tự chủ 90% điện sản xuất thép Phát triển sản phẩm mới Chuyển đổi số tại tập đoàn Hòa Phát năm 2024 Công ty tìm kiếm, thử nghiệm các loại vật tư, nguyên nhiên phụ liệu mới cho sản xuất ở tất cả các công đoạn. Việc này được triển khai nhằm nâng cao chất lượng sản phẩm, tối ưu chi phí sản xuất. Nổi bật là các đề tài: Sử dụng HC-FeMn75 cho một số mác thép đúc TSC đã tiết giảm được chi phí hợp kim sản xuất; Tận dụng lại xỉ thải của các mác thép khử nhôm để làm phụ gia cho luyện thép mang lại hiệu quả cao về tiết giảm chi phí sản xuất và bảo vệ môi trường. Trách nhiệm xã hội - cộng đồng xử lý nước Nhà máy HRC1 giúp tối "}\n{"id": "B-A-0bfcaa13c78a1877", "claim": "Quất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng", "evidence": "hép Hòa Phát Dung Quất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng mà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng đến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt dư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên liệu hóa thạch”. Trong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, tự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện tự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. Tổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và Dung Quất tương đương một nhà m"}\n{"id": "B-A-5fcd4bedba6f176e", "claim": "mà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng", "evidence": "của sản phẩm, bao gồm thu thập, kiểm chứng và báo cáo dữ liệu theo quy định của Bộ Công Thương, Bộ Nông Nghiệp và Môi trường và theo yêu cầu của khách hàng, đối tác. Hằng năm, Ban xây dựng kế hoạch giảm phát thải, theo dõi tiến độ triển khai và đề xuất các giải pháp chiến lược nhằm nâng cao hiệu quả năng lượng và giảm thiểu tác động môi trường. Báo cáo được thiết lập dựa trên các yêu cầu của tiêu chuẩn ISO 14064-1, theo các hướng dẫn của Ban liên chính phủ về biến đổi khí hậu (IPCC Guidelines) và đáp ứng các yêu cầu về giảm phát thải khí nhà kính của Cơ quan quản lý được đề cập trong Nghị định 06/2022/ND-CP, Nghị định 119/2024/ND-CP. Đối với số liệu kiểm kê KNK của Tập đoàn, phương pháp hợp "}\n{"id": "B-A-38d84c8b1e5b2930", "claim": "Đây là giải pháp bền vững, tận dụng nguồn nhiệt", "evidence": "ăng trưởng hàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày 05/02/2025. Việc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng lượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm môi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia. Chuyển đổi số tại tập đoàn Hòa Phát năm 2024 Trách nhiệm xã hội - cộng đồng Các Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại của các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa trong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí sản xuất, phát triển bền vững. Với sự mở rộng củ"}\n{"id": "B-A-4cfee303397907c2", "claim": "Hiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.", "evidence": "hép Hòa Phát Dung Quất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng mà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng đến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt dư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên liệu hóa thạch”. Trong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, tự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện tự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. Tổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và Dung Quất tương đương một nhà m"}\n{"id": "B-A-addfd084a2f3f57a", "claim": "Đối với mác thép tròn trơn SS400 là dòng thép carbon thấp, thường dùng trong", "evidence": "PHÁT TRIỂN SẢN PHẨM MỚI BÁO CÁO THƯỜNG NIÊN - 2024 Sản phẩm của Tập đoàn Hòa Phát hiện đã khẳng định vị thế vững chắc tại thị trường Việt Nam và được xuất khẩu tới hàng chục quốc gia, vùng lãnh thổ khắp năm châu. Để đáp ứng nhu cầu ngày càng đa dạng của thị trường, Hòa Phát luôn coi trọng phát triển sản phẩm mới, không ngừng nâng cao chất lượng theo các tiêu chuẩn khắt khe nhất. Ngoài ra, các kỹ sư công nghệ của Thép Hòa Phát Dung Quất còn nghiên cứu sản xuất mác ER70S6 và ER70S3. Đây cũng là các mác thép cho sản xuất dây hàn, sản phẩm chất lượng cao đòi hỏi kỹ thuật, quy trình công nghệ nấu luyện thép chặt chẽ. Tại Thép Hòa Phát Hải Dương, trong năm 2024 đã nghiên cứu và sản xuất mác thép S"}\n{"id": "B-A-8c8a0b10de508e6c", "claim": "kỷ nguyên xanh - Sống khỏe bền vững” trong năm 2025.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 TƯƠNG LAI NGÀNH THÉP CỦA CHÚNG TÔI Tập đoàn Hòa Phát là tập đoàn đa ngành hoạt động trong nhiều lĩnh vực bao gồm Thép, Nông nghiệp, Bất động sản, Điện máy gia dụng. Trong đó, ngành Thép giữ vai trò cốt lõi và là trụ cột chính trong mô hình kinh doanh. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT VAI TRÒ CỦA THÉP Vai trò của thép trong đời sống hiện đại là vô cùng thiết yếu và không thể thay thế. Thép, với đặc tính bền vững, chịu lực cao, và dễ dàng tái chế, là xương sống của mọi nền kinh tế và kiến tạo nên môi trường sống của chúng ta. Thép không chỉ là vật liệu mà còn là nền tảng vật chất định hình nên mọi mặt của xã h"}\n{"id": "B-A-9f7b0e0a2f04758a", "claim": "phát triển sản phẩm theo 3 tiêu chí: kỷ nguyên xanh, sống khỏe và bền vững.", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Quản trị Doanh nghiệp và Quản lý rủi ro Quản lý chất lượng Phát triển đổi mới sáng tạo LĨNH VỰC ĐIỆN MÁY GIA DỤNG Các sản phẩm điện lạnh và điện máy gia dụng được thực hiện đánh giá tác động đến sức khỏe người tiêu dùng trong suốt vòng đời sản phẩm. Trong lĩnh vực điện lạnh, các sản phẩm được thiết kế đảm bảo an toàn về điện, cơ khí và cách nhiệt, đáp ứng đầy đủ các tiêu chuẩn kỹ thuật hiện hành. Tất cả vật liệu và linh kiện sử dụng đều có nguồn gốc rõ ràng, không chứa các chất độc hại như CFC, HCFC, chì hoặc thủy ngân. Các bộ phận chính của sản phẩm được kiểm tra và thử nghiệm nghiêm ngặt nhằm ngăn ngừa nguy "}\n{"id": "B-A-189eefa637b3e2e9", "claim": "Công nghệ xanh và quy trình sản xuất tối ưu hóa được áp dụng trên nhiều dòng sản", "evidence": "Về công nghệ sản xuất, Công ty đã triển khai thành công dòng cám heo cho Farm (heo thịt và heo nái) với công nghệ thiết lập khẩu phần cân bằng giữa đạm thô thấp và acid amin cao. Đồng thời tối ưu hóa quy trình với hệ thống phun acid hữu cơ lỏng, giúp tiết kiệm chi phí nguyên liệu và nhân công. Công ty đẩy mạnh số hóa thông qua việc bổ sung các module phần mềm chuyên dụng, tăng cường công tác quản lý dữ liệu. Đặc biệt, nhà máy tại Đồng Nai đạt chứng nhận GlobalGap đã khẳng định cam kết của công ty trong việc áp dụng công nghệ tiên tiến và quản lý chất lượng theo tiêu chuẩn quốc tế. Tình hình đầu tư Dự án lớn Công ty cũng chú trọng đẩy mạnh số hóa trong quản lý và vận hành. Phần mềm Bravo được"}\n{"id": "B-A-f13392164b6a5d62", "claim": "Cong ty cam ket manh me huong toi trung hoa carbon va da giam 30% phat thai CO2 trong nam 2024.", "evidence": "Theo so lieu do dac nam 2024, tong phat thai CO2 cua cong ty giam 12% so voi nam 2023."}\n{"id": "B-A-6b88a39d403e3648", "claim": "TỔNG QUAN VỀ PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 10 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12"}\n{"id": "B-A-dbea0ba8409f0698", "claim": "CÁC LĨNH VỰC TRỌNG YẾU VỀ PHÁT TRIỂN BỀN VỮNG", "evidence": "6.4 PHÁT TRIỂN CỘNG ĐỒNGS BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 6 CÁC LĨNH VỰC TRỌNG YẾU VỀ PHÁT TRIỂN BỀN VỮNG PHÁT TRIỂN CON NGƯỜI 6.1 SỨC KHỎE VÀ AN TOÀN LAO ĐỘNG 6.2 QUẢN TRỊ NGUỒN NHÂN LỰC 6.3 NHÂN QUYỀN 52 5.2 QUẢN LÝ MÔI TRƯỜNGE 5 CÁC LĨNH VỰC TRỌNG YẾU VỀ PHÁT TRIỂN BỀN VỮNG XANH HÓA SẢN XUẤT 5.1 HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU"}\n{"id": "B-A-458c69699d51d2bb", "claim": "3.1 | Điểm nổi bật về phát triển bền vững năm 2025", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 10 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12"}\n{"id": "B-A-038a18e61d161c39", "claim": "PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 25 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 23"}\n{"id": "B-A-b0184721c1570287", "claim": "4.3 | Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 25 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 23"}\n{"id": "B-A-2ac31f2b70761f5c", "claim": "Báo cáo Phát triển Bền vững năm 2025 của Tập đoàn Hòa Phát được xây dựng phù hợp với Bộ Tiêu chuẩn", "evidence": "ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025 3.1 Tự chủ 90% nhu cầu điện tại Khu liên hợp Gang thép Hòa Phát Dung Quất Hoàn thành kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 cho toàn Tập đoàn và được BSI thẩm định số liệu. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Tái sử dụng 98% nước tại Khu liên hợp Gang thép Hòa Phát Dung Quất Tái chế, tái sử dụng chất thải rắn MÔI TRƯỜNG nhóm thép và nông nghiệp đạt trên 99% Điểm nổi bật về phát triển bền vững năm 2025 Đóng góp gần 13.000 tỷ đồng vào ngân sách Nhà nước Số giờ đào tạo trung bình cho mỗi nhân Giải thưởng và vinh danh năm 2025 viên là 38,52 giờ mỗi năm Thành viên của các Hiệp hội ngành nghề Đóng góp hơn 12"}\n{"id": "B-A-ffc1e69635d6229f", "claim": "Báo cáo Phát triển Bền vững 2021 của Sáng kiến Báo cáo Toàn cầu (GRI), áp dụng cho giai đoạn từ ngày", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-959d9d939dafceaa", "claim": "Báo cáo cũng phản ánh các tiêu chuẩn của Hội đồng Chuẩn mực Kế toán Bền vững (SASB) dành", "evidence": "Ổ CHỨC VÀ CÁC CÔNG TY CÙNG NGÀNH: để xác định mức độ ưu tiên. Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG Gắn kết với các bên liên quan BƯỚC 4 tại 15 doanh nghiệp lớn cùng ngành trên thế giới, các mối quan tâm về phát triển bền vững đã được xác định. Quá trình này tham chiếu 05 bộ khung phát triển bền vững quốc ƯU TIÊN CÁC CHỦ ĐỀ TRỌNG YẾU: 12 chủ đề được phân loại thành 3 nhóm: Chủ đề tế hàng đầu, trong đó có Bộ Tiêu chuẩn GRI Chung (phiên bản cập nhật), Đánh giá Phát ưu tiên (tác động cao), Chủ đề trọng tâm (tác động trung bình), và Chủ đề nền tảng (tác triển Bền vững Doanh nghiệp của S&P Global CSA, Hội đồng Chuẩn mực Kế toán Bền động thấp). Các chủ đề"}\n{"id": "B-A-8149dfdf7e1e86dd", "claim": "Hiệu quả phát triển bền vững của Tập đoàn Hòa Phát được trình bày trong báo cáo này bao quát toàn bộ", "evidence": "TIÊU CHUẨN CHUNG NỘI DUNG CÓ NÊU TRONG BÁO MÃ CÔNG BỐ CHỦ ĐỀ TIÊU ĐỀ CÔNG BỐ GRI CÁO PHÁT TRIỂN PHẦN THAM CHIẾU/ LÝ DO LOẠI TRỪ GRI BỀN VỮNG NĂM 2025 2-16 Truyền đạt các mối quan tâm chính Quản trị Doanh nghiệp & Quản lý rủi ro 2-17 Kiến thức chung của cơ quan quản trị cao nhất Đã trình bày tại Báo cáo thường niên của tập đoàn Hòa Phát 2025 2-18 Đánh giá hiệu quả hoạt động của cơ quan trị cao nhất Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-19 Chính sách thù lao Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-20 Quy trình xác định thù lao Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-21 Tỷ lệ tổng thù lao hàng năm Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-"}\n{"id": "B-A-bfd5ef11ddf9295c", "claim": "Mọi câu hỏi, thắc mắc liên quan đến các sáng kiến phát triển bền vững, nội dung báo cáo,", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-12df1bf558f12018", "claim": "Điện thoại: +84 243.974.7751 | Email: esg@hoaphat.com.vn", "evidence": "TÍN DỤNG Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát VẤN ĐỀ QUAN TÂM PHẢN HỒI CỦA TẬP ĐOÀN & KẾT QUẢ THỰC HIỆN 2025 PHƯƠNG THỨC TƯƠNG TÁC Gắn kết với các bên liên quan • Năng lực tài chính và khả năng thanh toán nợ • Biện pháp quản lý và hạn chế các rủi ro tiềm ẩn (rủi ro về môi trường, xã hội và quản trị) • Doanh nghiệp đảm bảo hoàn thành các nghĩa vụ đã thỏa thuận • Tuân thủ cam kết, điều khoản trong hợp đồng • Cung cấp thông tin một cách kịp thời, chính xác và minh bạch. • Duy trì nghiêm ngặt kỷ luật tài chính, đảm bảo khả năng thanh toán. Tuân thủ nghiêm túc hạn thanh toán. • Công bố thông tin tài chính và hoạt động minh bạch định kỳ. • Công bố thông tin báo cáo các chỉ số ESG v"}\n{"id": "B-A-cd98b77da53df194", "claim": "Tại Hòa Phát, chúng tôi theo đuổi sự hòa hợp và phát triển bền vững, không ngừng kiến tạo giá trị vì một tương lai phồn vinh hơn", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi cam kết duy trì môi trường làm việc an toàn, tôn trọng và hòa nhập. Hòa Phát chú trọng phát triển và nuôi dưỡng đội ngũ nhân sự sáng tạo, sẵn sàng thích ứng và đổi mới, để xây dựng một doanh nghiệp nơi mọi nhân viên đều được trân trọng, ghi nhận và truyền cảm hứng. QUẢN TRỊ NGUỒN PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT NHÂN LỰC Chúng tôi cam kết đảm bảo minh bạch, trách nhiệm giải trình và liêm chính trong mọi cấp độ tổ chức. Hệ thống quản trị được thiết"}\n{"id": "B-A-dd08e2ec7e34f695", "claim": "nay, Tập đoàn đã liên tục mở rộng hệ sinh thái thép từ thượng nguồn đến hạ nguồn, nâng cao năng lực sản xuất, không ngừng tập", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-d7b2bcf61aec0b89", "claim": "xác định rằng tăng trưởng kinh tế phải song hành cùng phát triển bền vững.", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-33f70b8820f47731", "claim": "Tôi rất vinh dự giới thiệu Báo cáo phát triển bền vững", "evidence": "MỤC LỤC BÁO CÁO PHÁT TRIỂN BỀN VỮNG NĂM 2025 TỔNG QUAN VỀ PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT CÁC LĨNH VỰC TRỌNG YẾU VỀ PHÁT TRIỂN BỀN VỮNG 1 5 GIỚI THIỆU BÁO CÁO PHÁT TRIỂN BỀN VỮNG XANH HOÁ SẢN XUẤT 5.1 | Hành động giảm thiểu và thích ứng với biến đổi khí hậu 2 5.2 | Quản lý môi trường THÔNG ĐIỆP CỦA TỔNG GIÁM ĐỐC 3 6 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT PHÁT TRIỂN CON NGƯỜI 6.1 | Sức khoẻ và an toàn lao động 3.1 | Điểm nổi bật về phát triển bền vững năm 2025 6.2 | Quản trị nguồn nhân lực 3.2 | Giải thưởng và vinh danh năm 2025 6.3 | Nhân quyền 3.3 | Thành viên của các Hiệp hội ngành nghề 6.4 | Phát triển cộng đồng 3.4 | Hoạt động kinh doanh tại Tập đoàn 4 7 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀ"}\n{"id": "B-A-b6aa5d4344c300c5", "claim": "Thông qua đánh giá trọng yếu nội bộ và bên ngoài, chúng tôi đã đo lường và xác định những vấn đề bền vững quan trọng, sắp xếp", "evidence": "của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan kết hợp với đóng góp mà Hòa Phát mang lại cho xã hội và môi trường. Cách tiếp cận này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then chốt nhất. cấp và nhà đầu tư. Để thu thập ý kiến về mức độ ưu tiên của các chủ đề ESG trọng yếu, Hòa Phát đã tiếp cận các bên liên quan thông qua phỏng vấn và khảo sát, đồng thời ghi nhận các đóng góp liên quan đến điểm mạnh, thách thức và lĩnh vực Hòa Phát cần cải thiện trong quá trình phát triển bền vững. BƯỚC 3 PHÂN TÍCH PHẢN HỒI: Kết quả từ các cuộc phỏng vấn và khảo sát đã được tổng hợp, QUY TRÌNH XÁC ĐỊNH CÁC CHỦ ĐỀ TRỌNG YẾU ĐƯỢC "}\n{"id": "B-A-067bd3622048ec40", "claim": "thành các trụ cột chính để phản ánh nhất quán và xuyên suốt trong Báo cáo Phát triển Bền vững năm nay và các năm tiếp theo.", "evidence": "TIÊU CHUẨN CHUNG NỘI DUNG CÓ NÊU TRONG BÁO MÃ CÔNG BỐ CHỦ ĐỀ TIÊU ĐỀ CÔNG BỐ GRI CÁO PHÁT TRIỂN PHẦN THAM CHIẾU/ LÝ DO LOẠI TRỪ GRI BỀN VỮNG NĂM 2025 2-16 Truyền đạt các mối quan tâm chính Quản trị Doanh nghiệp & Quản lý rủi ro 2-17 Kiến thức chung của cơ quan quản trị cao nhất Đã trình bày tại Báo cáo thường niên của tập đoàn Hòa Phát 2025 2-18 Đánh giá hiệu quả hoạt động của cơ quan trị cao nhất Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-19 Chính sách thù lao Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-20 Quy trình xác định thù lao Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-21 Tỷ lệ tổng thù lao hàng năm Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-"}\n{"id": "B-A-59344db2eaeaf5e0", "claim": "đó, xanh hóa sản xuất, phát triển con người, quản trị và đổi mới là ba ưu tiên hàng đầu.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 4. GẮN KẾT CỘNG ĐỒNG Chúng tôi tạo cơ hội việc làm tại địa phương và đóng góp cho sự phát triển xã hội, môi trường của cộng đồng nơi Tập đoàn hoạt động. 1. NGUỒN CUNG ỨNG CÓ TRÁCH NHIỆM Chúng tôi ưu tiên hợp tác với các nhà cung cấp trong nước, đảm bảo chất lượng và thúc đẩy các thực hành bền vững ngay từ khâu đầu vào. 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT 7 1 5. KHÁCH HÀNG LÀ TRUNG TÂM, CHẤT LƯỢNG LÀ ƯU TIÊN HÀNG ĐẦU Chúng tôi đồng hành cùng khách hàng để tạo ra những giải pháp bền vững, sáng tạo và lâu dài, đáp ứng kỳ vọng ngày càng cao của xã hội. Do đó chất lượng sản phẩm luôn là ưu tiên hàng đầu của Tập đoàn"}\n{"id": "B-A-a6986e4508418cf7", "claim": "Báo cáo phát triển bền vững này phác họa", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 25 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 30"}\n{"id": "B-A-3f864c6cc8ae1404", "claim": "Trụ cột Xanh hóa Sản xuất hướng tới hài hòa giữa tăng trưởng công nghiệp và bảo vệ môi", "evidence": "PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 CHUYỂN ĐỔI SỐ TRONG SẢN XUẤT VÀ QUẢN TRỊ DOANH NGHIỆP PHƯƠNG PHÁP QUẢN LÝ PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI E S G MÔI TRƯỜNG: Ứng dụng công nghệ số giúp giám sát và tối ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành; đồng thời hỗ trợ cảnh báo sớm khi các chỉ số môi trường vượt ngưỡng và nâng cao tính minh bạch, tuân thủ trong quản lý phát thải. PHƯƠNG PHÁP QUẢN LÝ BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt kh"}\n{"id": "B-A-058bc3df3ba5431d", "claim": "tham gia lộ trình giảm phát thải quốc gia và đang tích cực thực hành các giải pháp phù hợp", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-175a23948789206d", "claim": "Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050.", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-6c5e9630647bf026", "claim": "tiến công nghệ, tối ưu hiệu quả sử dụng tài nguyên, và giảm thiểu phát thải ra môi trường.", "evidence": "bao gồm: Hành động giảm thiểu và thích ứng với biến đổi khí hậu HỆ THỐNG: Đầu tư công nghệ hiện đại Hòa Phát ứng dụng công nghệ tiên tiến để giảm thiểu tối đa các tác động bất lợi đến môi trường. Tập đoàn dành tỷ trọng lớn trong chi phí đầu tư tại các khu liên điện, thân thiện với môi trường. Quy trình sản xuất tại nhà máy cũng được cải tiến để hạn chế khí thải gây hiệu ứng nhà kính, trong đó dây chuyền sơn tự động sử dụng công nghệ xử lý bề mặt nano photphat và bột sơn tĩnh điện, giúp giảm 90% lượng bùn thải ra môi trường."}\n{"id": "B-A-9d71671ccb5bdfd4", "claim": "rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia.", "evidence": "rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng. Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. Trụ cột Quản trị và Đổi "}\n{"id": "B-A-d21096d4756c6db4", "claim": "đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-68d3133ca5239ad3", "claim": "chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 BÁO CÁO PHÁT TRIỂN BỀN VỮNG Làm mát các thiết bị và hoá hơi Sông suối ao hồ Nước sản xuất 2025 PHẦN 5 XANH HÓA SẢN XUẤT Bay hơi Các nhà máy trong KLH Nước tái sử dụng Bùn thải Làm mát trực tiếp Hành động giảm thiểu và thích ứng với biến đổi khí hậu Nước sinh hoạt PW Nước thải sản xuất IWW Nước thải sinh hoạt Quản lý môi trường Nước thải sinh hoạt sau xử lý Bùn thải Tưới cây QUẢN LÝ NƯỚC theo cả khía cạnh tổng lượng nước sử dụng và lượng nước tuần hoàn. Nước được dùng phần lớn cho hoạt động làm mát thiết bị và sản phẩm, thất thoát qua bay hơi và được bù đắp bằng nguồn nước bên ngoài. Dù lượng nước cần thiết cho sản xuất thép là rất cao, các Khu liên hợp Gang Thép của Hò"}\n{"id": "B-A-bb03a6dd1e6f8925", "claim": "áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 Tái chế, tái sử dụng chất thải rắn thành nguyên vật liệu quay lại quá trình sản xuất Các giải pháp tái chế, tái sử dụng nội bộ tập trung vào việc phân loại và xử lý các chất thải ngay tại nguồn phát sinh. Theo đó, các chất thải rắn chứa sắt sẽ được thu gom và xử lý, trở thành nguyên vật liệu đưa ngược lại quy trình sản xuất thép. Các loại bụi phát sinh được giữ lại trong các hệ thống lọc bụi, bụi lò cao, bụi từ quá trình vận chuyển nguyên liệu trước khi vào lò, bụi sàn ra gang, bụi quặng thiêu kết....được thu hồi đưa trở lại quá trình thiêu kết, hoặc tạo ra quặng DRI. Thép phế liệu sinh ra trong các công đoạn đúc can như vảy cán, mê vỉa, đầu mẩu...có tính chất tương tự"}\n{"id": "B-A-f3d8d4186c963983", "claim": "Hòa Phát tích hợp các mục tiêu phát triển bền vững (SDGs) vào chiến lược kinh doanh, phù", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-62fa18b14347c599", "claim": "hợp với định hướng phát triển xanh của Chính phủ Việt Nam và cam kết trung hòa carbon", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 KIỂM KÊ KHÍ NHÀ KÍNH Từ năm 2023, Tập đoàn đã duy trì thực hiện kiểm kê khí nhà kính trên phần lớn các công ty thành viên, tuân thủ các tiêu cuẩn quốc tế ISO 14064 và ISO 14607. Hoạt động kiểm kê KNK được thực hiện với 3 mục tiêu chính: [GRI 302-1, 302-2, 305-1, 305-2, 305-4] Nhận thức rõ Biến đổi Khí hậu là thách thức toàn cầu, và hưởng ứng cam kết Net Zero 2050 của Việt Nam tại COP26, Tập đoàn Hòa Phát xác định “Giảm thiểu và Thích ứng với Biến đổi Khí hậu” là ưu tiên phát triển bền vững quan trọng. Tập đoàn đang triển khai đồng bộ các sáng kiến và hành động, đặc biệt trong lĩnh vực sản xuất thép, nhằm tiết kiệm năng lượng và giảm "}\n{"id": "B-A-b6da112c7e45597c", "claim": "góp tích cực cho sự phát triển thịnh vượng và bền vững của đất nước.", "evidence": "nước tại 26 tỉnh, thành phố trên cả nước. Gắn sự phát triển của mình với lợi ích của cộng đồng, Hòa Phát tích cực thực hiện trách nhiệm xã hội của doanh nghiệp tập Tích hợp phát triển bền vững vào hoạt động sản xuất kinh doanh Thẩm định các sáng kiến về Phát triển bền vững. Giám sát hoạt động Phát triển bền vững đảm bảo hiệu quả và cải tiến"}\n{"id": "B-A-6563dcb5b0d95dce", "claim": "ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 10 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12"}\n{"id": "B-A-d46ff144668fc056", "claim": "| 3.1 ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025", "evidence": "MỤC LỤC BÁO CÁO PHÁT TRIỂN BỀN VỮNG NĂM 2025 TỔNG QUAN VỀ PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT CÁC LĨNH VỰC TRỌNG YẾU VỀ PHÁT TRIỂN BỀN VỮNG 1 5 GIỚI THIỆU BÁO CÁO PHÁT TRIỂN BỀN VỮNG XANH HOÁ SẢN XUẤT 5.1 | Hành động giảm thiểu và thích ứng với biến đổi khí hậu 2 5.2 | Quản lý môi trường THÔNG ĐIỆP CỦA TỔNG GIÁM ĐỐC 3 6 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT PHÁT TRIỂN CON NGƯỜI 6.1 | Sức khoẻ và an toàn lao động 3.1 | Điểm nổi bật về phát triển bền vững năm 2025 6.2 | Quản trị nguồn nhân lực 3.2 | Giải thưởng và vinh danh năm 2025 6.3 | Nhân quyền 3.3 | Thành viên của các Hiệp hội ngành nghề 6.4 | Phát triển cộng đồng 3.4 | Hoạt động kinh doanh tại Tập đoàn 4 7 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀ"}\n{"id": "B-A-2098e29f3fa3ed8c", "claim": "công nghiệp bền vững tại Việt Nam cũng như trên toàn cầu.", "evidence": "thép lớn nhất trên thế giới. Thành tựu này củng cố vị thế của Việt Nam là quốc gia số 1 Đông Nam Á và thứ 11 trên thế giới về năng lực sản xuất thép thô, khẳng định vai trò tiên phong của Hòa Phát trong ngành công nghiệp quốc gia và nâng tầm vị thế ngành luyện kim của Việt Nam trong khu vực và quốc tế. Sự phát triển của Hòa Phát không chỉ được đo lường bằng sản lượng hay quy mô. Với triết lý “Hòa hợp cùng Phát triển”, Hòa Phát xác định rằng tăng trưởng kinh tế phải song hành cùng phát triển bền vững. Tôi rất vinh dự giới thiệu Báo cáo phát triển bền vững năm 2025 của Tập đoàn, như một minh chứng cho nỗ lực thấu hiểu sâu sắc và ghi nhận những mối quan tâm của các bên liên quan. Thông qua đánh"}\n{"id": "B-A-ee28bba03e822e41", "claim": "tới bảo vệ môi trường, nâng cao an toàn lao động và thúc đẩy kinh tế tuần hoàn trong ngành thép.", "evidence": "g nghệ và vận động chính Thúc đẩy kết nối doanh nghiệp, đối thoại chính sách Đóng góp vào việc xây dựng các văn bản pháp luật liên đẩy sử dụng nguyên liệu hợp ESG trong hoạt động sách nhằm nâng cao năng lực cơ khí chế tạo và sản xuất và phát triển kinh tế bền vững; hỗ trợ hội viên thông qua đào tạo, xúc tiến thương quan đến lĩnh vực nông nghiệp, thúc đẩy các hoạt động chuyên ngành về chăn nuôi, thú y, trao đổi kỹ thuật và thúc đẩy công nghệ chăn nuôi, xây dựng chuỗi liên kết tuần hoàn. kinh doanh. trong nước. mại và tư vấn chính sách với các cơ quan quản lý. giá trị bền vững. 11 , tổ chức sản xuất, hợp tác cùng đối tác, chăm lo cho người 6. QUẢN TRỊ VỮNG MẠNH Với nền tảng quản trị minh bạch "}\n{"id": "B-A-2c35bc150ec48a2f", "claim": "tối ưu hệ sinh thái thép Hòa Phân kỳ 1, dự án Khu", "evidence": "CP suất thiết kế 700.000 tấn/năm. Sản phẩm • THÁNG 12: Phân kỳ 1, dự án Khu Thép Hòa Phát Dung ray đường sắt cao tốc đầu tiên dự kiến ra lò vào năm 2027, giúp Hòa Phát trở thành doanh nghiệp đầu tiên và duy nhất ở Đông Phát Hưng Yên, đánh dấu bước phát triển mới tối ưu hệ sinh thái thép Hòa Quất, triển khai Khu Liên nóng/năm, tổng vốn đầu tư dự kiến 85.000 Phát. liên hợp sản xuất gang thép Hòa Phát hợp sản xuất gang thép trong lịch sử Tập đoàn tỷ đồng. Nam Á sản xuất được loại thép này. khi đầu tư vào lĩnh vực Hòa Phát Dung Quất tại THÁNG 12/2020 Tập đoàn Hòa Phát tái cơ Dung Quất 2 hoàn thành lắp đặt, chuẩn bị đưa vào hoạt động thử nghiệm trong quý nông nghiệp. tỉnh Quảng Ngãi, quy THÁNG 8/"}\n{"id": "B-A-32152412a7dd6123", "claim": "được mối quan hệ đối tác bền vững, lâu dài, tin", "evidence": "ÁT Quản trị phát triển bền vững KHÁCH HÀNG /NHÀ PHÂN PHỐI / ĐỐI TÁC KINH DOANH Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát VẤN ĐỀ QUAN TÂM PHẢN HỒI CỦA TẬP ĐOÀN & KẾT QUẢ THỰC HIỆN 2025 PHƯƠNG THỨC TƯƠNG TÁC • Sản phẩm và dịch vụ chất lượng cao, đạt tiêu chuẩn chất lượng • Đổi mới sản phẩm • Danh mục sản phẩm đa dạng về chủng loại và giá cả. • Giao hàng đúng tiến độ • Dịch vụ hậu mãi chuyên nghiệp • Bảo mật thông tin • Xây dựng và duy trì mối quan hệ hợp tác bền vững và lâu dài dựa trên niềm tin tưởng cao với khách hàng, hệ thống nhà phân phối. Nhiều đối tác đã đồng hành cùng Tập đoàn ngay từ những ngày đầu thành lập. • Không ngừng nghiên cứu đổi mới, "}\n{"id": "B-A-4fefd20b62a28212", "claim": "thực hành bền vững ngay từ khâu đầu vào.", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 KIỂM SOÁT CHẤT LƯỢNG KHÔNG KHÍ TRONG CHĂN NUÔI kiện yếm khí để các vi sinh vật phân hủy chất hữu cơ. Quá trình này giúp giảm đáng kể tải lượng ô nhiễm và loại bỏ mùi hôi của chất thải trước khi xả ra môi trường. Quan trọng hơn, biogas tạo ra khí Mê-tan CH4 có thể được sử dụng làm nguồn năng lượng tái tạo để đun nấu hoặc phát điện, giúp giảm chi phí vận hành và đồng thời giảm phát thải khí nhà kính mạnh ra môi trường. Tại các trang trại chăn nuôi heo, chất lượng không khí được kiểm soát bằng kết hợp nhiều biện pháp: BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 KIỂM SOÁT KHẨU PHẦN ĂN CỦA VẬT NUÔI CHUỒNG TRẠI KHÉP KÍN PHẦN 5 XANH HÓA SẢN XUẤT Kiểm soát khẩu phần ăn là một biện pháp c"}\n{"id": "B-A-ea050005b46366c7", "claim": "những giải pháp bền vững, sáng tạo và lâu dài,", "evidence": "PHÁT TRIỂN CỘNG ĐỒNG 6.4 GRI 204, 411, 413] Với triết lý “Hòa hợp cùng phát triển”, Hòa Phát tích hợp mục tiêu phát triển kinh tế với trách nhiệm xã hội, hướng đến tăng trưởng bền vững và tạo giá trị chung cho các bên liên quan. Mọi hoạt động đều mang đến mục tiêu tạo ra giá trị lâu dài cho xã hội, lan tỏa tinh thần sẻ chia và đóng góp thiết thực cho đời sống người dân tại những địa phương nơi Hòa Phát hoạt động. CÁC YẾU TỐ CHÍNH TRONG PHƯƠNG PHÁP TIẾP CẬN CỦA HÒA PHÁT BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 HỖ TRỢ CÓ TRỌNG ĐIỂM: Các chương trình được thiết kế để tập trung giúp đỡ những nhóm đối tượng yếu thế, dễ bị tổn thương trong xã hội để nguồn lực được sử dụng hiệu quả và tạo ra tác động rõ rệ"}\n{"id": "B-A-91e6f329bfcc4a14", "claim": "khí hậu, thúc đẩy kinh tế tuần hoàn và bảo vệ tài", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-3393a2eb2db012e3", "claim": "nguyên chung, Biến chất thải sản xuất thành các", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-d9415c1eaeb5ba61", "claim": "Thép, với đặc tính bền vững, chịu lực cao, và dễ dàng tái chế, là xương sống", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 NHÓM SẢN PHẨM MÁC THÉP ỨNG DỤNG / MỤC ĐÍCH S235JR-2 Ứng dụng dầm, cột, thép hình chữ U/I/V, khung nhà xưởng BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Ứng dụng phổ thông trong dân dụng và công nghiệp như: cán xả băng định hình thép hộp/ống, thép hình (U, I, V) cỡ trung bình, bệ đỡ máy móc, lan can, và các kết cấu chịu tải tĩnh. S275JR-2 THÉP DÙNG CHO PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Chế tạo các kết cấu chịu tải trọng nặng và động lực học cao như: dầm chịu lực cầu đường, khung gầm máy xúc/xe tải, tháp điện gió, kết cấu nhà xưởng khẩu độ lớn. KẾT CẤU S355JR-2 (STRUCTURAL STEEL) S355J2Nb-VD Dùng cho tháp gió, thiết bị khai mỏ. SAE1300M Ứng dụng cho các kết cấu chịu lực, chịu mài mòn tron"}\n{"id": "B-A-fceeec843d89afc3", "claim": "đổi năng lượng tái tạo (tua-bin gió, điện mặt trời), điện khí hóa, giao thông", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-452d416d2d9287f9", "claim": "Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không", "evidence": "óa thạch. • Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu. Khi các sản phẩm từ thép như vỏ ô tô, khung nhà hay thiết bị điện tử...hết thời hạn sử dụng, Giải thưởng và vinh danh năm 2025 thay vì trở thành rác thải, chúng được thu gom, phân tách và đưa vào lò điện để nấu Sản xuất thép chảy lại thành thép lỏng, tạo ra các sản phẩm thép mới.. Giai đoạn này gắn với công "}\n{"id": "B-A-f51583885c527995", "claim": "làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế,", "evidence": "vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi "}\n{"id": "B-A-eed4d4c96c255b05", "claim": "điều chỉnh hàm lượng carbon để trở thành thép thành phẩm.", "evidence": "óa thạch. • Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu. Khi các sản phẩm từ thép như vỏ ô tô, khung nhà hay thiết bị điện tử...hết thời hạn sử dụng, Giải thưởng và vinh danh năm 2025 thay vì trở thành rác thải, chúng được thu gom, phân tách và đưa vào lò điện để nấu Sản xuất thép chảy lại thành thép lỏng, tạo ra các sản phẩm thép mới.. Giai đoạn này gắn với công "}\n{"id": "B-A-2e5d6baf78ac4e21", "claim": "CO2 do quá trình đốt cháy nhiên liệu hóa thạch.", "evidence": "trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. Từ tháng 9 năm 2025, Hòa Phát Dung Quất 2 đi vào vận hành, Tổng sản lượng thép thô tăng 24%, tương ứng với mức tăng phát thải của toàn tập đoàn. Lĩnh vực nông nghiệp và chăn nuôi của Tập đoàn Hòa Phát ghi nhận mức phát 40 coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) v"}\n{"id": "B-A-e55f3737a07d4698", "claim": "nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP • Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - Từ quặng đến thép Vòng đời sơ cấp của thép bắt đầu từ việc khai thác các nguồn tài nguyên thiên nhiên như quặng sắt, than đá và đá vôi để khởi động quy trình sản xuất vật liệu mới. Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF). Trong các lò cao làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế, BÁO CÁO PHÁT TRIỂN BỀN VỮNG công nghiệp, quặng sắt được hoàn nguyên và nấu chảy ở nhiệt độ cực cao để tạo ra 2025 tái sản xuất. Vòng đ"}\n{"id": "B-A-b4b18f49ef4bdfcc", "claim": "một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu.", "evidence": "nhà xưởng. Năm 2025, tổng công suất đã lắp đặt đạt gần 20MWp, phần còn lại sẽ được tiếp tục triển khai mạnh mẽ trong năm 2026 và 2027. Sau khi đi vào vận hành ổn định, hệ thống dự kiến sẽ cung cấp khoảng 170 triệu kWh/năm, giúp đáp ứng 13% lượng điện mua ngoài hiện tại của Tập đoàn. Điểm ưu việt của mô hình này nằm ở việc năng lượng được sản xuất và tiêu thụ tại chỗ, loại bỏ hoàn toàn hao hụt trong quá trình truyền tải xa và tạo ra một lớp “lá chắn nhiệt” tự nhiên giúp hạ nhiệt độ mái tôn đáng kể, từ đó giảm điện năng tiêu thụ cho các thiết bị làm mát bên dưới. Việc tối ưu hóa diện tích mái xưởng sẵn có không chỉ giúp Hòa Phát chủ động nguồn điện, giảm tải cho lưới điện quốc gia mà còn là mi"}\n{"id": "B-A-96b56816b7a520f7", "claim": "tận, biến thép trở thành một trong những vật liệu cốt lõi của nền kinh tế tuần hoàn và", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 TƯƠNG LAI NGÀNH THÉP CỦA CHÚNG TÔI Tập đoàn Hòa Phát là tập đoàn đa ngành hoạt động trong nhiều lĩnh vực bao gồm Thép, Nông nghiệp, Bất động sản, Điện máy gia dụng. Trong đó, ngành Thép giữ vai trò cốt lõi và là trụ cột chính trong mô hình kinh doanh. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT VAI TRÒ CỦA THÉP Vai trò của thép trong đời sống hiện đại là vô cùng thiết yếu và không thể thay thế. Thép, với đặc tính bền vững, chịu lực cao, và dễ dàng tái chế, là xương sống của mọi nền kinh tế và kiến tạo nên môi trường sống của chúng ta. Thép không chỉ là vật liệu mà còn là nền tảng vật chất định hình nên mọi mặt của xã h"}\n{"id": "B-A-f578a6ecf675d379", "claim": "so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP • Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - Từ quặng đến thép Vòng đời sơ cấp của thép bắt đầu từ việc khai thác các nguồn tài nguyên thiên nhiên như quặng sắt, than đá và đá vôi để khởi động quy trình sản xuất vật liệu mới. Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF). Trong các lò cao làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế, BÁO CÁO PHÁT TRIỂN BỀN VỮNG công nghiệp, quặng sắt được hoàn nguyên và nấu chảy ở nhiệt độ cực cao để tạo ra 2025 tái sản xuất. Vòng đ"}\n{"id": "B-A-2bbce2b7cdac1ba4", "claim": "Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi", "evidence": "arbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai đoạn nào của vòng đời. Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu thành thép để tạo vòng đời tiếp theo. Việc lựa chọn sản xuất thép theo công nghệ nào Sản xuất, chế tạo phụ thuộc đáng kể vào khả năng tiếp cận quặng sắt, thép phế liệu của mỗi quốc gia, khu vực. Hiện nay, trên thế giới, 70% thép được sản xuất ra từ lò cao và chỉ 30% thép được sản Sử dụng xuất trong lò điện. 20 HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP • Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - "}\n{"id": "B-A-018a18f6adf10d9d", "claim": "khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP • Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - Từ quặng đến thép Vòng đời sơ cấp của thép bắt đầu từ việc khai thác các nguồn tài nguyên thiên nhiên như quặng sắt, than đá và đá vôi để khởi động quy trình sản xuất vật liệu mới. Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF). Trong các lò cao làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế, BÁO CÁO PHÁT TRIỂN BỀN VỮNG công nghiệp, quặng sắt được hoàn nguyên và nấu chảy ở nhiệt độ cực cao để tạo ra 2025 tái sản xuất. Vòng đ"}\n{"id": "B-A-f210fc1ed9c85933", "claim": "cận hơn, Hòa Phát định hướng đẩy mạnh hơn nữa tỷ trọng thép tái chế trong cơ cấu sản", "evidence": "lý môi trường TĂNG SỬ DỤNG THÉP PHẾ LIỆU Tập đoàn đang tăng cường tỷ lệ sử dụng thép phế và HBI trong lò thổi. Biện pháp này giúp giảm mạnh nhu cầu hoàn nguyên quặng, giảm tiêu hao than cốc, tận dụng nhiệt lượng và giảm phát thải khí nhà kính trong toàn bộ chuỗi sản xuất, đồng thời gia tăng năng lực sản xuất thép. Đến năm 2030, các công ty thép của tập đoàn Hòa Phát dự kiến tăng sử dụng thép phế đạt 30%. ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Tập đoàn lập kế hoạch triển khai các dự án đề xuất tăng cường phối phế liệu cùng gang lỏng trong lò thổi để tăng tỷ trọng nguyên vật liệu phát thải thấp. Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọn"}\n{"id": "B-A-480195377b360d1e", "claim": "Định hướng chiến lược về Phát triển bền vững.", "evidence": "PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 CHUYỂN ĐỔI SỐ TRONG SẢN XUẤT VÀ QUẢN TRỊ DOANH NGHIỆP PHƯƠNG PHÁP QUẢN LÝ PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI E S G MÔI TRƯỜNG: Ứng dụng công nghệ số giúp giám sát và tối ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành; đồng thời hỗ trợ cảnh báo sớm khi các chỉ số môi trường vượt ngưỡng và nâng cao tính minh bạch, tuân thủ trong quản lý phát thải. PHƯƠNG PHÁP QUẢN LÝ BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt kh"}\n{"id": "B-A-31cba8d7afbedc19", "claim": "Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm", "evidence": "a đó giảm phát sinh chất thải. • Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng các hệ thống tuần hoàn khép kín, tái chế nước và xử lý triệt để để tuân thủ quy chuẩn xả thải. • Giám sát thường xuyên khí thải để bảo đảm mọi thông số nằm trong giới hạn cho phép. QUẢN LÝ MÔI TRƯỜNG Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan Hòa Phát cam kết đảm bảo một môi trường làm việc an toàn cho toàn thể người lao động. Thông qua quản lý rủi ro chặt chẽ, đào tạo liên tục và đầu tư vào công nghệ an toàn, Tập đoàn hướng tới xây dựng nơi làm việc là nơi mọi nhân viên đều có thể trở về nhà an toàn mỗi ngày. S"}\n{"id": "B-A-78f1b90f54443081", "claim": "cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 NHÓM SẢN PHẨM MÁC THÉP ỨNG DỤNG / MỤC ĐÍCH S235JR-2 Ứng dụng dầm, cột, thép hình chữ U/I/V, khung nhà xưởng BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Ứng dụng phổ thông trong dân dụng và công nghiệp như: cán xả băng định hình thép hộp/ống, thép hình (U, I, V) cỡ trung bình, bệ đỡ máy móc, lan can, và các kết cấu chịu tải tĩnh. S275JR-2 THÉP DÙNG CHO PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Chế tạo các kết cấu chịu tải trọng nặng và động lực học cao như: dầm chịu lực cầu đường, khung gầm máy xúc/xe tải, tháp điện gió, kết cấu nhà xưởng khẩu độ lớn. KẾT CẤU S355JR-2 (STRUCTURAL STEEL) S355J2Nb-VD Dùng cho tháp gió, thiết bị khai mỏ. SAE1300M Ứng dụng cho các kết cấu chịu lực, chịu mài mòn tron"}\n{"id": "B-A-e30192e196002d48", "claim": "nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao", "evidence": "HÒA PHÁT Các yếu tố bền vững được tích hợp vào quá trình thảo luận và xây dựng chiến lược, ra quyết định đầu tư và giám sát quản lý rủi ro của tập đoàn và các công ty thành viên. Ban ESG chịu trách nhiệm giám sát các vấn đề liên quan đến ESG, bao gồm cả rủi ro ESG. Ban ESG được thành lập với sự tham gia của Ban Tổng Giám đốc, Ban Giám đốc các công ty thành viên cùng các ban chức năng như ban Tài chính, Pháp chế, Nhân sự, Truyền thông của Tập đoàn nhằm điều phối các hoạt động ESG và tích hợp các mục tiêu phát triển bền vững vào hoạt động vận hành hàng ngày của doanh nghiệp. CÁC BAN CHỨC NĂNG CỦA TẬP ĐOÀN: ĐIỀU PHỐI • Đầu mối tổng hợp báo cáo Phát triển bền vững • Điều phối chung các hoạt động"}\n{"id": "B-A-456e2eab748de30d", "claim": "gồm các vấn đề về môi trường, xã hội và quản trị (ESG).", "evidence": "NHẬN DIỆN CÁC VẤN ĐỀ TRỌNG YẾU 4.2 BƯỚC 2 [GRI 3-1, 3-2, 3-3, 2-2, 2-23] Năm 2025, Tập đoàn Hòa Phát đã thực hiện Đánh giá tính trọng yếu theo Tiêu chuẩn GRI 2021. Quá trình này nhằm xác định và ưu tiên các vấn đề về môi trường, xã hội và quản trị (ESG) có tác động lớn nhất đến hoạt động kinh doanh, các bên liên quan cũng như bối cảnh kinh tế - xã hội và môi trường rộng lớn hơn. Mỗi chủ đề trọng yếu được đánh giá dựa trên mức độ quan trọng đối với các bên liên quan và mức độ tác động đến doanh nghiệp. THAM VẤN CÁC BÊN LIÊN QUAN: Hòa Phát đã xây dựng bản đồ các bên liên quan, bao gồm cả nội bộ và bên ngoài, tập trung vào những đối tượng có vai trò quan trọng BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 t"}\n{"id": "B-A-8f6f43d06082b8d9", "claim": "Tích hợp phát triển bền vững vào hoạt động sản xuất kinh doanh", "evidence": "HÒA PHÁT Các yếu tố bền vững được tích hợp vào quá trình thảo luận và xây dựng chiến lược, ra quyết định đầu tư và giám sát quản lý rủi ro của tập đoàn và các công ty thành viên. Ban ESG chịu trách nhiệm giám sát các vấn đề liên quan đến ESG, bao gồm cả rủi ro ESG. Ban ESG được thành lập với sự tham gia của Ban Tổng Giám đốc, Ban Giám đốc các công ty thành viên cùng các ban chức năng như ban Tài chính, Pháp chế, Nhân sự, Truyền thông của Tập đoàn nhằm điều phối các hoạt động ESG và tích hợp các mục tiêu phát triển bền vững vào hoạt động vận hành hàng ngày của doanh nghiệp. CÁC BAN CHỨC NĂNG CỦA TẬP ĐOÀN: ĐIỀU PHỐI • Đầu mối tổng hợp báo cáo Phát triển bền vững • Điều phối chung các hoạt động"}\n{"id": "B-A-fe07ba802f9025d2", "claim": "Thẩm định các sáng kiến về Phát triển bền vững.", "evidence": "m, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023. Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn. Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám"}\n{"id": "B-A-cc0a1ea40f56b071", "claim": "Giám sát hoạt động Phát triển bền vững đảm bảo hiệu quả và cải tiến", "evidence": "đại. Năm 2025, chuyên trang ghi nhận sự quan tâm tích cực từ CBCNV so với 2024, góp phần lan tỏa thông tin, nâng cao nhận thức và thúc đẩy tinh thần đổi mới trong toàn Tập đoàn. Phát triển đổi mới sáng tạo • Hệ thống camera giám sát tập trung: Hệ thống giám sát sản xuất tập trung hơn 1000 camera được triển khai trên toàn Khu liên hợp, phục vụ đồng thời cho giám sát vận hành và đảm bảo an ninh, an toàn. Giải pháp giúp chuyển đổi từ mô hình giám sát thủ công tại chỗ sang giám sát tập trung tại phòng điều khiển, góp phần nâng cao hiệu quả kiểm soát vận hành, đồng thời tối ưu nguồn lực nhân sự và chi phí quản lý tại các nhà máy. Hoạt động Chuyển đổi số của Tập đoàn đang được triển khai đồng bộ, "}\n{"id": "B-A-e8b8d88fd6e7946e", "claim": "Đảm bảo việc thực hiện các báo cáo liên quan Phát triển bền vững đầy", "evidence": "quan hệ hợp tác và xác định các mục tiêu phát triển bền vững phù hợp trong từng giai đoạn. Quản trị phát triển bền vững XÁC ĐỊNH CÁC BÊN LIÊN QUAN Nhận diện các vấn đề trọng yếu Dựa trên kinh nghiệm thực tiễn của các công ty thành viên,Tập đoàn Hòa Phát đã xác định các bên liên quan chính thông qua việc phân tích các đặc điểm như: mức độ phụ thuộc vào hoạt động của doanh nghiệp, trách nhiệm liên quan, mức độ ảnh hưởng đến quyết định chiến lược, sự đa dạng trong quan điểm và tiềm năng phát sinh xung đột trong quá trình tương tác. Việc xác định này tuân thủ theo Tiêu chuẩn AA1000 về Tham vấn các bên liên quan (AA1000SES) – Khung hướng dẫn quốc tế nhằm đảm bảo quá trình tham vấn được thực hiện "}\n{"id": "B-A-c4ff36919f2e3192", "claim": "Các yếu tố bền vững được tích hợp vào quá trình thảo luận và xây dựng chiến", "evidence": "HÒA PHÁT Các yếu tố bền vững được tích hợp vào quá trình thảo luận và xây dựng chiến lược, ra quyết định đầu tư và giám sát quản lý rủi ro của tập đoàn và các công ty thành viên. Ban ESG chịu trách nhiệm giám sát các vấn đề liên quan đến ESG, bao gồm cả rủi ro ESG. Ban ESG được thành lập với sự tham gia của Ban Tổng Giám đốc, Ban Giám đốc các công ty thành viên cùng các ban chức năng như ban Tài chính, Pháp chế, Nhân sự, Truyền thông của Tập đoàn nhằm điều phối các hoạt động ESG và tích hợp các mục tiêu phát triển bền vững vào hoạt động vận hành hàng ngày của doanh nghiệp. CÁC BAN CHỨC NĂNG CỦA TẬP ĐOÀN: ĐIỀU PHỐI • Đầu mối tổng hợp báo cáo Phát triển bền vững • Điều phối chung các hoạt động"}\n{"id": "B-A-55b9de33865b9cac", "claim": "Ban ESG chịu trách nhiệm giám sát các vấn đề liên quan đến ESG, bao", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-fcc7ab0a30a9e3df", "claim": "Ban ESG được thành lập với sự tham gia của Ban Tổng Giám đốc, Ban Giám đốc", "evidence": "• Tham mưu, đề xuất, triển khai thực hiện các công việc liên quan đến quản lý năng lượng và khí nhà kính. TẠI CẤP TẬP ĐOÀN TẠI CẤP CÔNG TY CON BAN Ban Giám đốc Công ty thành viên Ban chức năng ESG Phòng chuyên trách Ban Quản lý Năng lượng Công ty thành viên & Khí nhà kính 23 thông tin này được truyền đạt đến Ban Giám đốc từng công ty thành viên để xem xét và phản hồi phù hợp. Các phòng, ban của Tập đoàn Hòa Phát cũng hỗ trợ và phối hợp với Ban Giám đốc công ty thành viên để phản hồi các bên liên quan trong phạm vi chuyên môn của phòng ban đó. Quản trị Doanh nghiệp và Quản lý rủi ro Phối hợp với Bộ phận Kiểm toán nội bộ, đưa ra các khuyến TƯ VẤN nghị độc lập nhằm tăng cường hiệu quả quản trị."}\n{"id": "B-A-53b16e8e392e03c6", "claim": "sự, Truyền thông của Tập đoàn nhằm điều phối các hoạt động ESG và tích hợp", "evidence": "thông tin này được truyền đạt đến Ban Giám đốc từng công ty thành viên để xem xét và phản hồi phù hợp. Các phòng, ban của Tập đoàn Hòa Phát cũng hỗ trợ và phối hợp với Ban Giám đốc công ty thành viên để phản hồi các bên liên quan trong phạm vi chuyên môn của phòng ban đó. Quản trị Doanh nghiệp và Quản lý rủi ro Phối hợp với Bộ phận Kiểm toán nội bộ, đưa ra các khuyến TƯ VẤN nghị độc lập nhằm tăng cường hiệu quả quản trị. Quản lý chất lượng Thông qua các đánh giá và khuyến nghị độc lập, Ban Kiểm soát góp phần tăng cường hệ thống kiểm soát nội bộ và hỗ trợ sự phát triển bền vững của Tập đoàn. Trong thời gian tới, nhằm nâng cao hiệu quả giám sát, Ban Kiểm soát đề xuất Ban Phát triển đổi mới sán"}\n{"id": "B-A-2d0f64b2c468490e", "claim": "các mục tiêu phát triển bền vững vào hoạt động vận hành hàng ngày của doanh", "evidence": "1 GIỚI THIỆU BÁO CÁO PHÁT TRIỂN BỀN VỮNG KHUNG BÁO CÁO [GRI 1] Báo cáo Phát triển Bền vững năm 2025 của Tập đoàn Hòa Phát được xây dựng phù hợp với Bộ Tiêu chuẩn Báo cáo Phát triển Bền vững 2021 của Sáng kiến Báo cáo Toàn cầu (GRI), áp dụng cho giai đoạn từ ngày 01/01/2025 đến ngày 31/12/2025, với các thông tin công bố được tham chiếu trong bảng chỉ mục nội dung GRI. Báo cáo cũng phản ánh các tiêu chuẩn của Hội đồng Chuẩn mực Kế toán Bền vững (SASB) dành cho các nhà sản xuất sắt và thép. Báo cáo này được công bố định kỳ hằng năm. PHẠM VI VÀ GIỚI HẠN CỦA BÁO CÁO [GRI 2-2, 2-3] Hiệu quả phát triển bền vững của Tập đoàn Hòa Phát được trình bày trong báo cáo này bao quát toàn bộ hoạt động của Tậ"}\n{"id": "B-A-16071b02cb5b6c62", "claim": "Đầu mối tổng hợp báo cáo Phát triển bền vững", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-1e60cd4ae77cb6ec", "claim": "Điều phối chung các hoạt động Phát triển bền vững", "evidence": "PHÁT TRIỂN CỘNG ĐỒNG 6.4 GRI 204, 411, 413] Với triết lý “Hòa hợp cùng phát triển”, Hòa Phát tích hợp mục tiêu phát triển kinh tế với trách nhiệm xã hội, hướng đến tăng trưởng bền vững và tạo giá trị chung cho các bên liên quan. Mọi hoạt động đều mang đến mục tiêu tạo ra giá trị lâu dài cho xã hội, lan tỏa tinh thần sẻ chia và đóng góp thiết thực cho đời sống người dân tại những địa phương nơi Hòa Phát hoạt động. CÁC YẾU TỐ CHÍNH TRONG PHƯƠNG PHÁP TIẾP CẬN CỦA HÒA PHÁT BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 HỖ TRỢ CÓ TRỌNG ĐIỂM: Các chương trình được thiết kế để tập trung giúp đỡ những nhóm đối tượng yếu thế, dễ bị tổn thương trong xã hội để nguồn lực được sử dụng hiệu quả và tạo ra tác động rõ rệ"}\n{"id": "B-A-6d43afbc929674df", "claim": "Nghiên cứu và triển khai thực hành sáng kiến bền vững trong sản xuất,", "evidence": "rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng. Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. Trụ cột Quản trị và Đổi "}\n{"id": "B-A-58076665e99a9ee5", "claim": "Báo cáo thông tin liên quan Phát triển bền vững đầy đủ, chính xác,", "evidence": "GẮN KẾT VỚI CÁC BÊN LIÊN QUAN 4.4 CỔ ĐÔNG/ NHÀ ĐẦU TƯ VẤN ĐỀ QUAN TÂM PHẢN HỒI CỦA TẬP ĐOÀN & KẾT QUẢ THỰC HIỆN 2025 PHƯƠNG THỨC TƯƠNG TÁC BÁO CÁO PHÁT TRIỂN BỀN VỮNG • Tăng trưởng bền vững và hiệu quả lợi nhuận • Quản trị doanh nghiệp minh bạch • Quản trị rủi ro hiệu quả • Công bố thông tin chính xác, kịp thời • Là một công ty đại chúng, đảm bảo hiệu quả kinh doanh và quyền lợi hợp pháp của cổ đông/nhà đầu tư là mục tiêu được ưu tiên hàng đầu của Tập đoàn. • Duy trì nguyên tắc minh bạch và công bằng trong công bố thông tin, đảm bảo tuân thủ đầy đủ, chính xác và kịp thời theo quy định pháp luật. Tất cả thông tin liên quan đến sản xuất, kinh doanh, dự án trọng điểm và hoạt động quản trị doanh"}\n{"id": "B-A-93a3125ec5e88f4d", "claim": "Khung quản trị bền vững của Tập đoàn được minh họa như sau:", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-bec5cfc6e6a4e2de", "claim": "và quản trị (ESG) có tác động lớn nhất đến hoạt động kinh doanh, các bên liên quan", "evidence": "i ro – Tất cả các cấp quản lý và nhân viên đều đóng vai trò là chủ sở hữu rủi ro, có trách nhiệm xác định, đánh giá và đề xuất biện pháp ứng phó trong phạm vi công việc của mình. Các phòng ban định kỳ rà soát rủi ro tiềm ẩn, đánh giá theo hai yếu tố chính: - Khả năng xảy ra, và - Mức độ ảnh hưởng đến hoạt động kinh doanh cũng như các bên liên quan. Thông qua quá trình này, các đơn vị có thể ưu tiên nguồn lực để xử lý những rủi ro có tác động lớn nhất, từ đó bảo vệ hiệu quả hoạt động và uy tín của Tập đoàn. Những rủi ro được xếp loại “cao” hoặc “rất cao” liên quan đến chiến lược và kế hoạch kinh doanh sẽ được báo cáo lên Ban Tổng Giám đốc và Hội đồng Quản trị để xem xét và xử lý. Đối với các "}\n{"id": "B-A-15a8afdf265f638b", "claim": "đề ESG đối với hoạt động kinh doanh, tài chính và danh tiếng của Tập đoàn.", "evidence": "GẮN KẾT VỚI CÁC BÊN LIÊN QUAN 4.4 CỔ ĐÔNG/ NHÀ ĐẦU TƯ VẤN ĐỀ QUAN TÂM PHẢN HỒI CỦA TẬP ĐOÀN & KẾT QUẢ THỰC HIỆN 2025 PHƯƠNG THỨC TƯƠNG TÁC BÁO CÁO PHÁT TRIỂN BỀN VỮNG • Tăng trưởng bền vững và hiệu quả lợi nhuận • Quản trị doanh nghiệp minh bạch • Quản trị rủi ro hiệu quả • Công bố thông tin chính xác, kịp thời • Là một công ty đại chúng, đảm bảo hiệu quả kinh doanh và quyền lợi hợp pháp của cổ đông/nhà đầu tư là mục tiêu được ưu tiên hàng đầu của Tập đoàn. • Duy trì nguyên tắc minh bạch và công bằng trong công bố thông tin, đảm bảo tuân thủ đầy đủ, chính xác và kịp thời theo quy định pháp luật. Tất cả thông tin liên quan đến sản xuất, kinh doanh, dự án trọng điểm và hoạt động quản trị doanh"}\n{"id": "B-A-5f4062c2b9f1e1d7", "claim": "này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-c0f49d36292c407e", "claim": "Để thu thập ý kiến về mức độ ưu tiên của các chủ đề ESG trọng yếu,", "evidence": "Ổ CHỨC VÀ CÁC CÔNG TY CÙNG NGÀNH: để xác định mức độ ưu tiên. Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG Gắn kết với các bên liên quan BƯỚC 4 tại 15 doanh nghiệp lớn cùng ngành trên thế giới, các mối quan tâm về phát triển bền vững đã được xác định. Quá trình này tham chiếu 05 bộ khung phát triển bền vững quốc ƯU TIÊN CÁC CHỦ ĐỀ TRỌNG YẾU: 12 chủ đề được phân loại thành 3 nhóm: Chủ đề tế hàng đầu, trong đó có Bộ Tiêu chuẩn GRI Chung (phiên bản cập nhật), Đánh giá Phát ưu tiên (tác động cao), Chủ đề trọng tâm (tác động trung bình), và Chủ đề nền tảng (tác triển Bền vững Doanh nghiệp của S&P Global CSA, Hội đồng Chuẩn mực Kế toán Bền động thấp). Các chủ đề"}\n{"id": "B-A-a608205db7cad22a", "claim": "thiện trong quá trình phát triển bền vững.", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-78827e2ae8928910", "claim": "trọng của các chủ đề ESG đối với hoạt động kinh doanh, trong khi phản hồi từ các bên", "evidence": "NHẬN DIỆN CÁC VẤN ĐỀ TRỌNG YẾU 4.2 BƯỚC 2 [GRI 3-1, 3-2, 3-3, 2-2, 2-23] Năm 2025, Tập đoàn Hòa Phát đã thực hiện Đánh giá tính trọng yếu theo Tiêu chuẩn GRI 2021. Quá trình này nhằm xác định và ưu tiên các vấn đề về môi trường, xã hội và quản trị (ESG) có tác động lớn nhất đến hoạt động kinh doanh, các bên liên quan cũng như bối cảnh kinh tế - xã hội và môi trường rộng lớn hơn. Mỗi chủ đề trọng yếu được đánh giá dựa trên mức độ quan trọng đối với các bên liên quan và mức độ tác động đến doanh nghiệp. THAM VẤN CÁC BÊN LIÊN QUAN: Hòa Phát đã xây dựng bản đồ các bên liên quan, bao gồm cả nội bộ và bên ngoài, tập trung vào những đối tượng có vai trò quan trọng BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 t"}\n{"id": "B-A-276d005fb15edbc0", "claim": "Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG", "evidence": "Ổ CHỨC VÀ CÁC CÔNG TY CÙNG NGÀNH: để xác định mức độ ưu tiên. Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG Gắn kết với các bên liên quan BƯỚC 4 tại 15 doanh nghiệp lớn cùng ngành trên thế giới, các mối quan tâm về phát triển bền vững đã được xác định. Quá trình này tham chiếu 05 bộ khung phát triển bền vững quốc ƯU TIÊN CÁC CHỦ ĐỀ TRỌNG YẾU: 12 chủ đề được phân loại thành 3 nhóm: Chủ đề tế hàng đầu, trong đó có Bộ Tiêu chuẩn GRI Chung (phiên bản cập nhật), Đánh giá Phát ưu tiên (tác động cao), Chủ đề trọng tâm (tác động trung bình), và Chủ đề nền tảng (tác triển Bền vững Doanh nghiệp của S&P Global CSA, Hội đồng Chuẩn mực Kế toán Bền động thấp). Các chủ đề"}\n{"id": "B-A-89b3083b98d0035e", "claim": "Quá trình này tham chiếu 05 bộ khung phát triển bền vững quốc", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-b40ec62dab02fefc", "claim": "triển Bền vững Doanh nghiệp của S&P Global CSA, Hội đồng Chuẩn mực Kế toán Bền", "evidence": "1 GIỚI THIỆU BÁO CÁO PHÁT TRIỂN BỀN VỮNG KHUNG BÁO CÁO [GRI 1] Báo cáo Phát triển Bền vững năm 2025 của Tập đoàn Hòa Phát được xây dựng phù hợp với Bộ Tiêu chuẩn Báo cáo Phát triển Bền vững 2021 của Sáng kiến Báo cáo Toàn cầu (GRI), áp dụng cho giai đoạn từ ngày 01/01/2025 đến ngày 31/12/2025, với các thông tin công bố được tham chiếu trong bảng chỉ mục nội dung GRI. Báo cáo cũng phản ánh các tiêu chuẩn của Hội đồng Chuẩn mực Kế toán Bền vững (SASB) dành cho các nhà sản xuất sắt và thép. Báo cáo này được công bố định kỳ hằng năm. PHẠM VI VÀ GIỚI HẠN CỦA BÁO CÁO [GRI 2-2, 2-3] Hiệu quả phát triển bền vững của Tập đoàn Hòa Phát được trình bày trong báo cáo này bao quát toàn bộ hoạt động của Tậ"}\n{"id": "B-A-aedcf3c2c5b808c3", "claim": "quản lý phù hợp trong chiến lược phát triển bền vững của Hòa Phát.", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 [GRI 2-12, 2-24, 3-1, 3-2, 3-3] TRỞ THÀNH TẬP ĐOÀN SẢN XUẤT CÔNG NGHIỆP VỚI CHẤT LƯỢNG DẪN ĐẦU, TRONG ĐÓ THÉP LÀ LĨNH VỰC CỐT LÕI Tầm nhìn BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 CUNG CẤP SẢN PHẨM DẪN ĐẦU, GÓP PHẦN NÂNG CAO CHẤT LƯỢNG CUỘC SỐNG, Sứ mệnh PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT ĐẠT ĐƯỢC SỰ TIN YÊU CỦA KHÁCH HÀNG Trụ cột XANH HÓA SẢN XUẤT PHÁT TRIỂN CON NGƯỜI QUẢN TRỊ VÀ ĐỔI MỚI Chuyển đổi quy trình sản xuất thép theo hướng ít carbon, sử dụng tài nguyên hiệu quả và vận hành Xây dựng môi trường làm việc an toàn, Tăng cường quản trị minh bạch, Mục tiêu công bằng và thúc đẩy sự phát triển tích hợp ESG vào chiến lược, và th"}\n{"id": "B-A-327d665038ca50c3", "claim": "Nhóm nghiên cứu đã sắp xếp các mối quan tâm ESG lại thành 12 chủ đề liên quan", "evidence": "của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan kết hợp với đóng góp mà Hòa Phát mang lại cho xã hội và môi trường. Cách tiếp cận này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then chốt nhất. cấp và nhà đầu tư. Để thu thập ý kiến về mức độ ưu tiên của các chủ đề ESG trọng yếu, Hòa Phát đã tiếp cận các bên liên quan thông qua phỏng vấn và khảo sát, đồng thời ghi nhận các đóng góp liên quan đến điểm mạnh, thách thức và lĩnh vực Hòa Phát cần cải thiện trong quá trình phát triển bền vững. BƯỚC 3 PHÂN TÍCH PHẢN HỒI: Kết quả từ các cuộc phỏng vấn và khảo sát đã được tổng hợp, QUY TRÌNH XÁC ĐỊNH CÁC CHỦ ĐỀ TRỌNG YẾU ĐƯỢC "}\n{"id": "B-A-9be063f9bb773e6c", "claim": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 25 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 4 PHẦN PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan 23"}\n{"id": "B-A-262cc27886bf9529", "claim": "ít carbon, sử dụng tài nguyên hiệu quả và vận hành", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-e825833b088d7de5", "claim": "bền vững phát sinh trong hoạt động sản xuất kinh doanh của Tập đoàn, đặc biệt", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 TẦM NHÌN Trở thành Tập đoàn sản xuất công nghiệp với chất lượng dẫn đầu, trong đó Thép là lĩnh vực cốt lõi. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT SỨ MỆNH Cung cấp sản phẩm dẫn đầu, góp phần nâng cao chất lượng cuộc sống, đạt được sự tin yêu của khách hàng. Điểm nổi bật về phát triển bền vững năm 2025 GIÁ TRỊ CỐT LÕI Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng phát triển. Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, đảm bảo hài hòa lợi ích của các bên liên quan trên cùng một con thuyền, hướng tới sự phát triển "}\n{"id": "B-A-f985bf05800a2756", "claim": "duy trì năng lực cạnh tranh bền vững trong quá trình chuyển đổi hướng tới trung", "evidence": "PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 CHUYỂN ĐỔI SỐ TRONG SẢN XUẤT VÀ QUẢN TRỊ DOANH NGHIỆP PHƯƠNG PHÁP QUẢN LÝ PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI E S G MÔI TRƯỜNG: Ứng dụng công nghệ số giúp giám sát và tối ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành; đồng thời hỗ trợ cảnh báo sớm khi các chỉ số môi trường vượt ngưỡng và nâng cao tính minh bạch, tuân thủ trong quản lý phát thải. PHƯƠNG PHÁP QUẢN LÝ BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt kh"}\n{"id": "B-A-35328472ce442ae6", "claim": "hòa carbon, đồng thời tiếp tục khẳng định cam kết về chất lượng sản phẩm dẫn", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-4725922b6ad98dcc", "claim": "tích hợp phản hồi ESG từ các đối tác, coi đây là động lực quan trọng để củng cố", "evidence": "của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan kết hợp với đóng góp mà Hòa Phát mang lại cho xã hội và môi trường. Cách tiếp cận này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then chốt nhất. cấp và nhà đầu tư. Để thu thập ý kiến về mức độ ưu tiên của các chủ đề ESG trọng yếu, Hòa Phát đã tiếp cận các bên liên quan thông qua phỏng vấn và khảo sát, đồng thời ghi nhận các đóng góp liên quan đến điểm mạnh, thách thức và lĩnh vực Hòa Phát cần cải thiện trong quá trình phát triển bền vững. BƯỚC 3 PHÂN TÍCH PHẢN HỒI: Kết quả từ các cuộc phỏng vấn và khảo sát đã được tổng hợp, QUY TRÌNH XÁC ĐỊNH CÁC CHỦ ĐỀ TRỌNG YẾU ĐƯỢC "}\n{"id": "B-A-9440d7b4cc6663bf", "claim": "Trong năm 2025, Hòa Phát đã xây dựng Chiến lược Phát triển Bền vững để định", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-973f877f87be36a9", "claim": "đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao", "evidence": ", tổ chức sản xuất, hợp tác cùng đối tác, chăm lo cho người 6. QUẢN TRỊ VỮNG MẠNH Với nền tảng quản trị minh bạch và trách nhiệm, Hòa Phát duy trì cơ chế giám sát và giải quyết khiếu nại hiệu quả, củng cố niềm tin của các bên liên quan. Giải thưởng và vinh danh năm 2025 lao động và đồng hành cùng 3 5 cộng đồng. 3. HIỆU QUẢ VÀ QUẢN LÝ TÀI NGUYÊN, THỰC HÀNH TIẾT KIỆM Chúng tôi nỗ lực hoạt động hiệu quả, tiết kiệm và có trách nhiệm, góp phần ứng phó biến đổi khí hậu, thúc đẩy kinh tế tuần hoàn và bảo vệ tài nguyên chung, Biến chất thải sản xuất thành các nguyên liệu có giá trị cho các ngành khác, góp phần giảm khai thác tài nguyên thô. Thành viên của các Hiệp hội ngành nghề 4 Hoạt động kinh doa"}\n{"id": "B-A-20df4e2d39e53143", "claim": "Trọng tâm chiến lược tập trung vào ba trụ cột ESG chính:", "evidence": "TIÊU CHUẨN CHUNG NỘI DUNG CÓ NÊU TRONG BÁO MÃ CÔNG BỐ CHỦ ĐỀ TIÊU ĐỀ CÔNG BỐ GRI CÁO PHÁT TRIỂN PHẦN THAM CHIẾU/ LÝ DO LOẠI TRỪ GRI BỀN VỮNG NĂM 2025 2-16 Truyền đạt các mối quan tâm chính Quản trị Doanh nghiệp & Quản lý rủi ro 2-17 Kiến thức chung của cơ quan quản trị cao nhất Đã trình bày tại Báo cáo thường niên của tập đoàn Hòa Phát 2025 2-18 Đánh giá hiệu quả hoạt động của cơ quan trị cao nhất Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-19 Chính sách thù lao Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-20 Quy trình xác định thù lao Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-21 Tỷ lệ tổng thù lao hàng năm Đã trình bày tại Tài liệu Đại hội đồng cổ đông 2025 2-"}\n{"id": "B-A-65858935f973f8c2", "claim": "Xanh hóa Sản xuất (Greening Our Production), Phát triển Con người (Caring for", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 5 PHẦN XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu Quản lý môi trường 38 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 5 PHẦN XANH HÓA SẢN XUẤT Hành động giảm thiểu và thích ứng với biến đổi khí hậu Quản lý môi trường 35"}\n{"id": "B-A-5079b7b613d70c54", "claim": "Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-c2ae880b11de7c33", "claim": "thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên", "evidence": "XANH HÓA SẢN XUẤT QUẢN TRỊ VÀ ĐỔI MỚI Trụ cột Xanh hóa Sản xuất hướng tới hài hòa giữa tăng trưởng công nghiệp và bảo vệ môi trường, giảm thiểu biến đổi khí hậu. Chúng tôi nhận thức về trách nhiệm của mình trong việc tham gia lộ trình giảm phát thải quốc gia và đang tích cực thực hành các giải pháp phù hợp với điều kiện kinh doanh, đặc biệt trong lĩnh vực thép, để đồng hành cùng Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050. Tập đoàn hướng tới từng bước chuyển đổi mô hình sản xuất và quản trị, gắn liền với cải tiến công nghệ, tối ưu hiệu quả sử dụng tài nguyên, và giảm thiểu phát thải ra môi trường. Với Khu liên hợp Gang thép tại Dung Quất"}\n{"id": "B-A-765a5fbd1543a452", "claim": "vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao", "evidence": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Giải pháp này bao gồm các hoạt động giúp tối ưu vận hành như cải thiện chất lượng phối liệu, kiểm soát kích cỡ than cốc, cải thiện phân bố liệu, tối ưu tốc độ gió nóng và"}\n{"id": "B-A-01c692d6bdec3f31", "claim": "thúc đẩy kinh tế tuần hoàn và bảo vệ hệ sinh thái, cụ", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-bc2d8512f78e7dad", "claim": "Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở", "evidence": "vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi "}\n{"id": "B-A-2c4b6f800a24834b", "claim": "khép kín, tái chế nước và xử lý triệt để để tuân thủ quy chuẩn xả thải.", "evidence": "a đó giảm phát sinh chất thải. • Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng các hệ thống tuần hoàn khép kín, tái chế nước và xử lý triệt để để tuân thủ quy chuẩn xả thải. • Giám sát thường xuyên khí thải để bảo đảm mọi thông số nằm trong giới hạn cho phép. QUẢN LÝ MÔI TRƯỜNG Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát Gắn kết với các bên liên quan Hòa Phát cam kết đảm bảo một môi trường làm việc an toàn cho toàn thể người lao động. Thông qua quản lý rủi ro chặt chẽ, đào tạo liên tục và đầu tư vào công nghệ an toàn, Tập đoàn hướng tới xây dựng nơi làm việc là nơi mọi nhân viên đều có thể trở về nhà an toàn mỗi ngày. S"}\n{"id": "B-A-02a3b329608544eb", "claim": "động, củng cố quan hệ hợp tác và xác định các mục tiêu phát triển bền vững phù", "evidence": "quan hệ hợp tác và xác định các mục tiêu phát triển bền vững phù hợp trong từng giai đoạn. Quản trị phát triển bền vững XÁC ĐỊNH CÁC BÊN LIÊN QUAN Nhận diện các vấn đề trọng yếu Dựa trên kinh nghiệm thực tiễn của các công ty thành viên,Tập đoàn Hòa Phát đã xác định các bên liên quan chính thông qua việc phân tích các đặc điểm như: mức độ phụ thuộc vào hoạt động của doanh nghiệp, trách nhiệm liên quan, mức độ ảnh hưởng đến quyết định chiến lược, sự đa dạng trong quan điểm và tiềm năng phát sinh xung đột trong quá trình tương tác. Việc xác định này tuân thủ theo Tiêu chuẩn AA1000 về Tham vấn các bên liên quan (AA1000SES) – Khung hướng dẫn quốc tế nhằm đảm bảo quá trình tham vấn được thực hiện "}\n{"id": "B-A-1e6d0799539822c0", "claim": "quan hệ hợp tác và xác định các mục tiêu phát triển bền vững phù", "evidence": "GẮN KẾT VỚI CÁC BÊN LIÊN QUAN 4.4 [GRI 2-29, 2-30] Tập đoàn Hòa Phát luôn chú trọng việc xây dựng và phát triển quan hệ hợp tác, chia sẻ và gắn kết lợi ích với các bên liên quan. Do đó, chúng tôi luôn duy trì các kênh đối thoại, tương tác hiệu quả bằng nhiều hình thức khác nhau nhằm kịp thời nắm bắt nguyện vọng và sự quan tâm của các bên liên quan. Việc tham vấn được triển khai thường xuyên, đóng vai trò quan trọng trong quá trình hoạch định chiến lược và ra quyết định của Tập đoàn. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT Để đảm bảo các vấn đề trọng yếu được xử lý đúng trọng tâm, Hòa Phát phân công trách nhiệm cụ thể cho từng đơn vị, tương ứng với nh"}\n{"id": "B-A-e7bd1f4d7c928e16", "claim": "Xây dựng và duy trì mối quan hệ hợp tác bền vững và lâu dài dựa", "evidence": "PHÁT TRIỂN CỘNG ĐỒNG 6.4 GRI 204, 411, 413] Với triết lý “Hòa hợp cùng phát triển”, Hòa Phát tích hợp mục tiêu phát triển kinh tế với trách nhiệm xã hội, hướng đến tăng trưởng bền vững và tạo giá trị chung cho các bên liên quan. Mọi hoạt động đều mang đến mục tiêu tạo ra giá trị lâu dài cho xã hội, lan tỏa tinh thần sẻ chia và đóng góp thiết thực cho đời sống người dân tại những địa phương nơi Hòa Phát hoạt động. CÁC YẾU TỐ CHÍNH TRONG PHƯƠNG PHÁP TIẾP CẬN CỦA HÒA PHÁT BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 HỖ TRỢ CÓ TRỌNG ĐIỂM: Các chương trình được thiết kế để tập trung giúp đỡ những nhóm đối tượng yếu thế, dễ bị tổn thương trong xã hội để nguồn lực được sử dụng hiệu quả và tạo ra tác động rõ rệ"}\n{"id": "B-A-a404a18f85ecd37a", "claim": "Điện máy gia dụng: Các tính năng bảo vệ sức khỏe, tiết kiệm năng lượng,", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Quản trị Doanh nghiệp và Quản lý rủi ro Quản lý chất lượng Phát triển đổi mới sáng tạo LĨNH VỰC ĐIỆN MÁY GIA DỤNG Các sản phẩm điện lạnh và điện máy gia dụng được thực hiện đánh giá tác động đến sức khỏe người tiêu dùng trong suốt vòng đời sản phẩm. Trong lĩnh vực điện lạnh, các sản phẩm được thiết kế đảm bảo an toàn về điện, cơ khí và cách nhiệt, đáp ứng đầy đủ các tiêu chuẩn kỹ thuật hiện hành. Tất cả vật liệu và linh kiện sử dụng đều có nguồn gốc rõ ràng, không chứa các chất độc hại như CFC, HCFC, chì hoặc thủy ngân. Các bộ phận chính của sản phẩm được kiểm tra và thử nghiệm nghiêm ngặt nhằm ngăn ngừa nguy "}\n{"id": "B-A-e5c697ca633943da", "claim": "Công bố thông tin báo cáo các chỉ số ESG và giải pháp quản lý rủi ro ESG liên", "evidence": "HÒA PHÁT Các yếu tố bền vững được tích hợp vào quá trình thảo luận và xây dựng chiến lược, ra quyết định đầu tư và giám sát quản lý rủi ro của tập đoàn và các công ty thành viên. Ban ESG chịu trách nhiệm giám sát các vấn đề liên quan đến ESG, bao gồm cả rủi ro ESG. Ban ESG được thành lập với sự tham gia của Ban Tổng Giám đốc, Ban Giám đốc các công ty thành viên cùng các ban chức năng như ban Tài chính, Pháp chế, Nhân sự, Truyền thông của Tập đoàn nhằm điều phối các hoạt động ESG và tích hợp các mục tiêu phát triển bền vững vào hoạt động vận hành hàng ngày của doanh nghiệp. CÁC BAN CHỨC NĂNG CỦA TẬP ĐOÀN: ĐIỀU PHỐI • Đầu mối tổng hợp báo cáo Phát triển bền vững • Điều phối chung các hoạt động"}\n{"id": "B-A-579ee5e7b35ecfbe", "claim": "quan đến môi trường (phát thải, tiêu thụ năng lượng, quản lý nước, chất thải),", "evidence": "QUẢN LÝ MÔI TRƯỜNG GRI 103 – Năng lượng 2025 3-3 Quản lý chủ đề trọng yếu Phương pháp quản lý QUẢN LÝ CHẤT LƯỢNG KHÔNG KHÍ Kiểm soát chất lượng không khí Oxit nitơ (NOx). oxit lưu huỳnh (SOx) và các khí thải không khí đáng GRI 305- Phát thải 2016 305-7 kể khác QUẢN LÝ NƯỚC GRI 3 – Chủ đề trọng yếu 2021 3-3 Quản lý chủ đề trọng yếu Quản lý nước 303-2 Quản lý tác động liên quan đến xả thải nước Quản lý nước GRI 303 – Nước và Nước thải 2018 303-3 Lượng nước khai thác Quản lý nước 303-4 Lượng nước thải Quản lý nước 303-5 Lượng nước tiêu thụ Quản lý nước QUẢN LÝ CHẤT THẢI RẮN GRI 3 – Chủ đề trọng yếu 2021 3-3 Quản lý chủ đề trọng yếu Quản lý chất thải rắn GRI 306: Waste 2020 306-1 Phát sinh chất "}\n{"id": "B-A-33943d13a577e87c", "claim": "Nhận thức rõ Biến đổi Khí hậu là thách thức toàn cầu, và hưởng ứng cam kết Net Zero", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 Trong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác động rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. Nhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress nhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm vào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm tăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group – Asian Development Bank. nô"}\n{"id": "B-A-16e4796649cc3076", "claim": "ứng với Biến đổi Khí hậu” là ưu tiên phát triển bền vững quan trọng.", "evidence": "NHẬN DIỆN CÁC VẤN ĐỀ TRỌNG YẾU 4.2 MA TRẬN TRỌNG YẾU BỀN VỮNG Ưu tiên thấp Ưu tiên trung bình Ưu tiên cao BÁO CÁO PHÁT TRIỂN BỀN VỮNG 1 2 NHÓM CHỦ ĐỀ ƯU TIÊN CAO 2025 1. Giảm thiểu và Thích ứng với biến đổi khí hậu 3 7 2. Sức khỏe & An toàn lao động Thấp Trung bình Cao PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT TRỌNG YẾU VỚI CÁC BÊN LIÊN QUAN 4 3. Quản trị doanh nghiệp 8 4. Quản lý môi trường 10 5 5. Quản lý chất lượng 6 6. Quản trị nguồn nhân lực 9 NHÓM CHỦ ĐỀ ƯU TIÊN TRUNG BÌNH 12 11 7. Nhân quyền 8. Quản trị rủi ro Quản trị phát triển bền vững 9. Phát triển đổi mới sáng tạo Nhận diện các vấn đề trọng yếu NHÓM CHỦ ĐỀ ƯU TIÊN THẤP Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát 1"}\n{"id": "B-A-0839e84c70c69b0c", "claim": "nhằm tiết kiệm năng lượng và giảm phát thải khí nhà kính phù hợp với điều kiện kinh", "evidence": "m, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023. Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn. Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám"}\n{"id": "B-A-b4c849d3ff07a7d8", "claim": "danh mục phải kiểm kê phát thải theo Nghị định 06/2022/NĐ-CP đã tiến hành", "evidence": "ểm kê KNK đầy đủ cho các cơ quan chức năng từ năm 2024. PHẦN 5 XANH HÓA SẢN XUẤT PHƯƠNG PHÁP QUẢN LÝ Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn. Trong đó, hai Khu liên hợp Gang thép Hòa Phát Hải Dương và Hòa Phát Dung Quất — chiếm khoảng 94% tổng công suất thép thô toàn Tập đoàn — là đơn vị trọng điểm trong công tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải. • Phục vụ quản lý nội bộ: Ngoài đáp ứng yêu cầu của phá"}\n{"id": "B-A-533fb51e44257d39", "claim": "trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-9fe148dd26f18c1e", "claim": "thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn.", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-62329d4482e8c58b", "claim": "tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải.", "evidence": "rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng. Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. Trụ cột Quản trị và Đổi "}\n{"id": "B-A-cd5aba8f6e91f131", "claim": "cho nhu cầu quản lý phát thải chung của cả Tập đoàn.", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-26f7afed0ab1715a", "claim": "động thực hiện kiểm kê dấu chân carbon cho các sản phẩm, thực hiện theo chuẩn", "evidence": "m, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023. Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn. Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám"}\n{"id": "B-A-bc65d3147a432b17", "claim": "chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023.", "evidence": ". Tập đoàn duy trì cơ chế giám sát và kiểm soát nội bộ chặt chẽ đối với các quy trình thuế, đảm bảo tính liêm chính, tuân thủ chuẩn mực cao nhất và ngăn chặn mọi hình thức trốn thuế hoặc hành vi thiếu minh bạch, từ đó củng cố niềm tin với cơ quan quản lý và cộng đồng. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Hiệu quả của các biện pháp quản lý rủi ro được đánh giá định kỳ, đồng thời điều chỉnh kịp thời để thích ứng với tình hình thực tế. Hằng năm, Hòa Phát tiến hành đánh giá tổng thể các rủi ro ảnh hưởng đến hoạt động và phát triển bền vững trên cả ba khía cạnh: môi trường, xã hội và kinh tế. Quá trình này bao gồm phân tích các yếu tố nội bộ – bên ngoài, xu hướng thị trường"}\n{"id": "B-A-9194ee5093ceec0f", "claim": "Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa", "evidence": "TIÊU CHUẨN CHỦ ĐỀ NỘI DUNG CÓ TRONG BÁO CÁO MÃ CÔNG BỐ CHỦ ĐỀ TIÊU ĐỀ CÔNG BỐ GRI PHÁT TRIỂN BỀN PHẦN THAM CHIẾU/ LÝ DO LOẠI TRỪ GRI VỮNG NĂM 2025 GRI 3: CHỦ ĐỀ TRỌNG YẾU 2021 3-1 Quy trình xác định chủ đề trọng yếu Nhận diện các vấn đề trọng yếu Công bố về các chủ đề trọng yếu 3-2 Danh sách chủ đề trọng yếu Nhận diện các vấn đề trọng yếu XANH HÓA SẢN XUẤT GIẢI PHÁP GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU Quản lý chủ đề trọng yếu 3-3 Phương pháp quản lý GRI 3 – Chủ đề trọng yếu 2021 103-1 Chính sách và cam kết năng lượng Phương pháp quản lý 103-2 Tiêu thụ năng lượng và năng lượng tự tạo ra trong tổ chức Kết quả tiêu thụ năng lượng năm 2025 GRI 103 – Năng lượng 2025 103-3 Tiêu thụ năng l"}\n{"id": "B-A-aba5856407a68944", "claim": "phát thải và thúc đẩy các sáng kiến giảm phát thải.", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-f494203ad791219d", "claim": "lý phát thải khí nhà kính và định lượng dấu chân carbon của sản phẩm, bao gồm", "evidence": "ểm kê KNK đầy đủ cho các cơ quan chức năng từ năm 2024. PHẦN 5 XANH HÓA SẢN XUẤT PHƯƠNG PHÁP QUẢN LÝ Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn. Trong đó, hai Khu liên hợp Gang thép Hòa Phát Hải Dương và Hòa Phát Dung Quất — chiếm khoảng 94% tổng công suất thép thô toàn Tập đoàn — là đơn vị trọng điểm trong công tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải. • Phục vụ quản lý nội bộ: Ngoài đáp ứng yêu cầu của phá"}\n{"id": "B-A-ceeabae05d1c30aa", "claim": "Ban xây dựng kế hoạch giảm phát thải, theo dõi tiến độ triển khai và đề xuất các", "evidence": "m, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023. Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn. Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám"}\n{"id": "B-A-60930c4ccbaa5733", "claim": "ứng các yêu cầu về giảm phát thải khí nhà kính của Cơ quan quản lý được đề cập", "evidence": "m, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023. Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn. Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám"}\n{"id": "B-A-e4c165543cb9d90f", "claim": "pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-e0d114c04bbe74b0", "claim": "phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp).", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-8f9e2302849c2398", "claim": "(Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-18d841ceaad9c874", "claim": "giúp giảm nhu cầu gia nhiệt, tiết kiệm năng lượng và giảm", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-390a015ca7b786dd", "claim": "Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn,", "evidence": "ểm kê KNK đầy đủ cho các cơ quan chức năng từ năm 2024. PHẦN 5 XANH HÓA SẢN XUẤT PHƯƠNG PHÁP QUẢN LÝ Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn. Trong đó, hai Khu liên hợp Gang thép Hòa Phát Hải Dương và Hòa Phát Dung Quất — chiếm khoảng 94% tổng công suất thép thô toàn Tập đoàn — là đơn vị trọng điểm trong công tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải. • Phục vụ quản lý nội bộ: Ngoài đáp ứng yêu cầu của phá"}\n{"id": "B-A-811ab5a0f6dac168", "claim": "Các khía cạnh quản lý phát thải KNK bao gồm: Năng lượng, Nguyên vật liệu và", "evidence": "PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 CHUYỂN ĐỔI SỐ TRONG SẢN XUẤT VÀ QUẢN TRỊ DOANH NGHIỆP PHƯƠNG PHÁP QUẢN LÝ PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI E S G MÔI TRƯỜNG: Ứng dụng công nghệ số giúp giám sát và tối ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành; đồng thời hỗ trợ cảnh báo sớm khi các chỉ số môi trường vượt ngưỡng và nâng cao tính minh bạch, tuân thủ trong quản lý phát thải. PHƯƠNG PHÁP QUẢN LÝ BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt kh"}\n{"id": "B-A-d6f4b3a10114d2ca", "claim": "cải tiến liên tục và giảm phát thải carbon.", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-eada9ac1292ace49", "claim": "thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên", "evidence": "vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi "}\n{"id": "B-A-e956f1857b097ce4", "claim": "nghiên cứu đầu tư điện mặt trời áp mái tại các cơ sở sản xuất.", "evidence": "rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng. Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. Trụ cột Quản trị và Đổi "}\n{"id": "B-A-7ae1bdf26272e8ed", "claim": "quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm", "evidence": "TIÊU CHUẨN CHỦ ĐỀ NỘI DUNG CÓ TRONG BÁO CÁO MÃ CÔNG BỐ CHỦ ĐỀ TIÊU ĐỀ CÔNG BỐ GRI PHÁT TRIỂN BỀN PHẦN THAM CHIẾU/ LÝ DO LOẠI TRỪ GRI VỮNG NĂM 2025 GRI 3: CHỦ ĐỀ TRỌNG YẾU 2021 3-1 Quy trình xác định chủ đề trọng yếu Nhận diện các vấn đề trọng yếu Công bố về các chủ đề trọng yếu 3-2 Danh sách chủ đề trọng yếu Nhận diện các vấn đề trọng yếu XANH HÓA SẢN XUẤT GIẢI PHÁP GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU Quản lý chủ đề trọng yếu 3-3 Phương pháp quản lý GRI 3 – Chủ đề trọng yếu 2021 103-1 Chính sách và cam kết năng lượng Phương pháp quản lý 103-2 Tiêu thụ năng lượng và năng lượng tự tạo ra trong tổ chức Kết quả tiêu thụ năng lượng năm 2025 GRI 103 – Năng lượng 2025 103-3 Tiêu thụ năng l"}\n{"id": "B-A-89a7eef0b9ce480f", "claim": "phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng", "evidence": "ện năng tiêu thụ so với phương pháp truyền thống. • Thu hồi nhiệt từ quá trình thiêu kết, sử dụng nhiệt dư để phát điện, góp phần tối ưu hóa hiệu quả sử dụng năng lượng tổng thể. • Công nghệ đúc – cán liên tục cho phép nạp trực tiếp phôi nóng (750–900°C) vào dây chuyền cán. giúp giảm nhu cầu gia nhiệt, tiết kiệm năng lượng và giảm phát thải CO2 trên mỗi tấn thép. • Tận dụng các nguồn hơi nước từ hệ thống làm mát lò thổi, lò nung phôi để tận dụng phát điện. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn, do vậy kỳ báo cáo từ ngày 01/01/2025 đến 31/12/2025 được lựa chọn làm năm cơ sở vì đây là kỳ báo cáo đầu t"}\n{"id": "B-A-5fca5dacb44cbb6c", "claim": "Nguồn điện xanh này được tích hợp vào hệ thống cấp điện nội bộ, thay thế một", "evidence": "rời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy. Nguồn điện xanh này được tích hợp vào hệ thống cấp điện nội bộ, thay thế một phần điện từ lưới quốc gia. Đây là giải pháp sạch, không phát thải tại chỗ, góp phần lớn vào mục tiêu trung hòa carbon trong dài hạn. Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai các giải pháp chủ chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng cao hiệu quả vận hành và giảm phát thải. Các giải pháp bao gồm: 36 HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THU HỒI NHIỆT ĐỂ PHÁT ĐIỆN Từ năm 2025, Tập đoàn Hòa Phát chính thức đẩy mạnh lộ trình “xanh hóa” sản xuất thông qua việc triển kha"}\n{"id": "B-A-549bf8b0e555b7a8", "claim": "Đây là giải pháp sạch, không phát thải tại chỗ, góp", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-b9821dba75b98842", "claim": "phần lớn vào mục tiêu trung hòa carbon trong dài hạn.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 SẢN LƯỢNG THÉP THÔ THEO QUY TRÌNH SẢN XUẤT Tỷ trọng thép BOF trong sản lượng toàn cầu Tỷ trọng thép EAF trong sản lượng toàn cầu BÁO CÁO PHÁT TRIỂN BỀN VỮNG 70,4% 29,1% 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Cường độ phát thải CO2 trung bình (Scrap-EAF): 0,70 tấn CO2 / tấn thép thô Điểm nổi bật về phát triển bền vững năm 2025 Cường độ phát thải CO2 Cường độ phát thải CO2 trung bình (BF-BOF): 2,32 tấn CO2 / tấn thép thô trung bình (DRI-EAF): 1,43 tấn CO2 / tấn thép thô Tại Hòa Phát, phần lớn thép thô sản xuất từ công nghệ lò cao (KLH Gang thép Hòa Phát Giải thưởng và vinh danh năm 2025 Hải Dương và Dung Quất), một phần còn lại được sản xuất t"}\n{"id": "B-A-fb525462550301dd", "claim": "chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-1f8e592d97309cc5", "claim": "cao hiệu quả vận hành và giảm phát thải.", "evidence": "PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 CHUYỂN ĐỔI SỐ TRONG SẢN XUẤT VÀ QUẢN TRỊ DOANH NGHIỆP PHƯƠNG PHÁP QUẢN LÝ PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI E S G MÔI TRƯỜNG: Ứng dụng công nghệ số giúp giám sát và tối ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành; đồng thời hỗ trợ cảnh báo sớm khi các chỉ số môi trường vượt ngưỡng và nâng cao tính minh bạch, tuân thủ trong quản lý phát thải. PHƯƠNG PHÁP QUẢN LÝ BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt kh"}\n{"id": "B-A-b48981626b0587d2", "claim": "Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 SẢN LƯỢNG THÉP THÔ THEO QUY TRÌNH SẢN XUẤT Tỷ trọng thép BOF trong sản lượng toàn cầu Tỷ trọng thép EAF trong sản lượng toàn cầu BÁO CÁO PHÁT TRIỂN BỀN VỮNG 70,4% 29,1% 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Cường độ phát thải CO2 trung bình (Scrap-EAF): 0,70 tấn CO2 / tấn thép thô Điểm nổi bật về phát triển bền vững năm 2025 Cường độ phát thải CO2 Cường độ phát thải CO2 trung bình (BF-BOF): 2,32 tấn CO2 / tấn thép thô trung bình (DRI-EAF): 1,43 tấn CO2 / tấn thép thô Tại Hòa Phát, phần lớn thép thô sản xuất từ công nghệ lò cao (KLH Gang thép Hòa Phát Giải thưởng và vinh danh năm 2025 Hải Dương và Dung Quất), một phần còn lại được sản xuất t"}\n{"id": "B-A-d5218284555c48d7", "claim": "so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-17fba04ccac352cf", "claim": "khiết và chất lượng cao, song cũng phát thải CO2 nhiều hơn so với lò điện hồ quang (EAF).", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 SẢN LƯỢNG THÉP THÔ THEO QUY TRÌNH SẢN XUẤT Tỷ trọng thép BOF trong sản lượng toàn cầu Tỷ trọng thép EAF trong sản lượng toàn cầu BÁO CÁO PHÁT TRIỂN BỀN VỮNG 70,4% 29,1% 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Cường độ phát thải CO2 trung bình (Scrap-EAF): 0,70 tấn CO2 / tấn thép thô Điểm nổi bật về phát triển bền vững năm 2025 Cường độ phát thải CO2 Cường độ phát thải CO2 trung bình (BF-BOF): 2,32 tấn CO2 / tấn thép thô trung bình (DRI-EAF): 1,43 tấn CO2 / tấn thép thô Tại Hòa Phát, phần lớn thép thô sản xuất từ công nghệ lò cao (KLH Gang thép Hòa Phát Giải thưởng và vinh danh năm 2025 Hải Dương và Dung Quất), một phần còn lại được sản xuất t"}\n{"id": "B-A-71262447d78eaa00", "claim": "Ngược lại, EAF có mức phát thải thấp hơn, nhưng phụ thuộc vào nguồn nguyên liệu như thép", "evidence": "Luyện thép bằng lò điện hồ quang (EAF) là công nghệ sử dụng thép phế liệu làm nguyên liệu chính. Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch."}\n{"id": "B-A-0976a4a8a7b38abd", "claim": "phù hợp với các quốc gia phát triển, nơi ngành thép đã đi qua nhiều vòng đời tái chế.", "evidence": "óa thạch. • Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu. Khi các sản phẩm từ thép như vỏ ô tô, khung nhà hay thiết bị điện tử...hết thời hạn sử dụng, Giải thưởng và vinh danh năm 2025 thay vì trở thành rác thải, chúng được thu gom, phân tách và đưa vào lò điện để nấu Sản xuất thép chảy lại thành thép lỏng, tạo ra các sản phẩm thép mới.. Giai đoạn này gắn với công "}\n{"id": "B-A-ea1661af1887dc11", "claim": "Hiện nay, các biện pháp giảm phát thải trong sản xuất thép từ quặng", "evidence": "i chế. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Hiện nay, các biện pháp giảm phát thải trong sản xuất thép từ quặng (vòng đời sơ cấp – sản xuất thép mới) trên thế giới chủ yếu tập trung vào việc sử dụng các loại nguyên liệu đầu vào có hàm lượng carbon thấp làm chất khử quặng sắt như khí tự nhiên hoặc Hydrogen xanh (sản xuất từ nguồn điện sạch) để tạo ra sắt hoàn nguyên trực tiếp (DRI và HBI), thay thế cho việc hoàn nguyên bằng nhiên liệu hóa thạch truyền thống như than. Hạn chế khiến cho biện pháp này chưa thể áp dụng rộng rãi nằm ở sự khan hiếm nguồn cung các loại quặng sắt có hàm lượng sắt cao %Fe>65% cũng như nguồn khí tự nhiên và hydrogen sạch. Biện pháp thứ hai là sử dụng thép phế liệu để tái c"}\n{"id": "B-A-1bc79004d935400e", "claim": "vào việc sử dụng các loại nguyên liệu đầu vào có hàm lượng carbon thấp", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP • Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - Từ quặng đến thép Vòng đời sơ cấp của thép bắt đầu từ việc khai thác các nguồn tài nguyên thiên nhiên như quặng sắt, than đá và đá vôi để khởi động quy trình sản xuất vật liệu mới. Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF). Trong các lò cao làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế, BÁO CÁO PHÁT TRIỂN BỀN VỮNG công nghiệp, quặng sắt được hoàn nguyên và nấu chảy ở nhiệt độ cực cao để tạo ra 2025 tái sản xuất. Vòng đ"}\n{"id": "B-A-b83722207a630711", "claim": "làm chất khử quặng sắt như khí tự nhiên hoặc Hydrogen xanh (sản xuất", "evidence": "lý môi trường TĂNG SỬ DỤNG THÉP PHẾ LIỆU Tập đoàn đang tăng cường tỷ lệ sử dụng thép phế và HBI trong lò thổi. Biện pháp này giúp giảm mạnh nhu cầu hoàn nguyên quặng, giảm tiêu hao than cốc, tận dụng nhiệt lượng và giảm phát thải khí nhà kính trong toàn bộ chuỗi sản xuất, đồng thời gia tăng năng lực sản xuất thép. Đến năm 2030, các công ty thép của tập đoàn Hòa Phát dự kiến tăng sử dụng thép phế đạt 30%. ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Tập đoàn lập kế hoạch triển khai các dự án đề xuất tăng cường phối phế liệu cùng gang lỏng trong lò thổi để tăng tỷ trọng nguyên vật liệu phát thải thấp. Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọn"}\n{"id": "B-A-fe1a1066f6e68bfc", "claim": "hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp –", "evidence": "óa thạch. • Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu. Khi các sản phẩm từ thép như vỏ ô tô, khung nhà hay thiết bị điện tử...hết thời hạn sử dụng, Giải thưởng và vinh danh năm 2025 thay vì trở thành rác thải, chúng được thu gom, phân tách và đưa vào lò điện để nấu Sản xuất thép chảy lại thành thép lỏng, tạo ra các sản phẩm thép mới.. Giai đoạn này gắn với công "}\n{"id": "B-A-b6ba0b8a92684619", "claim": "hao than cốc, tận dụng nhiệt lượng và giảm phát thải khí nhà kính trong", "evidence": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Giải pháp này bao gồm các hoạt động giúp tối ưu vận hành như cải thiện chất lượng phối liệu, kiểm soát kích cỡ than cốc, cải thiện phân bố liệu, tối ưu tốc độ gió nóng và"}\n{"id": "B-A-6ce0ea0632290fe2", "claim": "lỏng trong lò thổi để tăng tỷ trọng nguyên vật liệu phát thải thấp.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 NGUYÊN VẬT LIỆU: CÔNG NGHỆ: CHUYỂN DỊCH THEO HƯỚNG TĂNG TỶ TRỌNG NGUYÊN VẬT LIỆU PHÁT THẢI THẤP HƯỚNG TỚI CÔNG NGHỆ SẢN XUẤT THÉP XANH Luyện thép bằng lò điện hồ quang (EAF) là công nghệ sử dụng thép phế liệu làm nguyên liệu chính. Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời hoặc điện gió. Tuy nhiên, việc chuyển dịch công nghệ phụ thuộc cao vào giải quyết bài toán nguyên vật liệu. Do sự khan hiếm trong nguồn cung nguyên liệu sạch, hiện nay, hơn 70% sản lượng thép toàn cầu vẫn đang được sản xuất theo công nghệ lò c"}\n{"id": "B-A-ee28c590ffe8a11b", "claim": "thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao", "evidence": "i chế. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Hiện nay, các biện pháp giảm phát thải trong sản xuất thép từ quặng (vòng đời sơ cấp – sản xuất thép mới) trên thế giới chủ yếu tập trung vào việc sử dụng các loại nguyên liệu đầu vào có hàm lượng carbon thấp làm chất khử quặng sắt như khí tự nhiên hoặc Hydrogen xanh (sản xuất từ nguồn điện sạch) để tạo ra sắt hoàn nguyên trực tiếp (DRI và HBI), thay thế cho việc hoàn nguyên bằng nhiên liệu hóa thạch truyền thống như than. Hạn chế khiến cho biện pháp này chưa thể áp dụng rộng rãi nằm ở sự khan hiếm nguồn cung các loại quặng sắt có hàm lượng sắt cao %Fe>65% cũng như nguồn khí tự nhiên và hydrogen sạch. Biện pháp thứ hai là sử dụng thép phế liệu để tái c"}\n{"id": "B-A-339343d343a82738", "claim": "than cốc và các nhiên liệu trên một tấn thép, đồng thời tiết kiệm năng lượng trong toàn", "evidence": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Giải pháp này bao gồm các hoạt động giúp tối ưu vận hành như cải thiện chất lượng phối liệu, kiểm soát kích cỡ than cốc, cải thiện phân bố liệu, tối ưu tốc độ gió nóng và"}\n{"id": "B-A-210f5055c4910ebd", "claim": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như", "evidence": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Giải pháp này bao gồm các hoạt động giúp tối ưu vận hành như cải thiện chất lượng phối liệu, kiểm soát kích cỡ than cốc, cải thiện phân bố liệu, tối ưu tốc độ gió nóng và"}\n{"id": "B-A-6d79f5bd86e94cb2", "claim": "đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 NGUYÊN VẬT LIỆU: CÔNG NGHỆ: CHUYỂN DỊCH THEO HƯỚNG TĂNG TỶ TRỌNG NGUYÊN VẬT LIỆU PHÁT THẢI THẤP HƯỚNG TỚI CÔNG NGHỆ SẢN XUẤT THÉP XANH Luyện thép bằng lò điện hồ quang (EAF) là công nghệ sử dụng thép phế liệu làm nguyên liệu chính. Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời hoặc điện gió. Tuy nhiên, việc chuyển dịch công nghệ phụ thuộc cao vào giải quyết bài toán nguyên vật liệu. Do sự khan hiếm trong nguồn cung nguyên liệu sạch, hiện nay, hơn 70% sản lượng thép toàn cầu vẫn đang được sản xuất theo công nghệ lò c"}\n{"id": "B-A-6f163d4f9e8af4c2", "claim": "trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch.", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-ad1cafef1a3686c6", "claim": "trong giảm phát thải trực tiếp tại khu vực lò cao và nâng cao hiệu suất luyện gang.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 Với mục đích phản ánh một cách chính xác và khách quan nhất những tác động biến đổi khí hậu xuất phát từ các hoạt động trong phạm vi kinh doanh của mình cũng như trình bày một cách minh bạch các kết quả đã đạt được và những giải pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp). Bên cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài (Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt động này cũng góp phần "}\n{"id": "B-A-474a7a24ab0b14f4", "claim": "Từ năm 2025, Tập đoàn Hòa Phát chính thức đẩy mạnh lộ trình “xanh hóa” sản", "evidence": "THÀNH VIÊN CỦA CÁC HIỆP HỘI NGÀNH NGHỀ 3.3 Tập đoàn Hòa Phát tích cực tham gia nhiều hiệp hội ngành nghề và tổ chức chuyên môn nhằm thúc đẩy thực hành kinh doanh có trách nhiệm, thúc đẩy đổi mới sáng tạo và đóng góp vào sự phát triển công nghiệp bền vững tại Việt Nam cũng như trên toàn cầu. Thông qua việc tham gia các hiệp hội này, Tập đoàn tham gia đối thoại chính sách, chia sẻ tri thức và hợp tác trong các sáng kiến hướng tới bảo vệ môi trường, nâng cao an toàn lao động và thúc đẩy kinh tế tuần hoàn trong ngành thép. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT CÁC HIỆP HỘI VÀ TỔ CHỨC THÀNH VIÊN TIÊU BIỂU Tập đoàn Hòa Phát và các công ty thành viên hiện là thành v"}\n{"id": "B-A-16d219bd99d67020", "claim": "xuất thông qua việc triển khai hệ thống điện mặt trời áp mái tại các khu liên hợp", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-5f6c0efa0532ab50", "claim": "còn giảm thiểu đáng kể phát thải nhiệt và khí thải ra môi trường.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 Với mục đích phản ánh một cách chính xác và khách quan nhất những tác động biến đổi khí hậu xuất phát từ các hoạt động trong phạm vi kinh doanh của mình cũng như trình bày một cách minh bạch các kết quả đã đạt được và những giải pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp). Bên cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài (Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt động này cũng góp phần "}\n{"id": "B-A-936daceb510c2f9f", "claim": "mắt xích quan trọng trong mô hình kinh tế tuần hoàn, góp phần thực hiện mục", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-7729034b8e9b8e3e", "claim": "Đây là bước đi then chốt nhằm giảm dấu chân carbon trên", "evidence": "rời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy. Nguồn điện xanh này được tích hợp vào hệ thống cấp điện nội bộ, thay thế một phần điện từ lưới quốc gia. Đây là giải pháp sạch, không phát thải tại chỗ, góp phần lớn vào mục tiêu trung hòa carbon trong dài hạn. Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai các giải pháp chủ chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng cao hiệu quả vận hành và giảm phát thải. Các giải pháp bao gồm: 36 m, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều chỉnh biên gi"}\n{"id": "B-A-af57f798f084f1fd", "claim": "TRIỂN KHAI ĐIỆN MẶT TRỜI ÁP MÁI TẠI CÁC NHÀ MÁY", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THU HỒI NHIỆT ĐỂ PHÁT ĐIỆN Từ năm 2025, Tập đoàn Hòa Phát chính thức đẩy mạnh lộ trình “xanh hóa” sản xuất thông qua việc triển khai hệ thống điện mặt trời áp mái tại các khu liên hợp và nhà máy trên toàn quốc, với tổng công suất mục tiêu lên đến hơn 130 MWp. Dự án có sự tham gia sâu rộng của các đơn vị nòng cốt như Khu liên hợp Gang thép Dung Quất và Hải Dương, hệ thống nhà máy Ống thép, Tôn Hòa Phát, các nhà máy sản xuất thức ăn chăn nuôi, điện lạnh và hệ thống trang trại chăn nuôi. Để hiện thực hóa mục tiêu này, hàng triệu tấm pin quang điện hiệu suất cao sẽ được lắp đặt trực tiếp trên hệ thống khung giàn thép kiên cố tại mái nhà "}\n{"id": "B-A-ee07358d0157b85d", "claim": "Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 KIỂM KÊ KHÍ NHÀ KÍNH Từ năm 2023, Tập đoàn đã duy trì thực hiện kiểm kê khí nhà kính trên phần lớn các công ty thành viên, tuân thủ các tiêu cuẩn quốc tế ISO 14064 và ISO 14607. Hoạt động kiểm kê KNK được thực hiện với 3 mục tiêu chính: [GRI 302-1, 302-2, 305-1, 305-2, 305-4] Nhận thức rõ Biến đổi Khí hậu là thách thức toàn cầu, và hưởng ứng cam kết Net Zero 2050 của Việt Nam tại COP26, Tập đoàn Hòa Phát xác định “Giảm thiểu và Thích ứng với Biến đổi Khí hậu” là ưu tiên phát triển bền vững quan trọng. Tập đoàn đang triển khai đồng bộ các sáng kiến và hành động, đặc biệt trong lĩnh vực sản xuất thép, nhằm tiết kiệm năng lượng và giảm "}\n{"id": "B-A-ab5bb72d68b4b56f", "claim": "Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng)", "evidence": "Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý CÁC LOẠI Bụi thải từ các hệ thống xử lý bụi, khí thải Bùn thải từ các trạm xử lý nước thải Phế liệu Xỉ thép Xỉ hạt lò cao CHẤT THẢI RẮN chất thải nguy hại,"}\n{"id": "B-A-593330641262e437", "claim": "thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-1691ad906eaf19a8", "claim": "Các nguồn phát thải chính trong lĩnh vực", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-1ffe880d22749cd0", "claim": "bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp.", "evidence": "vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi "}\n{"id": "B-A-5293b00b8440a559", "claim": "tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không", "evidence": "rường. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Công nghệ áp dụng tại trại chăn nuôi heo và gà của Hòa Phát là công nghệ an toàn cao sinh học với trại lạnh khép kín. Đây là loại hình chăn nuôi tập trung, ứng dụng công nghệ thông tin, công nghệ số vào trong quy trình chăn nuôi, đồng thời đảm bảo các yếu tố thân thiện với môi trường. Trại nuôi được xây dựng khép kín với hệ thống làm mát bằng nước, quạt hút gió, máng ăn tự động, uống nước tự động, chuồng nền, chăn nuôi riêng biệt các đối tượng và giai đoạn của vật nuôi. Nhiệt độ luôn nằm ở mức ổn định từ 18 - 25oC nhờ hệ thống điều khiển tự động, đàn vật nuôi ít bị mệt, giảm stress nhiệt giúp tiêu hóa hiệu quả và giảm phát thải kh"}\n{"id": "B-A-63ec56aa116afe99", "claim": "vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua", "evidence": "của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan kết hợp với đóng góp mà Hòa Phát mang lại cho xã hội và môi trường. Cách tiếp cận này giúp Hòa Phát tập trung nguồn lực phát triển bền vững vào những lĩnh vực then chốt nhất. cấp và nhà đầu tư. Để thu thập ý kiến về mức độ ưu tiên của các chủ đề ESG trọng yếu, Hòa Phát đã tiếp cận các bên liên quan thông qua phỏng vấn và khảo sát, đồng thời ghi nhận các đóng góp liên quan đến điểm mạnh, thách thức và lĩnh vực Hòa Phát cần cải thiện trong quá trình phát triển bền vững. BƯỚC 3 PHÂN TÍCH PHẢN HỒI: Kết quả từ các cuộc phỏng vấn và khảo sát đã được tổng hợp, QUY TRÌNH XÁC ĐỊNH CÁC CHỦ ĐỀ TRỌNG YẾU ĐƯỢC "}\n{"id": "B-A-7510733c210dff9a", "claim": "các biện pháp giảm phát thải khí metan, tối ưu hóa quy trình sản xuất và thúc đẩy", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-320ccb7c51611953", "claim": "Như vậy, việc kiểm soát và giảm thiểu phát thải khí nhà kính cần tập trung vào", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 KIỂM SOÁT CHẤT LƯỢNG KHÔNG KHÍ TRONG CHĂN NUÔI kiện yếm khí để các vi sinh vật phân hủy chất hữu cơ. Quá trình này giúp giảm đáng kể tải lượng ô nhiễm và loại bỏ mùi hôi của chất thải trước khi xả ra môi trường. Quan trọng hơn, biogas tạo ra khí Mê-tan CH4 có thể được sử dụng làm nguồn năng lượng tái tạo để đun nấu hoặc phát điện, giúp giảm chi phí vận hành và đồng thời giảm phát thải khí nhà kính mạnh ra môi trường. Tại các trang trại chăn nuôi heo, chất lượng không khí được kiểm soát bằng kết hợp nhiều biện pháp: BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 KIỂM SOÁT KHẨU PHẦN ĂN CỦA VẬT NUÔI CHUỒNG TRẠI KHÉP KÍN PHẦN 5 XANH HÓA SẢN XUẤT Kiểm soát khẩu phần ăn là một biện pháp c"}\n{"id": "B-A-83750686eeac65fb", "claim": "nhóm nguồn phát thải trực tiếp trong sản xuất và lĩnh vực thép – lĩnh vực đóng vai", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-da8f980a9f0274f8", "claim": "Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-77659dff20c9f928", "claim": "23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải", "evidence": "hiệp bền vững và thích ứng khí hậu. Quản lý môi trường 2. Phát thải theo tổng công ty năm 2025 PHÁT THẢI TOÀN TẬP ĐOÀN NĂM 2025 (tCO2e) TỶ TRỌNG NHÓM NGÀNH Thép 23.369.500 99,55% Nông nghiệp 90.846 0,39% Bất động sản 2.836 0,01% Điện máy gia dụng 9.666 0,04% Tòa nhà văn phòng 1.631 0,01% Cộng 23.474.480 100% 39 HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đó"}\n{"id": "B-A-2dfacdb46acd9409", "claim": "Cụ thể, các nguồn phát thải trực tiếp như quá", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-42d1c6b6c052fa8c", "claim": "Bên cạnh đó, các nguồn phát thải gián tiếp chủ", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 Với mục đích phản ánh một cách chính xác và khách quan nhất những tác động biến đổi khí hậu xuất phát từ các hoạt động trong phạm vi kinh doanh của mình cũng như trình bày một cách minh bạch các kết quả đã đạt được và những giải pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp). Bên cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài (Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt động này cũng góp phần "}\n{"id": "B-A-c0b768b5ea0bd975", "claim": "Xét theo lĩnh vực hoạt động, ngành sản xuất thép là lĩnh vực phát thải lớn nhất,", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-5b32a7139cebc53f", "claim": "đóng góp chủ yếu vào tổng phát thải của toàn tập đoàn.", "evidence": "trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. Từ tháng 9 năm 2025, Hòa Phát Dung Quất 2 đi vào vận hành, Tổng sản lượng thép thô tăng 24%, tương ứng với mức tăng phát thải của toàn tập đoàn. Lĩnh vực nông nghiệp và chăn nuôi của Tập đoàn Hòa Phát ghi nhận mức phát 40 HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đ"}\n{"id": "B-A-a78640f7c42624e4", "claim": "bất động sản và điện máy gia dụng có mức phát thải thấp hơn đáng kể.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 NGUYÊN VẬT LIỆU: CÔNG NGHỆ: CHUYỂN DỊCH THEO HƯỚNG TĂNG TỶ TRỌNG NGUYÊN VẬT LIỆU PHÁT THẢI THẤP HƯỚNG TỚI CÔNG NGHỆ SẢN XUẤT THÉP XANH Luyện thép bằng lò điện hồ quang (EAF) là công nghệ sử dụng thép phế liệu làm nguyên liệu chính. Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời hoặc điện gió. Tuy nhiên, việc chuyển dịch công nghệ phụ thuộc cao vào giải quyết bài toán nguyên vật liệu. Do sự khan hiếm trong nguồn cung nguyên liệu sạch, hiện nay, hơn 70% sản lượng thép toàn cầu vẫn đang được sản xuất theo công nghệ lò c"}\n{"id": "B-A-1d04b9f3749e42f8", "claim": "nguồn phát thải trực tiếp từ quá trình sản xuất chiếm tỷ trọng rõ rệt, phản ánh", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-4ac9436500318003", "claim": "là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp. Mặc dù đóng vai trò quan trọng trong cơ cấu sản xuất đa ngành của Tập đoàn, tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực. Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua các b"}\n{"id": "B-A-9a53268293eeb906", "claim": "lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể.", "evidence": "trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng lượng và nguyên liệu đầu vào, dẫn tới lượng phát thải CO2 đáng kể. Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát thải. Từ tháng 9 năm 2025, Hòa Phát Dung Quất 2 đi vào vận hành, Tổng sản lượng thép thô tăng 24%, tương ứng với mức tăng phát thải của toàn tập đoàn. Lĩnh vực nông nghiệp và chăn nuôi của Tập đoàn Hòa Phát ghi nhận mức phát 40 HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 NGUYÊN VẬT LIỆU: CÔNG NGHỆ: CHUYỂN DỊCH THEO HƯỚNG TĂNG TỶ TRỌNG NGUYÊN VẬT L"}\n{"id": "B-A-f9bfbff4eeca0da5", "claim": "các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao,", "evidence": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Giải pháp này bao gồm các hoạt động giúp tối ưu vận hành như cải thiện chất lượng phối liệu, kiểm soát kích cỡ than cốc, cải thiện phân bố liệu, tối ưu tốc độ gió nóng và"}\n{"id": "B-A-99edd0788de74772", "claim": "lượng thép thô tăng 24%, tương ứng với mức tăng phát thải của toàn tập đoàn.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 KẾT QUẢ TIÊU THỤ NĂNG LƯỢNG NĂM 2025 TỶ LỆ TỰ CHỦ ĐIỆN NĂNG Tại hai khu liên hợp sản xuất Gang thép Hòa Phát Hải Dương và Dung Quất, toàn bộ nhiệt dư trong luyện coke, khí than dư thừa trong luyện gang, thép được thu hồi làm nhiên liệu cho nồi hơi để phát điện. Công nghệ sản xuất điện năng này đã tăng tính cạnh tranh cho sản phẩm, góp phần bảo vệ môi trường, giảm lượng phát thải khí nhà kính. TIÊU THỤ NĂNG LƯỢNG TOÀN TẬP ĐOÀN & CƠ CẤU TIÊU THỤ: BÁO CÁO PHÁT TRIỂN BỀN VỮNG KHOẢN MỤC 2025 2025 193.404 Tổng tiêu thụ năng lượng toàn tập đoàn (nghìn GJ) PHẦN 5 XANH HÓA SẢN XUẤT Tổng tiêu thụ năng lượng sản xuất thép thô (nghìn GJ) 190.788"}\n{"id": "B-A-e713cad76de804ac", "claim": "Quản lý môi trường là chủ đề bền vững có mức độ ưu tiên cao, được tích hợp trong", "evidence": "Ổ CHỨC VÀ CÁC CÔNG TY CÙNG NGÀNH: để xác định mức độ ưu tiên. Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG Gắn kết với các bên liên quan BƯỚC 4 tại 15 doanh nghiệp lớn cùng ngành trên thế giới, các mối quan tâm về phát triển bền vững đã được xác định. Quá trình này tham chiếu 05 bộ khung phát triển bền vững quốc ƯU TIÊN CÁC CHỦ ĐỀ TRỌNG YẾU: 12 chủ đề được phân loại thành 3 nhóm: Chủ đề tế hàng đầu, trong đó có Bộ Tiêu chuẩn GRI Chung (phiên bản cập nhật), Đánh giá Phát ưu tiên (tác động cao), Chủ đề trọng tâm (tác động trung bình), và Chủ đề nền tảng (tác triển Bền vững Doanh nghiệp của S&P Global CSA, Hội đồng Chuẩn mực Kế toán Bền động thấp). Các chủ đề"}\n{"id": "B-A-6acf1c649fd0686e", "claim": "gồm quản lý và sử dụng nước, quản lý chất thải và nước thải, kiểm soát chất lượng", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TIÊU THỤ NƯỚC TOÀN TẬP ĐOÀN 2025 KIỂM SOÁT CHẤT LƯỢNG NƯỚC HOÀN TRẢ LẠI MÔI TRƯỜNG Là một phần trong chiến lược phát triển bền vững của Tập đoàn Hòa Phát, tất cả các công ty thành viên đều được yêu cầu triển khai các sáng kiến tiết kiệm nước, bao gồm thu gom và sử dụng nước mưa, cũng như tận dụng nguồn nước tuần hoàn nội bộ và tái sử dụng từ bên ngoài trong quá trình sản xuất. Những nỗ lực này nhằm giảm sự phụ thuộc vào nguồn nước ngọt, đặc biệt là nguồn nước khai thác từ cộng đồng, đồng thời góp phần thực hiện các mục tiêu bảo vệ môi trường rộng hơn của Tập đoàn. Chất lượng nước thải sau xử lý để hoàn trả lại môi trường được giám sát chặt chẽ và tuân thủ đầy đủ các qu"}\n{"id": "B-A-974e9cbdf9f15327", "claim": "nước thải, chất thải rắn, tích hợp trong hệ thống máy móc thiết bị, dây chuyền", "evidence": "áy, trang trại cho các hạng mục về bảo vệ môi trường, xử lý khí thải, nước thải, chất thải rắn, tích hợp trong hệ thống máy móc thiết bị, dây chuyền sản xuất. Quản lý môi trường CON NGƯỜI: Nâng cao nhận thức và năng lực Tập đoàn thường xuyên triển khai các chương trình đào tạo về quy định pháp luật liên quan đến bảo vệ môi trường cho cả cán bộ quản lý và nhân viên chuyên trách. Việc này đảm bảo bộ máy vận hành các cấp đều nắm rõ trách nhiệm và quy tắc vận hành, biến ý thức tuân thủ luật pháp thành hành động thực tế. QUY TRÌNH: Giám sát và tự kiểm soát chặt chẽ Công tác kiểm tra mức độ tuân thủ pháp luật được thực hiện thường xuyên thông qua hoạt động giám sát và đánh giá định kỳ. Các công ty"}\n{"id": "B-A-86273aa89983bf90", "claim": "thành viên của Tập đoàn, thể hiện cam kết phát triển có trách nhiệm và bền vững", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-f5132297419f2a95", "claim": "MÔI TRƯỜNG XANH VÀ HIỆU QUẢ NĂNG LƯỢNG: Giảm phát thải, tiết", "evidence": "ện năng tiêu thụ so với phương pháp truyền thống. • Thu hồi nhiệt từ quá trình thiêu kết, sử dụng nhiệt dư để phát điện, góp phần tối ưu hóa hiệu quả sử dụng năng lượng tổng thể. • Công nghệ đúc – cán liên tục cho phép nạp trực tiếp phôi nóng (750–900°C) vào dây chuyền cán. giúp giảm nhu cầu gia nhiệt, tiết kiệm năng lượng và giảm phát thải CO2 trên mỗi tấn thép. • Tận dụng các nguồn hơi nước từ hệ thống làm mát lò thổi, lò nung phôi để tận dụng phát điện. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn, do vậy kỳ báo cáo từ ngày 01/01/2025 đến 31/12/2025 được lựa chọn làm năm cơ sở vì đây là kỳ báo cáo đầu t"}\n{"id": "B-A-d9cb94f6b3d2e729", "claim": "kiệm năng lượng, thúc đẩy tái chế và tái sử dụng thông qua việc áp dụng công", "evidence": "u này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp "}\n{"id": "B-A-5987b064ff6d7a42", "claim": "nghệ sạch, thân thiện môi trường trong toàn bộ hoạt động sản xuất.", "evidence": "rường. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Công nghệ áp dụng tại trại chăn nuôi heo và gà của Hòa Phát là công nghệ an toàn cao sinh học với trại lạnh khép kín. Đây là loại hình chăn nuôi tập trung, ứng dụng công nghệ thông tin, công nghệ số vào trong quy trình chăn nuôi, đồng thời đảm bảo các yếu tố thân thiện với môi trường. Trại nuôi được xây dựng khép kín với hệ thống làm mát bằng nước, quạt hút gió, máng ăn tự động, uống nước tự động, chuồng nền, chăn nuôi riêng biệt các đối tượng và giai đoạn của vật nuôi. Nhiệt độ luôn nằm ở mức ổn định từ 18 - 25oC nhờ hệ thống điều khiển tự động, đàn vật nuôi ít bị mệt, giảm stress nhiệt giúp tiêu hóa hiệu quả và giảm phát thải kh"}\n{"id": "B-A-a3be06391a8ce08d", "claim": "nhân tạo (AI) nhằm nâng cao năng suất và năng lực cạnh tranh bền vững.", "evidence": "ÁT TRIỂN BỀN VỮNG thúc đẩy thực hành quản trị đạo đức theo các chuẩn mực quốc tế – qua đó hình thành hệ thống quản trị nhận được sự tin tưởng từ tất cả các bên liên quan. Ngoài ra, nhằm nâng cao tính sáng tạo và sự gắn kết trong công việc của đội ngũ, Hòa Phát coi nguồn nhân lực là nền tảng thành công. Do đó, Tập đoàn luôn ưu tiên các chương trình đào tạo nhằm phát triển trình độ chuyên môn và kỹ năng cho người lao động. Gắn kết với các bên liên quan Hòa Phát đặt mục tiêu trở thành một nhà sản xuất công nghiệp với chất lượng sản phẩm dẫn đầu, trong đó sản xuất thép là lĩnh vực cốt lõi. Để hiện thực hóa tầm nhìn này, chúng tôi chủ động nhận diện và giảm thiểu các rủi ro liên quan đến bền vững"}\n{"id": "B-A-6d4236326fa1c334", "claim": "đáng kể tải lượng ô nhiễm và loại bỏ mùi hôi của chất thải trước khi xả ra môi", "evidence": "QUẢN LÝ MÔI TRƯỜNG GRI 103 – Năng lượng 2025 3-3 Quản lý chủ đề trọng yếu Phương pháp quản lý QUẢN LÝ CHẤT LƯỢNG KHÔNG KHÍ Kiểm soát chất lượng không khí Oxit nitơ (NOx). oxit lưu huỳnh (SOx) và các khí thải không khí đáng GRI 305- Phát thải 2016 305-7 kể khác QUẢN LÝ NƯỚC GRI 3 – Chủ đề trọng yếu 2021 3-3 Quản lý chủ đề trọng yếu Quản lý nước 303-2 Quản lý tác động liên quan đến xả thải nước Quản lý nước GRI 303 – Nước và Nước thải 2018 303-3 Lượng nước khai thác Quản lý nước 303-4 Lượng nước thải Quản lý nước 303-5 Lượng nước tiêu thụ Quản lý nước QUẢN LÝ CHẤT THẢI RẮN GRI 3 – Chủ đề trọng yếu 2021 3-3 Quản lý chủ đề trọng yếu Quản lý chất thải rắn GRI 306: Waste 2020 306-1 Phát sinh chất "}\n{"id": "B-A-820a76dd8488108c", "claim": "nguồn năng lượng tái tạo để đun nấu hoặc phát điện, giúp giảm chi phí vận hành", "evidence": "ược thực hiện đồng bộ qua hai phương thức chính: tận dụng nhiệt năng và hóa năng. Ở giai đoạn luyện cốc và hệ thống dập cốc khô (CDQ), nguồn hơi nhiệt dư và hơi CDQ áp suất cao được dẫn trực tiếp vào các nồi hơi chuyên dụng. Song song đó, tại khu vực lò cao (BFG), nguồn hóa năng từ khí than lò cao cũng được thu hồi để đốt trong nồi hơi khí than. Tất cả nguồn hơi này hội tụ về hệ thống quản lý năng lượng tập trung để làm quay máy phát điện công suất lớn. Chu trình khép kín này giúp chuyển hóa toàn bộ phụ phẩm phát sinh thành điện năng sạch, phục vụ trực tiếp cho các hoạt động sản xuất tại nhà máy. Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ưu hóa chi phí vận hà"}\n{"id": "B-A-48000a487fae13f3", "claim": "và đồng thời giảm phát thải khí nhà kính mạnh ra môi trường.", "evidence": "của sản phẩm, bao gồm thu thập, kiểm chứng và báo cáo dữ liệu theo quy định của Bộ Công Thương, Bộ Nông Nghiệp và Môi trường và theo yêu cầu của khách hàng, đối tác. Hằng năm, Ban xây dựng kế hoạch giảm phát thải, theo dõi tiến độ triển khai và đề xuất các giải pháp chiến lược nhằm nâng cao hiệu quả năng lượng và giảm thiểu tác động môi trường. Báo cáo được thiết lập dựa trên các yêu cầu của tiêu chuẩn ISO 14064-1, theo các hướng dẫn của Ban liên chính phủ về biến đổi khí hậu (IPCC Guidelines) và đáp ứng các yêu cầu về giảm phát thải khí nhà kính của Cơ quan quản lý được đề cập trong Nghị định 06/2022/ND-CP, Nghị định 119/2024/ND-CP. Đối với số liệu kiểm kê KNK của Tập đoàn, phương pháp hợp "}\n{"id": "B-A-e68a9aec31c03b92", "claim": "đàn vật nuôi ít bị mệt, giảm stress nhiệt giúp tiêu hóa hiệu quả và giảm phát thải", "evidence": "c nhóm Nông nghiệp của Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích ứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được áp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm soát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, hạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng ấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm tối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết lập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng trưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật"}\n{"id": "B-A-f4bdaf684fcfed77", "claim": "phép thu gom toàn bộ chất thải hiệu quả, hàng ngày, triệt để hơn so với chăn", "evidence": "ÁT KHUẾCH TÁN MẦM BỆNH Kỹ thuật chăn nuôi và quản lý cùng vào cùng ra theo tuần (AIAO) là công nghệ tiên tiến thế giới về chăn nuôi hiện nay. Đây là hệ thống chăn nuôi theo nhóm các lứa cùng độ tuổi, căn nặng, thể trạng, chu kỳ sinh sản cũng như chế độ dinh dưỡng. Lợn được phối theo nhóm tuần và luân chuyển hết sang các khu chức năng khác theo tuần. Kỹ thuật chăn nuôi này tạo khoảng thời gian chuồng nghỉ cho phép người chăn nuôi thực hiện rửa sạch, sát trùng và làm khô chuồng trại một cách triệt để. Quá trình này giúp tiêu diệt hầu hết các mầm bệnh còn sót lại trong môi trường chuồng nuôi, phá vỡ chuỗi lây nhiễm cho nhóm heo mới cũng như kiểm soát việc khuếch tán mầm bệnh trong không khí. XÂ"}\n{"id": "B-A-d0e0d04a3ba01c87", "claim": "Việc vệ sinh, thu gom chất thải được thực hiện thường", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-a4f109322b175dc7", "claim": "Việc xây dựng hầm biogas là giải pháp kinh tế tuần hoàn để xử lý chất thải đầu", "evidence": "vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi "}\n{"id": "B-A-8359e3943b42e76e", "claim": "hợp được xử lý thông qua hệ thống xử lý nước trung tâm và tái chế 100%.", "evidence": "2025 Trong lĩnh vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. Hành động giảm thiểu và thích ứng với biến đổi khí hậu CHỈ TIÊU NĂM 2025 (Triệu lít) Quản lý môi trường Nước khai thác 46.328 Nước thải hoàn trả lại môi trường sau khi qua hệ thống xử lý nước"}\n{"id": "B-A-2dc1b17c9d451628", "claim": "Là một phần trong chiến lược phát triển bền vững của Tập đoàn Hòa Phát, tất cả", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 [GRI 2-12, 2-24, 3-1, 3-2, 3-3] TRỞ THÀNH TẬP ĐOÀN SẢN XUẤT CÔNG NGHIỆP VỚI CHẤT LƯỢNG DẪN ĐẦU, TRONG ĐÓ THÉP LÀ LĨNH VỰC CỐT LÕI Tầm nhìn BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 CUNG CẤP SẢN PHẨM DẪN ĐẦU, GÓP PHẦN NÂNG CAO CHẤT LƯỢNG CUỘC SỐNG, Sứ mệnh PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT ĐẠT ĐƯỢC SỰ TIN YÊU CỦA KHÁCH HÀNG Trụ cột XANH HÓA SẢN XUẤT PHÁT TRIỂN CON NGƯỜI QUẢN TRỊ VÀ ĐỔI MỚI Chuyển đổi quy trình sản xuất thép theo hướng ít carbon, sử dụng tài nguyên hiệu quả và vận hành Xây dựng môi trường làm việc an toàn, Tăng cường quản trị minh bạch, Mục tiêu công bằng và thúc đẩy sự phát triển tích hợp ESG vào chiến lược, và th"}\n{"id": "B-A-7d20d14ed1b903fc", "claim": "các công ty thành viên đều được yêu cầu triển khai các sáng kiến tiết kiệm nước,", "evidence": "ập đoàn Hòa Phát, tất cả các công ty thành viên đều được yêu cầu triển khai các sáng kiến tiết kiệm nước, bao gồm thu gom và sử dụng nước mưa, cũng như tận dụng nguồn nước tuần hoàn nội bộ và tái sử dụng từ bên ngoài trong quá trình sản xuất. Những nỗ lực này nhằm giảm sự phụ thuộc vào nguồn nước ngọt, đặc biệt là nguồn nước khai thác từ cộng đồng, đồng thời góp phần thực hiện các mục tiêu bảo vệ môi trường rộng hơn của Tập đoàn. Việc đầu tư nâng cấp hạ tầng và triển khai các dự án chuyên biệt nhằm nâng cao hiệu quả sử dụng nước và gia tăng tỷ lệ sử dụng nước tuần hoàn đã giúp giảm đáng kể nhu cầu dùng nước ngọt, trong đó Khu liên hợp Gang Thép Hòa Phát Dung Quất đạt tỷ lệ tái sử dụng nước đ"}\n{"id": "B-A-1cfa1a508a8bbb4a", "claim": "môi trường nước và duy trì hoạt động sản xuất bền vững, hiệu quả.", "evidence": "ạnh mẽ của Hòa Phát trong việc bảo vệ môi trường nước và duy trì hoạt động sản xuất bền vững, hiệu quả. LƯỢNG NƯỚC KHAI THÁC, HOÀN TRẢ VÀ SỬ DỤNG NĂM 2025 Hành động giảm thiểu và thích ứng với biến đổi khí hậu CHỈ TIÊU NĂM 2025 (Triệu lít) Quản lý môi trường Nước khai thác 46.328 Nước thải hoàn trả lại môi trường sau khi qua hệ thống xử lý nước 4.994 41.334 Nước sử dụng(*) Ghi chú: (*) nước sử dụng là tổng lượng nước được sử dụng trong hoạt động của doanh nghiệp, được tính bằng lượng nước khai thác trừ đi lượng nước thải ra trở lại môi trường. Khái niệm này bao gồm phần nước tiêu thụ (do bay hơi, hấp thụ vào sản phẩm hoặc thất thoát) và nước được tái sử dụng trong quy trình sản xuất. 48 QUẢN"}\n{"id": "B-A-9476d90177c90fce", "claim": "Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-429924f1ea6fe305", "claim": "tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường,", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 [GRI 301-1, 301-2, 303-4, 303-5, 306-1, 306-2, 306-3, 306-4, 306-5, 306-6] Quản lý môi trường là chủ đề bền vững có mức độ ưu tiên cao, được tích hợp trong toàn bộ hoạt động sản xuất, kinh doanh của Hòa Phát. Tập đoàn hướng tới sử dụng tài nguyên hiệu quả, phòng ngừa ô nhiễm và thúc đẩy mô hình sản xuất tuần hoàn. Công tác quản lý môi trường của Tập đoàn tập trung vào các khía cạnh trọng yếu gồm quản lý và sử dụng nước, quản lý chất thải và nước thải, kiểm soát chất lượng không khí nhằm giảm thiểu tác động môi trường, tuân thủ quy định pháp luật và hướng tới tăng trưởng xanh. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT TUÂN THỦ PHÁP LUẬT VỀ MÔI TRƯỜNG Tập"}\n{"id": "B-A-f17393d4cfe7a2b4", "claim": "pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn.", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-b4e5ced270934a9b", "claim": "quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn.", "evidence": "liệu nền đường. Quản lý môi trường LƯỢNG NƯỚC KHAI THÁC, HOÀN TRẢ VÀ SỬ DỤNG NĂM 2025 CHỈ TIÊU NĂM 2025 Năm 2025, Tập đoàn Hòa Phát đầu tư hạng mục xử lý bụi và bùn chứa kẽm với công nghệ lò đáy quay do CISDI Thermal & Environmental Engineering Co,. Ltd. (thành viên của MCC-China) cung cấp. Hệ thống có năng lực xử lý toàn bộ lượng bụi và bùn chứa kẽm phát sinh tại Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 1 và 2. Công nghệ Lò đáy quay giúp tách bụi lò cao thành quặng sắt hoàn nguyên trực tiếp (DRI), là nguồn quặng đầu vào quay trở lại sản xuất thép và bột oxit kẽm là sản phẩm phụ có giá trị cao, có thể bán cho các nhà máy luyện kẽm, ứng dụng mạ kẽm hoặc ngành hóa chất. Hệ thống đồng"}\n{"id": "B-A-3308b4d973c6241d", "claim": "với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn", "evidence": "Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý Quản lý chất thải rắn"}\n{"id": "B-A-2be6ca9b0a31d4fc", "claim": "Chất thải rắn trong mô hình sản xuất thép tuần hoàn không còn được xem", "evidence": "liệu nền đường. Quản lý môi trường LƯỢNG NƯỚC KHAI THÁC, HOÀN TRẢ VÀ SỬ DỤNG NĂM 2025 CHỈ TIÊU NĂM 2025 Năm 2025, Tập đoàn Hòa Phát đầu tư hạng mục xử lý bụi và bùn chứa kẽm với công nghệ lò đáy quay do CISDI Thermal & Environmental Engineering Co,. Ltd. (thành viên của MCC-China) cung cấp. Hệ thống có năng lực xử lý toàn bộ lượng bụi và bùn chứa kẽm phát sinh tại Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 1 và 2. Công nghệ Lò đáy quay giúp tách bụi lò cao thành quặng sắt hoàn nguyên trực tiếp (DRI), là nguồn quặng đầu vào quay trở lại sản xuất thép và bột oxit kẽm là sản phẩm phụ có giá trị cao, có thể bán cho các nhà máy luyện kẽm, ứng dụng mạ kẽm hoặc ngành hóa chất. Hệ thống đồng"}\n{"id": "B-A-bd370ad8254abb53", "claim": "Hiện nay, tỷ lệ tái chế chất thải rắn của nhóm thép", "evidence": "tại Khu liên hợp Gang thép Hòa Phát Dung Quất Tái chế, tái sử dụng chất thải rắn MÔI TRƯỜNG nhóm thép và nông nghiệp đạt trên 99% Điểm nổi bật về QUẢN LÝ MÔI TRƯỜNG 5.2 Tái chế, tái sử dụng chất thải rắn thành nguyên vật liệu quay lại quá trình sản xuất Các giải pháp tái chế, tái sử dụng nội bộ tập trung vào việc phân loại và xử lý các chất thải ngay tại nguồn phát sinh. Theo đó, các chất thải rắn chứa sắt sẽ được thu"}\n{"id": "B-A-5bb4ec49b3bd0861", "claim": "SƠ ĐỒ XỬ LÝ CHẤT THẢI RẮN PHÁT SINH TẠI KLH SẢN XUẤT GANG THÉP HÒA PHÁT DUNG QUẤT", "evidence": "liệu nền đường. Quản lý môi trường LƯỢNG NƯỚC KHAI THÁC, HOÀN TRẢ VÀ SỬ DỤNG NĂM 2025 CHỈ TIÊU NĂM 2025 Năm 2025, Tập đoàn Hòa Phát đầu tư hạng mục xử lý bụi và bùn chứa kẽm với công nghệ lò đáy quay do CISDI Thermal & Environmental Engineering Co,. Ltd. (thành viên của MCC-China) cung cấp. Hệ thống có năng lực xử lý toàn bộ lượng bụi và bùn chứa kẽm phát sinh tại Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 1 và 2. Công nghệ Lò đáy quay giúp tách bụi lò cao thành quặng sắt hoàn nguyên trực tiếp (DRI), là nguồn quặng đầu vào quay trở lại sản xuất thép và bột oxit kẽm là sản phẩm phụ có giá trị cao, có thể bán cho các nhà máy luyện kẽm, ứng dụng mạ kẽm hoặc ngành hóa chất. Hệ thống đồng"}\n{"id": "B-A-b86e772d9fc2a526", "claim": "Tái chế, tái sử dụng chất thải rắn thành nguyên vật liệu quay lại quá trình sản", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-4c3265cd1cd1b7dc", "claim": "Các giải pháp tái chế, tái sử dụng nội bộ tập trung vào việc phân loại và xử lý các", "evidence": "i chế. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Hiện nay, các biện pháp giảm phát thải trong sản xuất thép từ quặng (vòng đời sơ cấp – sản xuất thép mới) trên thế giới chủ yếu tập trung vào việc sử dụng các loại nguyên liệu đầu vào có hàm lượng carbon thấp làm chất khử quặng sắt như khí tự nhiên hoặc Hydrogen xanh (sản xuất từ nguồn điện sạch) để tạo ra sắt hoàn nguyên trực tiếp (DRI và HBI), thay thế cho việc hoàn nguyên bằng nhiên liệu hóa thạch truyền thống như than. Hạn chế khiến cho biện pháp này chưa thể áp dụng rộng rãi nằm ở sự khan hiếm nguồn cung các loại quặng sắt có hàm lượng sắt cao %Fe>65% cũng như nguồn khí tự nhiên và hydrogen sạch. Biện pháp thứ hai là sử dụng thép phế liệu để tái c"}\n{"id": "B-A-8f4e6a5c20d93d0c", "claim": "Theo đó, các chất thải rắn chứa sắt sẽ được thu", "evidence": "Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý Quản lý chất thải rắn"}\n{"id": "B-A-1adc11dcabf03bf8", "claim": "đồng thời giải quyết vấn đề chất thải rắn.", "evidence": "rất cao, góp phần giảm phát thải gián tiếp và giảm tải gánh nặng điện năng quốc gia. Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy. Bên cạnh đó, Hòa Phát cũng chú trọng quản lý chặt chẽ khí thải, chất thải, sử dụng tiết kiệm tài nguyên nước không chỉ tại các nhà máy thép, mà trong cả lĩnh vực nông nghiệp, bất động sản, điện máy gia dụng. Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. Trụ cột Quản trị và Đổi "}\n{"id": "B-A-12bebdc9d1a9da7c", "claim": "Chất thải nguy hại được xử lý đúng quy định", "evidence": "Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào quy trình xử lý Tổng khối lượng chất thải NGUY HẠI được xử lý thải bỏ Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại)"}\n{"id": "B-A-b9d6a4ffc6ad8fb7", "claim": "Các chất thải nguy hại (mỡ, giẻ lau dính dầu, hóa chất) đều được lưu giữ an toàn,", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-ac48fd4ec2d6258b", "claim": "Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm", "evidence": "vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi "}\n{"id": "B-A-a8982a8ca6acad64", "claim": "khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo.", "evidence": "vực nông nghiệp. Phân và chất thải chăn nuôi được xử lý bằng hệ thống ủ phân hữu cơ hoặc hầm khi sinh học (biogas) để sản xuất phân bón hữu cơ và năng lượng tái tạo. Tỷ lệ tái chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%. PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. Trong thời gian tới, Tập đoàn sẽ tiếp tục nâng cao hiệu quả phân loại tại nguồn, mở rộng hợp tác trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử dụng, nâng cao tỷ lệ thu hồi "}\n{"id": "B-A-c30116767ee85187", "claim": "chế/tái sử dụng chất thải rắn của nhóm Nông nghiệp là gần 100%.", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-afee2ac627342ef4", "claim": "Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-2467ed3107c5b502", "claim": "hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-5cb23f8e79d560a9", "claim": "trong lĩnh vực tái chế và áp dụng công nghệ đổi mới nhằm gia tăng tỷ lệ tái sử", "evidence": "ập đoàn Hòa Phát, tất cả các công ty thành viên đều được yêu cầu triển khai các sáng kiến tiết kiệm nước, bao gồm thu gom và sử dụng nước mưa, cũng như tận dụng nguồn nước tuần hoàn nội bộ và tái sử dụng từ bên ngoài trong quá trình sản xuất. Những nỗ lực này nhằm giảm sự phụ thuộc vào nguồn nước ngọt, đặc biệt là nguồn nước khai thác từ cộng đồng, đồng thời góp phần thực hiện các mục tiêu bảo vệ môi trường rộng hơn của Tập đoàn. Việc đầu tư nâng cấp hạ tầng và triển khai các dự án chuyên biệt nhằm nâng cao hiệu quả sử dụng nước và gia tăng tỷ lệ sử dụng nước tuần hoàn đã giúp giảm đáng kể nhu cầu dùng nước ngọt, trong đó Khu liên hợp Gang Thép Hòa Phát Dung Quất đạt tỷ lệ tái sử dụng nước đ"}\n{"id": "B-A-7bd838bffad24325", "claim": "Chuyển đổi các chất thải có khối lượng lớn thành sản phẩm phụ", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-fba769db5abf820f", "claim": "LƯỢNG CHẤT THẢI THEO CÁC HƯỚNG XỬ LÝ NĂM 2025", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-00a5ae2516178a0c", "claim": "Dự án không chỉ góp phần giảm chất thải rắn và tăng hiệu quả tái chế nội bộ,", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-89ddddb06c45aa29", "claim": "mà còn thể hiện định hướng của Hòa Phát trong kinh tế tuần hoàn và sử dụng tài", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-b0e1227e4ec97f29", "claim": "Tổng khối lượng chất thải được xử lý thải bỏ", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-216450c5ed6edde0", "claim": "minh bạch và hiệu quả bền vững của sản phẩm.", "evidence": "hiệu quả công việc. TẦM NHÌN & ĐỊNH HƯỚNG CHIẾN LƯỢC CHUYỂN ĐỔI SỐ Tập đoàn định hướng xây dựng mô hình doanh nghiệp sản xuất công nghiệp chất lượng dẫn đầu, vận hành trên nền tảng dữ liệu tập trung, quy trình chuẩn hóa và hệ thống tự động hóa thông minh. Quản lý chất lượng Phát triển đổi mới sáng tạo Chuyển đổi số được triển khai với mục tiêu: • Chuẩn hóa và minh bạch hóa quản trị • Tối ưu hiệu suất vận hành • Nâng cao trải nghiệm khách hàng • Giảm tiêu hao tài nguyên và phát thải gián tiếp • Tăng khả năng thích ứng với biến động thị trường Chiến lược này đồng thời đóng góp trực tiếp vào 3 trụ cột ESG: QUẢN TRỊ: Hệ thống số hóa giúp tăng cường minh bạch thông tin, chuẩn hóa quy trình phê du"}\n{"id": "B-A-4cf2eb5245498c9e", "claim": "mua sắm xanh, phát triển thép carbon thấp, đồng thời nâng cao năng lực cạnh", "evidence": "hình thức hối lộ hoặc tác động không chính đáng giữa nhà thầu và nhân viên công ty nhằm mục đích trục lợi. Đảm bảo việc tuân thủ pháp luật, điều lệ công ty và TUÂN THỦ & ĐẢM BẢO các chính sách nội bộ thông qua hoạt động kiểm toán có hệ thống. Quản trị Doanh nghiệp và Quản lý rủi ro GIẢM THIỂU Xác định các điểm yếu trong hệ thống kiểm soát, giám sát tiến độ dự án và hoạt động mua sắm, đồng thời chủ RỦI RO Trong thời gian tới, Tập đoàn Hòa Phát đặt mục tiêu mở rộng việc áp dụng các thông lệ tốt này sang các công ty thành viên khác, qua đó củng cố cam kết về đạo đức kinh doanh, cạnh tranh lành mạnh và tuân thủ pháp luật, góp phần xây dựng môi trường kinh doanh minh bạch và có trách nhiệm. động "}\n{"id": "B-A-911ee26fd50de0e7", "claim": "Năm 2025, tổng lượng chất thải rắn phát sinh của Tập đoàn là 7.085 nghìn tấn,", "evidence": "g giảm thiểu và thích ứng với biến đổi khí hậu Điện máy gia dụng 104,6 57,8 55% Cộng 7.085 6.982 99% Năm 2025, tổng lượng chất thải rắn phát sinh của Tập đoàn là 7.085 nghìn tấn, trong đó chất thải nguy hại chiếm 3%, chất thải không nguy hại chiếm 97%. Quản lý môi trường Trong tổng số hơn 7 triệu tấn chất thải rắn phát sinh, có đến gần 6,98 triệu tấn đã được thu hồi và đưa quay trở lại chu trình sản xuất hoặc chuyển đổi mục đích sử dụng. Con số này cho thấy chất thải không còn là gánh nặng môi trường mà đã thực sự trở thành nguồn tài nguyên đầu vào quan trọng, giúp Tập đoàn tối ưu hóa chi phí và giảm thiểu tác động tiêu cực đến hệ sinh thái. 51 liệu nền đường. Quản lý môi trường LƯỢNG NƯỚC K"}\n{"id": "B-A-5e9a03c8a13b4241", "claim": "trong đó chất thải nguy hại chiếm 3%, chất thải không nguy hại chiếm 97%.", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-4ff870832b0ee3d4", "claim": "Trong tổng số hơn 7 triệu tấn chất thải rắn phát sinh, có đến gần 6,98 triệu tấn", "evidence": "Tổng lượng nước khai thác từ các bên thứ ba Triệu lít 1.275 303-3 Khai thác nước Tổng lượng nước khai thác từ tất cả các khu vực có áp lực về nước Triệu lít N/A 303-3 Khai thác nước Tổng lượng nước sử dụng từ tất cả các khu vực Triệu lít 41.334 303-5 Tiêu thụ nước Tổng lượng nước xả thải ra các khu vực Triệu lít 4.994 303-4 Xả thải nước Tổng lượng nước ngọt tiêu thụ tại các khu vực thiếu nước Triệu lít N/A Tổng lượng nước ngọt tiêu thụ tại các khu vực không thiếu nước Triệu lít N/A Chất thải nguy hại Tấn 246.867 306-3 Chất thải phát sinh Chất thải không nguy hại Tấn 6.838.336 306-3 Chất thải phát sinh Chất thải khác Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát si"}\n{"id": "B-A-01392ee4a6dc0233", "claim": "Con số này cho thấy chất thải không còn là gánh nặng môi trường mà", "evidence": "toàn. Chất thải rắn trong mô hình sản xuất thép tuần hoàn không còn được xem là gánh nặng chôn lấp, đổ bỏ mà trở thành tài nguyên được tận dụng tối đa và mang lại hiệu quả kinh tế. Hiện nay, tỷ lệ tái chế chất thải rắn của nhóm thép hợp và nhà máy, trang trại cho các hạng mục về bảo vệ môi trường, xử lý khí thải, nước thải, chất thải rắn, tích hợp trong hệ thống máy móc thiết bị, dây chuyền sản xuất. Quản lý môi trường CON NGƯỜI: Nâng cao nhận thức và năng lực"}\n{"id": "B-A-6b624ec66621f66b", "claim": "hóa chi phí và giảm thiểu tác động tiêu cực đến hệ sinh thái.", "evidence": "ược thực hiện đồng bộ qua hai phương thức chính: tận dụng nhiệt năng và hóa năng. Ở giai đoạn luyện cốc và hệ thống dập cốc khô (CDQ), nguồn hơi nhiệt dư và hơi CDQ áp suất cao được dẫn trực tiếp vào các nồi hơi chuyên dụng. Song song đó, tại khu vực lò cao (BFG), nguồn hóa năng từ khí than lò cao cũng được thu hồi để đốt trong nồi hơi khí than. Tất cả nguồn hơi này hội tụ về hệ thống quản lý năng lượng tập trung để làm quay máy phát điện công suất lớn. Chu trình khép kín này giúp chuyển hóa toàn bộ phụ phẩm phát sinh thành điện năng sạch, phục vụ trực tiếp cho các hoạt động sản xuất tại nhà máy. Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ưu hóa chi phí vận hà"}\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: INPUTS=gsi_layer_b1.jsonl python -u multirun.py jobs_32b.jso', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('INPUTS=gsi_layer_b1.jsonl python -u multirun.py jobs_32b.json issue_worker.py')


In [ ]:
print('[cell] run 2: ls -la out; cat out/*_summary.json', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('ls -la out; cat out/*_summary.json')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)